# Pipeline estéreo + filtro térmico + trajetória do jato — versão revisada v6

Esta versão concentra os ajustes em uma única célula de configuração com imagem-base. As células de vídeo não possuem mais sliders de ajuste: elas apenas executam a detecção, estimam distância, planejam a trajetória e mostram a mira do jato.

Pasta esperada do notebook:

```text
vision/fire_detect/jet_automation
```

Fluxo recomendado:

1. Execute as células 1 e 2.
2. Na célula 3, selecione uma imagem térmica de base, escolha a escala, ajuste filtros, morfologia e trajetória, e salve o setup.
3. Execute uma das células de vídeo:
   - simulação com apagamento artificial;
   - uso real sem apagamento artificial.


In [1]:
# ============================================================
# CÉLULA 1 - Imports, caminhos e calibração estéreo
# ============================================================

from pathlib import Path
import os
import json
import time
import math
import glob
import warnings

import numpy as np
import cv2

from IPython.display import display, clear_output
import ipywidgets as widgets

# ------------------------------------------------------------
# Localização portátil da raiz do projeto
# ------------------------------------------------------------
def localizar_raiz_blaze(start=None):
    start = Path(start or Path.cwd()).resolve()

    for pasta in [start, *start.parents]:
        if (pasta / ".git").exists() and (pasta / "vision").exists():
            return pasta

        if (
            (pasta / "vision").exists()
            and (pasta / "src" / "blaze_paths.py").exists()
        ):
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto BLAZE. "
        "Abra o notebook a partir de uma pasta dentro do repositório."
    )


PROJECT_ROOT = localizar_raiz_blaze()
BASE_DIR = (
    PROJECT_ROOT
    / "vision"
    / "fire_detect"
    / "jet_automation"
)

if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR não encontrada: {BASE_DIR}")

os.chdir(BASE_DIR)

DATA_DIR = BASE_DIR / "data"
PARAMS_DIR = BASE_DIR / "parameters_setups"
PARAMS_DIR.mkdir(parents=True, exist_ok=True)
THERMAL_SETUPS_FILE = PARAMS_DIR / "thermal_preprocess_setups.json"

# Pode ser ajustado aqui apenas se as câmeras tiverem outro índice.
CAM1_INDEX = 0
CAM2_INDEX = 2

# Tamanho solicitado para captura. Se a câmera não aceitar, o OpenCV usa o possível.
CAP_WIDTH = 640
CAP_HEIGHT = 480

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("THERMAL_SETUPS_FILE:", THERMAL_SETUPS_FILE)


BASE_DIR: vision/fire_detect/jet_automation
DATA_DIR: vision/fire_detect/jet_automation/data
THERMAL_SETUPS_FILE: vision/fire_detect/jet_automation/parameters_setups/thermal_preprocess_setups.json


In [2]:
# ============================================================
# CÉLULA 2 - Carregar calibração, paletas e funções auxiliares
# ============================================================

# ------------------------------------------------------------
# Calibração estéreo
# ------------------------------------------------------------
def find_calibration_file():
    candidates = [
        PARAMS_DIR / "stereo_calibration2.npz",
        PARAMS_DIR / "stereo_calibration.npz",
        BASE_DIR / "stereo_calibration2.npz",
        BASE_DIR / "stereo_calibration.npz",
        BASE_DIR.parent / "stereo_calibration2.npz",
        BASE_DIR.parent / "stereo_calibration.npz",
    ]
    for c in candidates:
        if c.exists():
            return c
    found = (
        list(PARAMS_DIR.glob("*.npz"))
        + list(BASE_DIR.glob("*.npz"))
        + list(BASE_DIR.parent.glob("*.npz"))
    )
    if found:
        return found[0]
    raise FileNotFoundError("Nenhum arquivo .npz de calibração encontrado em jet_automation ou fire_detect.")

CALIB_FILE = find_calibration_file()
data = np.load(CALIB_FILE)

K1, D1 = data["K1"], data["D1"]
K2, D2 = data["K2"], data["D2"]
R1, R2 = data["R1"], data["R2"]
P1, P2 = data["P1"], data["P2"]

print("Calibração carregada:", CALIB_FILE)
print("K1:", K1.shape, "P1:", P1.shape)

# ------------------------------------------------------------
# Paletas térmicas
# ------------------------------------------------------------
def criar_lut_por_pontos(pontos_rgb, n=256):
    pontos_rgb = sorted(pontos_rgb, key=lambda x: x[0])
    lut = np.zeros((n, 1, 3), dtype=np.uint8)
    xs = np.array([p[0] for p in pontos_rgb], dtype=np.float32)
    rs = np.array([p[1][0] for p in pontos_rgb], dtype=np.float32)
    gs = np.array([p[1][1] for p in pontos_rgb], dtype=np.float32)
    bs = np.array([p[1][2] for p in pontos_rgb], dtype=np.float32)
    x_new = np.linspace(0, 1, n)
    r_new = np.interp(x_new, xs, rs)
    g_new = np.interp(x_new, xs, gs)
    b_new = np.interp(x_new, xs, bs)
    # OpenCV usa BGR
    lut[:, 0, 0] = b_new.astype(np.uint8)
    lut[:, 0, 1] = g_new.astype(np.uint8)
    lut[:, 0, 2] = r_new.astype(np.uint8)
    return lut

PALETTE_LUTS = {
    "1. Arco-íris / Rainbow": criar_lut_por_pontos([
        (0.00, (90, 0, 255)),
        (0.20, (0, 80, 255)),
        (0.40, (0, 255, 255)),
        (0.60, (0, 255, 0)),
        (0.80, (255, 255, 0)),
        (1.00, (255, 0, 0)),
    ]),
    "2. Inferno / Plasma térmico": criar_lut_por_pontos([
        (0.00, (20, 0, 60)),
        (0.25, (100, 0, 120)),
        (0.50, (220, 0, 80)),
        (0.75, (255, 150, 0)),
        (1.00, (255, 255, 180)),
    ]),
    "3. Cinza / Grayscale": criar_lut_por_pontos([
        (0.00, (0, 0, 0)),
        (1.00, (255, 255, 255)),
    ]),
    "4. Verde-amarelo térmico": criar_lut_por_pontos([
        (0.00, (40, 20, 0)),
        (0.30, (120, 80, 0)),
        (0.60, (180, 220, 0)),
        (0.85, (240, 255, 120)),
        (1.00, (255, 255, 245)),
    ]),
    "5. Preto-roxo-vermelho-branco": criar_lut_por_pontos([
        (0.00, (0, 0, 0)),
        (0.20, (60, 0, 120)),
        (0.45, (120, 0, 0)),
        (0.70, (255, 0, 0)),
        (0.90, (255, 180, 180)),
        (1.00, (255, 255, 255)),
    ]),
    # Nesta escala, o extremo quente foi colocado no vermelho escuro,
    # pois é o comportamento desejado para a imagem térmica do projeto.
    "6. Branco-amarelo-vermelho-escuro": criar_lut_por_pontos([
        (0.00, (255, 255, 255)),
        (0.25, (255, 255, 0)),
        (0.45, (255, 130, 0)),
        (0.65, (255, 0, 0)),
        (0.85, (130, 0, 0)),
        (1.00, (30, 0, 0)),
    ]),
}
PALETTE_NAMES = list(PALETTE_LUTS.keys())

PALETTE_ALIASES = {
    "CUSTOM_branco_amarelo_vermelho_escuro": "6. Branco-amarelo-vermelho-escuro",
    "branco_amarelo_vermelho_escuro": "6. Branco-amarelo-vermelho-escuro",
    "azul_verde_amarelo_vermelho": "1. Arco-íris / Rainbow",
    "jet_opencv": "1. Arco-íris / Rainbow",
    "inferno_opencv": "2. Inferno / Plasma térmico",
    "cinza_preto_branco": "3. Cinza / Grayscale",
}

def normalize_palette_name(name):
    name = PALETTE_ALIASES.get(str(name), str(name))
    if name not in PALETTE_LUTS:
        name = "6. Branco-amarelo-vermelho-escuro"
    return name

# ------------------------------------------------------------
# Setup global atual
# ------------------------------------------------------------
def default_params():
    return {
        "palette_name": "6. Branco-amarelo-vermelho-escuro",
        "temp_min_c": 300.0,
        "temp_max_c": 1025.0,
        "temp_filter_min_c": 575.0,
        "temp_filter_max_c": 1025.0,
        "blur_ksize": 3,
        "open_iter": 1,
        "close_iter": 1,
        "dilate_iter": 1,
        "min_area": 300,
        "max_area": 999999,
        "use_rectification": False,
        "large_region_area": 2500,
        "offset_y_px": 8,
        "route_step_px": 18,
        "centroid_dwell_points": 4,
        "move_step_px": 15,
        "jet_pressure_calib": 22.0,
        "jet_cooling_radius_px": 28,
        "human_safety_margin_px": 30,
        "enable_yolo_human_safety": False,
        "enable_arduino": False,
        "serial_port": "/dev/ttyACM0",
        "serial_baud": 9600,
        "servo_x_min": 20,
        "servo_x_max": 160,
        "servo_y_min": 20,
        "servo_y_max": 160,
    }

CURRENT_PREPROCESS_PARAMS = default_params()

# ------------------------------------------------------------
# Arquivos de setup
# ------------------------------------------------------------
def load_all_setups():
    if THERMAL_SETUPS_FILE.exists():
        try:
            with open(THERMAL_SETUPS_FILE, "r", encoding="utf-8") as f:
                setups = json.load(f)
            if isinstance(setups, dict):
                return setups
        except Exception as e:
            print("Aviso: falha ao carregar setups:", e)
    return {}

def save_setup(name, params):
    setups = load_all_setups()
    params = dict(params)
    params["palette_name"] = normalize_palette_name(params.get("palette_name"))
    setups[name] = params
    with open(THERMAL_SETUPS_FILE, "w", encoding="utf-8") as f:
        json.dump(setups, f, indent=2, ensure_ascii=False)
    return THERMAL_SETUPS_FILE

def slugify(txt):
    txt = str(txt).lower()
    repl = {
        "á":"a", "à":"a", "ã":"a", "â":"a",
        "é":"e", "ê":"e", "í":"i", "ó":"o", "ô":"o", "õ":"o",
        "ú":"u", "ç":"c", "/":"", ".":"", "-":"_"
    }
    for k,v in repl.items():
        txt = txt.replace(k,v)
    out = []
    for ch in txt:
        out.append(ch if ch.isalnum() else "_")
    s = "".join(out)
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")

# ------------------------------------------------------------
# Mapa cor -> índice térmico por cubo 32x32x32
# ------------------------------------------------------------
_PALETTE_CUBE_CACHE = {}

def build_palette_index_cube(palette_name, levels=32):
    palette_name = normalize_palette_name(palette_name)
    key = (palette_name, levels)
    if key in _PALETTE_CUBE_CACHE:
        return _PALETTE_CUBE_CACHE[key]

    lut = PALETTE_LUTS[palette_name][:, 0, :].astype(np.int16)  # BGR, 256 x 3
    vals = np.linspace(0, 255, levels).astype(np.int16)
    cube = np.zeros((levels, levels, levels), dtype=np.uint8)

    for bi, b in enumerate(vals):
        for gi, g in enumerate(vals):
            # Vetoriza em R
            colors = np.stack([
                np.full(levels, b, dtype=np.int16),
                np.full(levels, g, dtype=np.int16),
                vals
            ], axis=1)
            # dist: levels x 256
            dist = ((colors[:, None, :] - lut[None, :, :]) ** 2).sum(axis=2)
            cube[bi, gi, :] = np.argmin(dist, axis=1).astype(np.uint8)

    _PALETTE_CUBE_CACHE[key] = cube
    return cube

def frame_to_temperature_index(frame_bgr, palette_name):
    cube = build_palette_index_cube(palette_name)
    q = np.clip(frame_bgr.astype(np.uint16) // 8, 0, 31)
    idx = cube[q[..., 0], q[..., 1], q[..., 2]]
    return idx.astype(np.uint8)

def idx_to_temperature(idx, params):
    tmin = float(params["temp_min_c"])
    tmax = float(params["temp_max_c"])
    return tmin + (idx.astype(np.float32) / 255.0) * (tmax - tmin)

def apply_thermal_preprocess(frame_bgr, params):
    params = dict(params)
    params["palette_name"] = normalize_palette_name(params.get("palette_name"))

    blur_k = int(params.get("blur_ksize", 3))
    if blur_k > 1:
        if blur_k % 2 == 0:
            blur_k += 1
        frame_proc = cv2.GaussianBlur(frame_bgr, (blur_k, blur_k), 0)
    else:
        frame_proc = frame_bgr.copy()

    idx = frame_to_temperature_index(frame_proc, params["palette_name"])
    temp = idx_to_temperature(idx, params)

    low = float(params.get("temp_filter_min_c", 575.0))
    high = float(params.get("temp_filter_max_c", 1025.0))
    if high < low:
        low, high = high, low

    mask = ((temp >= low) & (temp <= high)).astype(np.uint8) * 255

    kernel = np.ones((3, 3), np.uint8)
    open_iter = int(params.get("open_iter", 0))
    close_iter = int(params.get("close_iter", 0))
    dilate_iter = int(params.get("dilate_iter", 0))

    if open_iter > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=open_iter)
    if close_iter > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)
    if dilate_iter > 0:
        mask = cv2.dilate(mask, kernel, iterations=dilate_iter)

    filtered = np.zeros_like(frame_bgr)
    filtered[mask > 0] = frame_bgr[mask > 0]

    return {
        "idx": idx,
        "temp": temp,
        "mask": mask,
        "filtered": filtered,
    }

# ------------------------------------------------------------
# Região, profundidade e trajetória
# ------------------------------------------------------------
def extract_hot_regions(frame_bgr, params):
    result = apply_thermal_preprocess(frame_bgr, params)
    mask = result["mask"]
    temp = result["temp"]

    n, labels, stats, centroids = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), connectivity=8)
    regs = []
    min_area = int(params.get("min_area", 300))
    max_area = int(params.get("max_area", 999999))

    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area or area > max_area:
            continue
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        comp = labels == i
        max_temp = float(temp[comp].max()) if comp.any() else 0.0
        mean_temp = float(temp[comp].mean()) if comp.any() else 0.0
        cx, cy = centroids[i]
        regs.append({
            "label": i,
            "bbox": (x, y, w, h),
            "area": area,
            "cx": float(cx),
            "cy": float(cy),
            "mask": comp,
            "max_temp": max_temp,
            "mean_temp": mean_temp,
            "depth_z": None,
        })

    if regs:
        amin = min(r["area"] for r in regs)
        amax = max(r["area"] for r in regs)
        tmin = min(r["max_temp"] for r in regs)
        tmax = max(r["max_temp"] for r in regs)
        for r in regs:
            area_norm = 1.0 if amax == amin else (r["area"] - amin) / max(1, amax - amin)
            temp_norm = 1.0 if tmax == tmin else (r["max_temp"] - tmin) / max(1e-6, tmax - tmin)
            r["priority"] = 0.65 * area_norm + 0.35 * temp_norm
        regs.sort(key=lambda r: r["priority"], reverse=True)

    result["regions"] = regs
    return result

def triangulate(pt1, pt2):
    p1 = np.array(pt1, dtype=float).reshape(2, 1)
    p2 = np.array(pt2, dtype=float).reshape(2, 1)
    p4d = cv2.triangulatePoints(P1, P2, p1, p2)
    if abs(float(p4d[3])) < 1e-9:
        return None
    p3d = (p4d[:3] / p4d[3]).flatten()
    return p3d

def assign_depths(regs1, regs2):
    used = set()
    for r1 in regs1:
        best_j = None
        best_d = 1e9
        for j, r2 in enumerate(regs2):
            if j in used:
                continue
            # Após retificação, regiões correspondentes tendem a ter y parecido.
            d = abs(r1["cy"] - r2["cy"]) + 0.20 * abs(r1["cx"] - r2["cx"])
            if d < best_d:
                best_d = d
                best_j = j
        if best_j is not None:
            used.add(best_j)
            p3d = triangulate((r1["cx"], r1["cy"]), (regs2[best_j]["cx"], regs2[best_j]["cy"]))
            if p3d is not None:
                r1["depth_z"] = float(p3d[2])
    return regs1

def make_region_route(region, params):
    mask = region["mask"]
    x, y, w, h = region["bbox"]
    area = int(region["area"])
    large_thr = int(params.get("large_region_area", 2500))
    step = max(3, int(params.get("route_step_px", 18)))
    offset_y = int(params.get("offset_y_px", 8))
    dwell = max(1, int(params.get("centroid_dwell_points", 4)))

    if area >= large_thr:
        # Estratégia de offset da borda inferior: percorre a base da região quente.
        pts = []
        x0, x1 = x, x + w - 1
        for xx in range(x0, x1 + 1, step):
            col = mask[:, xx]
            ys = np.where(col)[0]
            if len(ys) > 0:
                yy = int(np.max(ys) + offset_y)
                yy = int(np.clip(yy, 0, mask.shape[0] - 1))
                pts.append((int(xx), yy))
        if not pts:
            pts = [(int(region["cx"]), int(region["cy"]))]
        mode = "offset_borda_inferior"
    else:
        # Estratégia por centróide: mantém alguns pontos no centro do foco.
        c = (int(region["cx"]), int(region["cy"]))
        pts = [c for _ in range(dwell)]
        mode = "centroide"

    return pts, mode

def plan_full_route(regions, params, human_boxes=None):
    human_boxes = human_boxes or []
    margin = int(params.get("human_safety_margin_px", 30))
    route = []
    route_meta = []

    def inside_human(pt):
        px, py = pt
        for (x, y, w, h) in human_boxes:
            if (x - margin) <= px <= (x + w + margin) and (y - margin) <= py <= (y + h + margin):
                return True
        return False

    for r in regions:
        pts, mode = make_region_route(r, params)
        safe_pts = [p for p in pts if not inside_human(p)]
        if not safe_pts:
            continue
        route.extend(safe_pts)
        route_meta.extend([{"mode": mode, "region": r} for _ in safe_pts])
    return route, route_meta

def compensate_aim_for_jet(impact_pt, depth_z, params):
    x, y = impact_pt
    pressure = max(1e-6, float(params.get("jet_pressure_calib", 22.0)))
    z = abs(float(depth_z)) if depth_z is not None and np.isfinite(depth_z) else 1.0
    # Modelo simples: quanto maior a distância e menor a pressão, maior a compensação vertical.
    drop_px = (z * z) / pressure
    aim_y = int(round(y - drop_px))
    return (int(round(x)), int(np.clip(aim_y, 0, 10000))), float(drop_px)

def move_point_towards(current, target, max_step):
    cx, cy = current
    tx, ty = target
    dx, dy = tx - cx, ty - cy
    d = float(np.hypot(dx, dy))
    if d <= max_step or d < 1e-6:
        return (int(tx), int(ty)), True
    s = max_step / d
    return (int(round(cx + dx * s)), int(round(cy + dy * s))), False

# ------------------------------------------------------------
# Câmera, retificação, visual e Arduino
# ------------------------------------------------------------
def abrir_camera(index):
    cap = cv2.VideoCapture(index)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAP_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAP_HEIGHT)
    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir a câmera {index}.")
    ok, frame = cap.read()
    if not ok or frame is None:
        cap.release()
        raise RuntimeError(f"A câmera {index} abriu, mas não retornou imagem.")
    return cap, frame

def create_rectification_maps(frame_shape):
    h, w = frame_shape[:2]
    map1x, map1y = cv2.initUndistortRectifyMap(K1, D1, R1, P1, (w, h), cv2.CV_32FC1)
    map2x, map2y = cv2.initUndistortRectifyMap(K2, D2, R2, P2, (w, h), cv2.CV_32FC1)
    return map1x, map1y, map2x, map2y

def maybe_rectify(frame1, frame2, maps, params):
    if not bool(params.get("use_rectification", False)):
        return frame1, frame2
    map1x, map1y, map2x, map2y = maps
    return (
        cv2.remap(frame1, map1x, map1y, cv2.INTER_LINEAR),
        cv2.remap(frame2, map2x, map2y, cv2.INTER_LINEAR),
    )

def draw_regions(frame, regions, color=(0, 255, 0)):
    out = frame.copy()
    for i, r in enumerate(regions, start=1):
        x, y, w, h = r["bbox"]
        cv2.rectangle(out, (x, y), (x+w, y+h), color, 2)
        z = r.get("depth_z", None)
        ztxt = "Z=?" if z is None else f"Z={z:.2f}"
        label = f"#{i} A={r['area']} T={r['max_temp']:.0f}C {ztxt}"
        cv2.putText(out, label, (x, max(20, y-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return out

def draw_trajectory_panel(base_frame, regions, route, aim_pt=None, impact_pt=None, route_index=0):
    panel = base_frame.copy()
    for r in regions:
        x, y, w, h = r["bbox"]
        cv2.rectangle(panel, (x, y), (x+w, y+h), (0, 180, 255), 1)

    if len(route) > 1:
        pts = np.array(route, dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(panel, [pts], False, (255, 0, 255), 2)
        for k, p in enumerate(route):
            c = (255, 0, 255) if k >= route_index else (80, 80, 80)
            cv2.circle(panel, p, 3, c, -1)

    if impact_pt is not None:
        cv2.circle(panel, impact_pt, 8, (0, 255, 255), 2)
        cv2.putText(panel, "impacto previsto", (impact_pt[0]+8, impact_pt[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,255), 1)
    if aim_pt is not None:
        cv2.drawMarker(panel, aim_pt, (0, 0, 255), cv2.MARKER_CROSS, 18, 2)
        cv2.putText(panel, "mira", (aim_pt[0]+8, max(15, aim_pt[1]-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0,0,255), 1)

    cv2.putText(panel, "trajetoria planejada do jato", (12, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,0), 2)
    return panel

def hstack_same_height(a, b):
    h = min(a.shape[0], b.shape[0])
    def resize_h(img):
        if img.shape[0] == h:
            return img
        scale = h / img.shape[0]
        return cv2.resize(img, (int(img.shape[1]*scale), h))
    return cv2.hconcat([resize_h(a), resize_h(b)])

# Arduino opcional
_serial_handle = None

def get_arduino(params):
    global _serial_handle
    if not bool(params.get("enable_arduino", False)):
        return None
    if _serial_handle is not None:
        return _serial_handle
    try:
        import serial
        _serial_handle = serial.Serial(str(params.get("serial_port", "/dev/ttyACM0")), int(params.get("serial_baud", 9600)), timeout=1)
        time.sleep(2.0)
        print("Arduino conectado.")
    except Exception as e:
        print("Arduino não conectado:", e)
        _serial_handle = None
    return _serial_handle

def send_aim_to_arduino(aim_pt, frame_shape, params):
    ard = get_arduino(params)
    if ard is None or aim_pt is None:
        return
    h, w = frame_shape[:2]
    x, y = aim_pt
    ax = np.interp(x, [0, w-1], [float(params.get("servo_x_min",20)), float(params.get("servo_x_max",160))])
    ay = np.interp(y, [0, h-1], [float(params.get("servo_y_min",20)), float(params.get("servo_y_max",160))])
    cmd = f"{int(round(ax))},{int(round(ay))}\n"
    try:
        ard.write(cmd.encode("utf-8"))
    except Exception as e:
        print("Falha ao enviar para Arduino:", e)

# YOLO humano opcional
_yolo_model = None

def detect_humans(frame_bgr, params):
    global _yolo_model
    if not bool(params.get("enable_yolo_human_safety", False)):
        return []
    try:
        from ultralytics import YOLO
        if _yolo_model is None:
            _yolo_model = YOLO("yolov8n.pt")
        res = _yolo_model.predict(frame_bgr, verbose=False, conf=0.35)[0]
        boxes = []
        for b in res.boxes:
            cls = int(b.cls[0])
            # classe 0 no COCO = pessoa
            if cls != 0:
                continue
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().astype(int).tolist()
            boxes.append((x1, y1, x2-x1, y2-y1))
        return boxes
    except Exception as e:
        # Mantém execução mesmo sem ultralytics instalado.
        return []

def draw_humans(frame, boxes, margin=30):
    out = frame.copy()
    for x, y, w, h in boxes:
        cv2.rectangle(out, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.rectangle(out, (x-margin, y-margin), (x+w+margin, y+h+margin), (255, 120, 0), 1)
        cv2.putText(out, "HUMANO - zona bloqueada", (x, max(20, y-7)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,0,0), 1)
    return out

print("Funções carregadas.")


Calibração carregada: vision/fire_detect/jet_automation/stereo_calibration2.npz
K1: (3, 3) P1: (3, 4)
Funções carregadas.


In [3]:
# ============================================================
# CÉLULA 3 - Configuração compacta dos filtros e trajetória
# Versão revisada: simulação fluida inspirada no notebook firefighting_simulator
# ============================================================

import cv2
import json
import time
import asyncio
import numpy as np
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# Pastas
# ------------------------------------------------------------
if "BASE_DIR" not in globals():
    PROJECT_ROOT = localizar_raiz_blaze()
    BASE_DIR = (
        PROJECT_ROOT
        / "vision"
        / "fire_detect"
        / "jet_automation"
    )

DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
PARAMS_DIR = BASE_DIR / "parameters_setups"
PARAMS_DIR.mkdir(parents=True, exist_ok=True)
THERMAL_SETUPS_FILE = PARAMS_DIR / "thermal_preprocess_setups.json"

# ------------------------------------------------------------
# Funções auxiliares de escala térmica
# ------------------------------------------------------------
def criar_lut_por_pontos(pontos_rgb, n=256):
    pontos_rgb = sorted(pontos_rgb, key=lambda x: x[0])
    lut = np.zeros((n, 1, 3), dtype=np.uint8)
    xs = np.array([p[0] for p in pontos_rgb])
    rs = np.array([p[1][0] for p in pontos_rgb])
    gs = np.array([p[1][1] for p in pontos_rgb])
    bs = np.array([p[1][2] for p in pontos_rgb])
    x_new = np.linspace(0, 1, n)
    lut[:, 0, 0] = np.interp(x_new, xs, bs).astype(np.uint8)
    lut[:, 0, 1] = np.interp(x_new, xs, gs).astype(np.uint8)
    lut[:, 0, 2] = np.interp(x_new, xs, rs).astype(np.uint8)
    return lut

PALETTE_LUTS = {
    "1. Arco-íris / Rainbow": criar_lut_por_pontos([
        (0.00, (90, 0, 255)), (0.20, (0, 80, 255)), (0.40, (0, 255, 255)),
        (0.60, (0, 255, 0)), (0.80, (255, 255, 0)), (1.00, (255, 0, 0)),
    ]),
    "2. Inferno / Plasma térmico": criar_lut_por_pontos([
        (0.00, (20, 0, 60)), (0.25, (100, 0, 120)), (0.50, (220, 0, 80)),
        (0.75, (255, 150, 0)), (1.00, (255, 255, 180)),
    ]),
    "3. Cinza / Grayscale": criar_lut_por_pontos([
        (0.00, (0, 0, 0)), (1.00, (255, 255, 255)),
    ]),
    "4. Verde-amarelo térmico": criar_lut_por_pontos([
        (0.00, (40, 20, 0)), (0.30, (120, 80, 0)), (0.60, (180, 220, 0)),
        (0.85, (240, 255, 120)), (1.00, (255, 255, 245)),
    ]),
    "5. Preto-roxo-vermelho-branco": criar_lut_por_pontos([
        (0.00, (0, 0, 0)), (0.20, (60, 0, 120)), (0.45, (120, 0, 0)),
        (0.70, (255, 0, 0)), (0.90, (255, 180, 180)), (1.00, (255, 255, 255)),
    ]),
    "6. Branco-amarelo-vermelho-escuro": criar_lut_por_pontos([
        (0.00, (255, 255, 255)), (0.25, (255, 255, 0)), (0.45, (255, 130, 0)),
        (0.65, (255, 0, 0)), (0.85, (130, 0, 0)), (1.00, (30, 0, 0)),
    ]),
    "7. Preto-branco-amarelo-vermelho-escuro": criar_lut_por_pontos([
        (0.00, (0, 0, 0)), (0.16, (255, 255, 255)), (0.38, (210, 255, 0)),
        (0.58, (255, 255, 0)), (0.76, (255, 60, 0)), (1.00, (120, 0, 0)),
    ]),
}
PALETTE_NAMES = list(PALETTE_LUTS.keys())
PALETTE_ALIASES = {
    "CUSTOM_branco_amarelo_vermelho_escuro": "6. Branco-amarelo-vermelho-escuro",
    "branco_amarelo_vermelho_escuro": "6. Branco-amarelo-vermelho-escuro",
    "azul_verde_amarelo_vermelho": "1. Arco-íris / Rainbow",
    "jet_opencv": "1. Arco-íris / Rainbow",
    "inferno_opencv": "2. Inferno / Plasma térmico",
    "cinza_preto_branco": "3. Cinza / Grayscale",
}

def normalize_palette_name(name):
    name = PALETTE_ALIASES.get(str(name), str(name))
    if name not in PALETTE_LUTS:
        name = "7. Preto-branco-amarelo-vermelho-escuro"
    return name

# ------------------------------------------------------------
# Conversão cor -> índice térmico pela paleta
# ------------------------------------------------------------
_PALETTE_CUBE_CACHE = {}

def build_palette_index_cube(palette_name, levels=32):
    palette_name = normalize_palette_name(palette_name)
    key = (palette_name, levels)
    if key in _PALETTE_CUBE_CACHE:
        return _PALETTE_CUBE_CACHE[key]
    lut = PALETTE_LUTS[palette_name][:, 0, :].astype(np.int16)
    vals = np.linspace(0, 255, levels).astype(np.int16)
    cube = np.zeros((levels, levels, levels), dtype=np.uint8)
    for bi, b in enumerate(vals):
        for gi, g in enumerate(vals):
            colors = np.stack([
                np.full(levels, b, dtype=np.int16),
                np.full(levels, g, dtype=np.int16),
                vals,
            ], axis=1)
            dist = ((colors[:, None, :] - lut[None, :, :]) ** 2).sum(axis=2)
            cube[bi, gi, :] = np.argmin(dist, axis=1).astype(np.uint8)
    _PALETTE_CUBE_CACHE[key] = cube
    return cube


def frame_to_temperature_index(frame_bgr, palette_name):
    palette_name = normalize_palette_name(palette_name)
    cube = build_palette_index_cube(palette_name)
    q = np.clip(frame_bgr.astype(np.uint16) // 8, 0, 31)
    return cube[q[..., 0], q[..., 1], q[..., 2]].astype(np.uint8)


def idx_to_temperature(idx, params):
    tmin = float(params["temp_min_c"])
    tmax = float(params["temp_max_c"])
    if tmax <= tmin:
        tmax = tmin + 1.0
    return tmin + (idx.astype(np.float32) / 255.0) * (tmax - tmin)


def palette_slice_hsv_mask(frame_bgr, palette_name, idx_min, idx_max, sat_floor=45, hue_tol=8, val_tol=55):
    palette_name = normalize_palette_name(palette_name)
    lut_bgr = PALETTE_LUTS[palette_name][:, 0, :]
    lo = int(min(idx_min, idx_max))
    hi = int(max(idx_min, idx_max))
    sel_bgr = lut_bgr[lo:hi + 1]
    if len(sel_bgr) == 0:
        return np.zeros(frame_bgr.shape[:2], dtype=np.uint8)
    sel_hsv = cv2.cvtColor(sel_bgr.reshape(-1, 1, 3), cv2.COLOR_BGR2HSV).reshape(-1, 3)
    sel_hsv = sel_hsv[sel_hsv[:, 1] >= 30]
    if len(sel_hsv) == 0:
        return np.zeros(frame_bgr.shape[:2], dtype=np.uint8)
    sample_step = max(1, len(sel_hsv) // 24)
    sample_hsv = sel_hsv[::sample_step]
    frame_hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    h = frame_hsv[..., 0].astype(np.int16)
    s = frame_hsv[..., 1].astype(np.int16)
    v = frame_hsv[..., 2].astype(np.int16)
    sat_min = max(20, int(sample_hsv[:, 1].min()) - 35, sat_floor)
    val_min = max(0, int(sample_hsv[:, 2].min()) - val_tol)
    val_max = min(255, int(sample_hsv[:, 2].max()) + val_tol)
    hue_mask = np.zeros(frame_bgr.shape[:2], dtype=bool)
    for hh, _, _ in sample_hsv:
        diff = np.abs(h - int(hh))
        diff = np.minimum(diff, 180 - diff)
        hue_mask |= (diff <= hue_tol)
    mask = hue_mask & (s >= sat_min) & (v >= val_min) & (v <= val_max)
    return (mask.astype(np.uint8) * 255)

# ------------------------------------------------------------
# Helpers de exibição
# ------------------------------------------------------------
def gerar_gradiente_lut(lut, width=420, height=42):
    base = np.linspace(0, 255, width, dtype=np.uint8)
    base = np.tile(base, (height, 1))
    return cv2.applyColorMap(base, lut)


def encode_png_bgr(img_bgr):
    ok, buffer = cv2.imencode('.png', img_bgr)
    if not ok:
        raise RuntimeError('Falha ao codificar PNG.')
    return buffer.tobytes()


def encode_jpg_bgr(img_bgr, quality=88):
    ok, buffer = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        raise RuntimeError('Falha ao codificar JPG.')
    return buffer.tobytes()

# ------------------------------------------------------------
# Segmentação térmica
# ------------------------------------------------------------
def apply_thermal_preprocess(frame_bgr, params):
    params = dict(params)
    params['palette_name'] = normalize_palette_name(params.get('palette_name'))
    blur_k = int(params.get('blur_ksize', 3))
    if blur_k > 1:
        if blur_k % 2 == 0:
            blur_k += 1
        frame_proc = cv2.GaussianBlur(frame_bgr, (blur_k, blur_k), 0)
    else:
        frame_proc = frame_bgr.copy()

    idx = frame_to_temperature_index(frame_proc, params['palette_name'])
    temp = idx_to_temperature(idx, params)

    low = float(params.get('temp_filter_min_c', 575.0))
    high = float(params.get('temp_filter_max_c', 1025.0))
    if high < low:
        low, high = high, low

    tmin = float(params.get('temp_min_c', 300.0))
    tmax = float(params.get('temp_max_c', 1025.0))
    idx_min = int(np.clip(255 * (low - tmin) / max(1e-6, tmax - tmin), 0, 255))
    idx_max = int(np.clip(255 * (high - tmin) / max(1e-6, tmax - tmin), 0, 255))

    mask_temp = ((temp >= low) & (temp <= high)).astype(np.uint8) * 255
    mask_color = palette_slice_hsv_mask(frame_proc, params['palette_name'], idx_min, idx_max, sat_floor=45, hue_tol=9, val_tol=60)
    hsv = cv2.cvtColor(frame_proc, cv2.COLOR_BGR2HSV)
    low_sat_mask = (hsv[..., 1] < 28).astype(np.uint8) * 255
    mask = cv2.bitwise_or(mask_temp, mask_color)
    if params['palette_name'] != '3. Cinza / Grayscale':
        mask[low_sat_mask > 0] = 0

    kernel = np.ones((3, 3), np.uint8)
    open_iter = int(params.get('open_iter', 0))
    close_iter = int(params.get('close_iter', 0))
    dilate_iter = int(params.get('dilate_iter', 0))
    if open_iter > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=open_iter)
    if close_iter > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)
    if dilate_iter > 0:
        mask = cv2.dilate(mask, kernel, iterations=dilate_iter)

    filtered = np.zeros_like(frame_bgr)
    filtered[mask > 0] = frame_bgr[mask > 0]
    return {'idx': idx, 'temp': temp, 'mask': mask, 'filtered': filtered}


def region_components(mask, min_area=15):
    mask_u8 = (mask > 0).astype(np.uint8)
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    regs = []
    for idx in range(1, n_labels):
        area = int(stats[idx, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        x = int(stats[idx, cv2.CC_STAT_LEFT])
        y = int(stats[idx, cv2.CC_STAT_TOP])
        w = int(stats[idx, cv2.CC_STAT_WIDTH])
        h = int(stats[idx, cv2.CC_STAT_HEIGHT])
        comp = labels == idx
        regs.append({
            'label': idx,
            'mask': comp,
            'area': area,
            'cx': float(centroids[idx][0]),
            'cy': float(centroids[idx][1]),
            'bbox': (x, y, w, h),
            'bbox_prod': int(w * h),
        })
    regs.sort(key=lambda r: r['area'], reverse=True)
    return regs


def extract_hot_regions(frame_bgr, params):
    result = apply_thermal_preprocess(frame_bgr, params)
    regs = region_components(result['mask'], min_area=max(10, int(params.get('min_area', 300))))
    result['regions'] = regs
    return result


def aplicar_preprocessamento(img_bgr, params):
    result = extract_hot_regions(img_bgr, params)
    mask = result['mask']
    filtered = result['filtered']
    regions = result['regions']
    overlay = img_bgr.copy()
    for r in regions:
        x, y, w, h = r['bbox']
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(overlay, f"A={int(r['area'])}", (x, max(18, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1, cv2.LINE_AA)
    mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    painel = np.hstack([
        cv2.resize(img_bgr, (360, 270)),
        cv2.resize(filtered, (360, 270)),
        cv2.resize(mask_bgr, (360, 270)),
        cv2.resize(overlay, (360, 270)),
    ])
    cor = (0,255,0)
    cv2.putText(painel, 'Original', (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor, 2)
    cv2.putText(painel, 'Filtro', (375, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor, 2)
    cv2.putText(painel, 'Mascara', (735, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor, 2)
    cv2.putText(painel, 'Regioes validas', (1095, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor, 2)
    return mask, regions, painel

# ------------------------------------------------------------
# Setups
# ------------------------------------------------------------
def nome_seguro(texto):
    texto = str(texto).lower()
    trocas = {'á':'a','à':'a','ã':'a','â':'a','é':'e','ê':'e','í':'i','ó':'o','ô':'o','õ':'o','ú':'u','ç':'c','/':'','.':'','-':'_',' ':'_'}
    for a,b in trocas.items():
        texto = texto.replace(a,b)
    texto = ''.join(c for c in texto if c.isalnum() or c == '_')
    while '__' in texto:
        texto = texto.replace('__','_')
    return texto.strip('_')


def carregar_setups():
    if THERMAL_SETUPS_FILE.exists():
        try:
            with open(THERMAL_SETUPS_FILE, 'r', encoding='utf-8') as f:
                setups = json.load(f)
            if isinstance(setups, dict):
                return setups
        except Exception as e:
            print('Aviso ao carregar setups:', e)
    return {}


def salvar_setups(setups):
    with open(THERMAL_SETUPS_FILE, 'w', encoding='utf-8') as f:
        json.dump(setups, f, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# Imagens disponíveis
# ------------------------------------------------------------
image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.webp']
image_files = []
for ext in image_extensions:
    image_files.extend(DATA_DIR.glob(ext))
image_files = sorted(image_files)
image_options = ['Nenhuma imagem encontrada'] if len(image_files) == 0 else [p.name for p in image_files]

# ------------------------------------------------------------
# Parâmetros padrão
# ------------------------------------------------------------
DEFAULT_PREPROCESS = {
    'palette_name': '7. Preto-branco-amarelo-vermelho-escuro',
    'temp_min_c': 300,
    'temp_max_c': 1025,
    'temp_filter_min_c': 750,
    'temp_filter_max_c': 1025,
    'blur_ksize': 3,
    'open_iter': 1,
    'close_iter': 1,
    'dilate_iter': 0,
    'min_area': 300,
    'max_area': 999999,
    'use_rectification': False,
    'area_threshold_offset': 2500,
    'offset_y': 8,
    'route_step_px': 18,
    'centroid_hold_frames': 4,
    'aim_speed_px': 15,
    'erase_radius': 28,
    'enable_human_yolo': False,
    'human_margin_px': 30,
    'enable_arduino': False,
    'serial_port': '/dev/ttyACM0',
    'serial_baud': 9600,
    'servo_x_min': 20,
    'servo_x_max': 160,
    'servo_y_min': 20,
    'servo_y_max': 160,
}
CURRENT_PREPROCESS_PARAMS = DEFAULT_PREPROCESS.copy()

# ------------------------------------------------------------
# Estado da simulação fluida
# ------------------------------------------------------------
SIM_STATE = {
    'base_fire': None,
    'fire_canvas': None,
    'route': [],
    'route_index': 0,
    'mode': 'borda',
    'movement_phase': 'aproximação/entre trajetórias',
    'regions': [],
    'frame': 0,
    'running': False,
    'animation_task': None,
    'x': 26,
    'y': 26,
    'pause_until_frame': -1,
    'current_target_region': None,
}

# ------------------------------------------------------------
# Widgets auxiliares
# ------------------------------------------------------------
def slider_longo(value, min_value, max_value, step, label, explanation, is_float=False):
    label_html = widgets.HTML(value=f"<div style='width:245px; line-height:1.15;'><b>{label}</b><br><span style='font-size:11px; color:#777;'>({explanation})</span></div>")
    if is_float:
        slider = widgets.FloatSlider(value=value, min=min_value, max=max_value, step=step, readout=True, continuous_update=False, layout=widgets.Layout(width='300px'))
    else:
        slider = widgets.IntSlider(value=value, min=min_value, max=max_value, step=step, readout=True, continuous_update=False, layout=widgets.Layout(width='300px'))
    return widgets.HBox([label_html, slider]), slider


def checkbox_longo(value, label, explanation):
    cb = widgets.Checkbox(value=value, description='', indent=False, layout=widgets.Layout(width='24px'))
    label_html = widgets.HTML(value=f"<div style='width:500px; line-height:1.15;'><b>{label}</b> <span style='font-size:11px; color:#777;'>({explanation})</span></div>")
    return widgets.HBox([cb, label_html]), cb

# ------------------------------------------------------------
# Referência visual das escalas
# ------------------------------------------------------------
palette_reference_rows = []
for name in PALETTE_NAMES:
    grad = gerar_gradiente_lut(PALETTE_LUTS[name], width=360, height=36)
    img_widget = widgets.Image(value=encode_png_bgr(grad), format='png', width=360)
    row = widgets.HBox([widgets.HTML(value=f"<div style='width:335px;'><b>{name}</b></div>"), img_widget])
    palette_reference_rows.append(row)

palette_reference_box = widgets.VBox([
    widgets.HTML("<h3>Referência das escalas térmicas</h3>"),
    widgets.HTML("<p>A escala 7 considera <b>preto</b> como frio, <b>branco</b> como segunda faixa fria e <b>vermelho escuro</b> como mais quente.</p>"),
    *palette_reference_rows,
])

# ------------------------------------------------------------
# Widgets principais
# ------------------------------------------------------------
dropdown_image = widgets.Dropdown(options=image_options, value=image_options[0], description='Imagem:', layout=widgets.Layout(width='620px'))
dropdown_palette = widgets.Dropdown(options=PALETTE_NAMES, value=DEFAULT_PREPROCESS['palette_name'], description='Escala:', layout=widgets.Layout(width='620px'))
setup_extra_name = widgets.Text(value='base', description='Nome extra:', placeholder='ex.: base, bancada, ensaio_01', layout=widgets.Layout(width='620px'))
button_save_setup = widgets.Button(description='Salvar setup atual', button_style='success', layout=widgets.Layout(width='220px'))
button_load_setup = widgets.Button(description='Carregar setup salvo', button_style='info', layout=widgets.Layout(width='220px'))
button_delete_setup = widgets.Button(description='Excluir setup salvo', button_style='danger', layout=widgets.Layout(width='220px'))
saved_setups = carregar_setups()
setup_dropdown = widgets.Dropdown(options=['Nenhum setup salvo'] if len(saved_setups) == 0 else sorted(saved_setups.keys()), description='Setup:', layout=widgets.Layout(width='620px'))

w_temp_min, s_temp_min = slider_longo(DEFAULT_PREPROCESS['temp_min_c'], 0, 1500, 5, 'Temp. mín. escala', 'menor temperatura representada no gradiente')
w_temp_max, s_temp_max = slider_longo(DEFAULT_PREPROCESS['temp_max_c'], 0, 1500, 5, 'Temp. máx. escala', 'maior temperatura representada no gradiente')
w_filter_min, s_filter_min = slider_longo(DEFAULT_PREPROCESS['temp_filter_min_c'], 0, 1500, 5, 'Filtro mín.', 'temperatura mínima aceita como foco quente')
w_filter_max, s_filter_max = slider_longo(DEFAULT_PREPROCESS['temp_filter_max_c'], 0, 1500, 5, 'Filtro máx.', 'temperatura máxima aceita como foco quente')
w_blur, s_blur = slider_longo(DEFAULT_PREPROCESS['blur_ksize'], 1, 21, 2, 'Blur', 'suaviza ruído antes da segmentação')
w_open, s_open = slider_longo(DEFAULT_PREPROCESS['open_iter'], 0, 8, 1, 'Abertura', 'remove pequenos ruídos isolados')
w_close, s_close = slider_longo(DEFAULT_PREPROCESS['close_iter'], 0, 8, 1, 'Fechamento', 'fecha falhas dentro das regiões')
w_dilate, s_dilate = slider_longo(DEFAULT_PREPROCESS['dilate_iter'], 0, 8, 1, 'Dilatação', 'expande a máscara filtrada')
w_min_area, s_min_area = slider_longo(DEFAULT_PREPROCESS['min_area'], 0, 20000, 50, 'Área mín.', 'descarta focos pequenos demais')
w_max_area, s_max_area = slider_longo(DEFAULT_PREPROCESS['max_area'], 1000, 1000000, 1000, 'Área máx.', 'descarta regiões grandes demais')
w_rectify, c_rectify = checkbox_longo(DEFAULT_PREPROCESS['use_rectification'], 'Usar retificação', 'mantido para as células seguintes do pipeline')

w_area_offset, s_area_offset = slider_longo(DEFAULT_PREPROCESS['area_threshold_offset'], 0, 50000, 100, 'Área p/ offset', 'acima deste valor usa estratégia por borda inferior')
w_offset_y, s_offset_y = slider_longo(DEFAULT_PREPROCESS['offset_y'], -80, 80, 1, 'Offset Y', 'deslocamento vertical aplicado na borda inferior')
w_route_step, s_route_step = slider_longo(DEFAULT_PREPROCESS['route_step_px'], 2, 80, 1, 'Passo rota', 'distância entre amostras consecutivas da trajetória')
w_hold, s_hold = slider_longo(DEFAULT_PREPROCESS['centroid_hold_frames'], 1, 60, 1, 'Perman. centróide', 'tempo de permanência em focos pequenos')
w_aim_speed, s_aim_speed = slider_longo(DEFAULT_PREPROCESS['aim_speed_px'], 1, 80, 1, 'Vel. mira', 'velocidade da mira nas aproximações e transições')
w_erase, s_erase = slider_longo(DEFAULT_PREPROCESS['erase_radius'], 1, 120, 1, 'Raio apag.', 'raio circular apagado pela atuação do jato')
w_human_margin, s_human_margin = slider_longo(DEFAULT_PREPROCESS['human_margin_px'], 0, 200, 5, 'Margem humano', 'mantido para a lógica futura de segurança')
w_yolo, c_yolo = checkbox_longo(DEFAULT_PREPROCESS['enable_human_yolo'], 'Ativar trava YOLO humano', 'mantido para as células seguintes')
w_arduino, c_arduino = checkbox_longo(DEFAULT_PREPROCESS['enable_arduino'], 'Enviar mira ao Arduino', 'mantido para as células seguintes')
serial_port_text = widgets.Text(value=DEFAULT_PREPROCESS['serial_port'], description='Porta serial:', layout=widgets.Layout(width='620px'))
serial_baud_text = widgets.IntText(value=DEFAULT_PREPROCESS['serial_baud'], description='Baud:', layout=widgets.Layout(width='300px'))

button_reset_fire = widgets.Button(description='Resetar incêndio', button_style='warning', layout=widgets.Layout(width='180px'))
button_play_fire = widgets.Button(description='Play', button_style='primary', layout=widgets.Layout(width='110px'))
button_stop_fire = widgets.Button(description='Stop', button_style='', layout=widgets.Layout(width='110px'))
slider_sim_delay = widgets.FloatSlider(value=0.05, min=0.02, max=0.30, step=0.01, description='Intervalo:', readout_format='.2f', continuous_update=False, layout=widgets.Layout(width='340px'))

coluna_esquerda = widgets.VBox([
    widgets.HTML('<h4>1) Temperatura e segmentação</h4>'),
    w_temp_min, w_temp_max, w_filter_min, w_filter_max, w_blur, w_open, w_close, w_dilate, w_min_area, w_max_area, w_rectify,
])
coluna_direita = widgets.VBox([
    widgets.HTML('<h4>2) Trajetória, jato e segurança</h4>'),
    w_area_offset, w_offset_y, w_route_step, w_hold, w_aim_speed,  w_erase, w_yolo, w_human_margin, w_arduino, serial_port_text, serial_baud_text,
])
parametros_2_colunas = widgets.HBox([coluna_esquerda, coluna_direita])

# Widgets persistentes de imagem (evitam flicker)
gradient_image_widget = widgets.Image(format='png', width=620)
preview_image_widget = widgets.Image(format='jpg', width=1200)
sim_image_widget = widgets.Image(format='jpg', width=780)
status_html = widgets.HTML()

# ------------------------------------------------------------
# Leitura dos parâmetros atuais
# ------------------------------------------------------------
def coletar_params():
    temp_min = int(s_temp_min.value)
    temp_max = int(s_temp_max.value)
    if temp_max <= temp_min:
        temp_max = temp_min + 5
        s_temp_max.value = temp_max
    filter_min = int(s_filter_min.value)
    filter_max = int(s_filter_max.value)
    if filter_max <= filter_min:
        filter_max = filter_min + 5
        s_filter_max.value = filter_max
    blur = int(s_blur.value)
    if blur < 1:
        blur = 1
    if blur % 2 == 0:
        blur += 1
        s_blur.value = blur
    return {
        'palette_name': normalize_palette_name(dropdown_palette.value),
        'temp_min_c': temp_min,
        'temp_max_c': temp_max,
        'temp_filter_min_c': filter_min,
        'temp_filter_max_c': filter_max,
        'blur_ksize': blur,
        'open_iter': int(s_open.value),
        'close_iter': int(s_close.value),
        'dilate_iter': int(s_dilate.value),
        'min_area': int(s_min_area.value),
        'max_area': int(s_max_area.value),
        'use_rectification': bool(c_rectify.value),
        'area_threshold_offset': int(s_area_offset.value),
        'offset_y': int(s_offset_y.value),
        'route_step_px': int(s_route_step.value),
        'centroid_hold_frames': int(s_hold.value),
        'aim_speed_px': int(s_aim_speed.value),
        'erase_radius': int(s_erase.value),
        'enable_human_yolo': bool(c_yolo.value),
        'human_margin_px': int(s_human_margin.value),
        'enable_arduino': bool(c_arduino.value),
        'serial_port': serial_port_text.value,
        'serial_baud': int(serial_baud_text.value),
        'servo_x_min': DEFAULT_PREPROCESS['servo_x_min'],
        'servo_x_max': DEFAULT_PREPROCESS['servo_x_max'],
        'servo_y_min': DEFAULT_PREPROCESS['servo_y_min'],
        'servo_y_max': DEFAULT_PREPROCESS['servo_y_max'],
    }

# ------------------------------------------------------------
# Estratégias de trajetória (inspiradas no notebook anexo)
# ------------------------------------------------------------
def move_point_towards(tx, ty, x, y, max_step=10, w_lim=None, h_lim=None):
    dx = tx - x
    dy = ty - y
    dist = float(np.hypot(dx, dy))
    if dist <= max_step or dist == 0:
        nx = int(round(tx))
        ny = int(round(ty))
        if w_lim is not None:
            nx = int(np.clip(nx, 0, w_lim - 1))
        if h_lim is not None:
            ny = int(np.clip(ny, 0, h_lim - 1))
        return nx, ny, True
    s = max_step / dist
    nx = int(round(x + dx * s))
    ny = int(round(y + dy * s))
    if w_lim is not None:
        nx = int(np.clip(nx, 0, w_lim - 1))
    if h_lim is not None:
        ny = int(np.clip(ny, 0, h_lim - 1))
    return nx, ny, False


def nearest_neighbor_route(points, start_xy):
    pts = [(int(p[0]), int(p[1])) for p in points]
    if not pts:
        return []
    unused = pts.copy()
    ordered = []
    current = (int(start_xy[0]), int(start_xy[1]))
    while unused:
        idx = int(np.argmin([np.hypot(current[0] - p[0], current[1] - p[1]) for p in unused]))
        nxt = unused.pop(idx)
        ordered.append(nxt)
        current = nxt
    return ordered


def build_bottom_route(mask, start_xy, step_px=18, y_offset=4):
    Hm, Wm = mask.shape[:2]
    ys, xs = np.where(mask)
    if xs.size == 0:
        return []
    x_left, x_right = int(xs.min()), int(xs.max())
    step_px = max(2, int(step_px))
    anchors_x = list(range(x_left, x_right + 1, step_px))
    if len(anchors_x) == 0 or anchors_x[-1] != x_right:
        anchors_x.append(x_right)
    route = []
    used = set()
    for xq in anchors_x:
        best = None
        best_key = None
        for dx in range(-12, 13):
            xc = int(np.clip(xq + dx, 0, Wm - 1))
            ys_col = np.where(mask[:, xc])[0]
            if ys_col.size == 0:
                continue
            y_bottom = int(ys_col.max())
            y_target = y_bottom
            if y_offset > 0:
                ys_valid = ys_col[ys_col <= y_bottom - int(y_offset)]
                if ys_valid.size > 0:
                    y_target = int(ys_valid.max())
            key = (abs(dx), -y_bottom)
            if best_key is None or key < best_key:
                best = (xc, y_target)
                best_key = key
        if best is not None and best not in used:
            route.append(best)
            used.add(best)
    compact = []
    for p in route:
        if not compact or np.hypot(p[0] - compact[-1][0], p[1] - compact[-1][1]) > 5:
            compact.append(p)
    if len(compact) <= 1:
        return compact
    p_first, p_last = compact[0], compact[-1]
    d_first = np.hypot(start_xy[0] - p_first[0], start_xy[1] - p_first[1])
    d_last = np.hypot(start_xy[0] - p_last[0], start_xy[1] - p_last[1])
    return compact if d_first <= d_last else list(reversed(compact))


def choose_nearest_offset_route(large_regions, start_xy, step_px=18, y_offset=4):
    candidates = []
    for reg in large_regions:
        route = build_bottom_route(reg['mask'], start_xy, step_px=step_px, y_offset=y_offset)
        if not route:
            continue
        d0 = float(np.hypot(start_xy[0] - route[0][0], start_xy[1] - route[0][1]))
        candidates.append((d0, reg, route))
    if not candidates:
        return None, []
    candidates.sort(key=lambda item: item[0])
    return candidates[0][1], candidates[0][2]


def remaining_colored_mask(img_bgr):
    """Máscara dos pixels ainda coloridos/acesos no incêndio filtrado."""
    if img_bgr is None:
        return None
    # A imagem filtrada usa fundo preto. Então qualquer pixel não escuro ainda é considerado região remanescente.
    return (np.max(img_bgr, axis=2) > 12).astype(np.uint8)


def remaining_colored_pixels(img_bgr):
    mask = remaining_colored_mask(img_bgr)
    if mask is None:
        return 0
    return int(np.count_nonzero(mask))


def fallback_regions_from_colored_pixels(img_bgr, min_area=1):
    """Usado quando ainda há cor na imagem, mas a segmentação térmica já não retorna regiões válidas."""
    mask = remaining_colored_mask(img_bgr)
    if mask is None:
        return []
    return region_components(mask, min_area=min_area)


def choose_route_from_current_fire(params):
    if SIM_STATE['fire_canvas'] is None:
        return []

    result = extract_hot_regions(SIM_STATE['fire_canvas'], params)
    regs = result['regions']

    # Importante: se ainda existem pixels coloridos, mas eles ficaram menores que a área mínima,
    # usa uma segmentação de emergência baseada em pixels não pretos. Assim o jato só para
    # quando a imagem filtrada realmente não tiver mais regiões coloridas.
    if not regs and remaining_colored_pixels(SIM_STATE['fire_canvas']) > 0:
        regs = fallback_regions_from_colored_pixels(SIM_STATE['fire_canvas'], min_area=1)

    SIM_STATE['regions'] = regs

    if not regs:
        SIM_STATE['route'] = []
        SIM_STATE['route_index'] = 0
        SIM_STATE['current_target_region'] = None
        return []

    start_xy = (SIM_STATE['x'], SIM_STATE['y'])
    area_thr = int(params.get('area_threshold_offset', 2500))
    large_regions = [r for r in regs if int(r['area']) >= area_thr]

    chosen_region, route = choose_nearest_offset_route(
        large_regions,
        start_xy,
        step_px=int(params.get('route_step_px', 18)),
        y_offset=int(params.get('offset_y', 4))
    )

    if route:
        SIM_STATE['mode'] = 'borda'
        SIM_STATE['current_target_region'] = chosen_region
    else:
        # Para não unir regiões desconexas em uma única trajetória,
        # escolhe apenas a região mais próxima do cursor.
        idx_near = int(np.argmin([np.hypot(start_xy[0] - r['cx'], start_xy[1] - r['cy']) for r in regs]))
        reg = regs[idx_near]
        hold = max(1, int(params.get('centroid_hold_frames', 4)))
        c = (int(round(reg['cx'])), int(round(reg['cy'])))
        route = [c for _ in range(hold)]
        SIM_STATE['mode'] = 'centroides'
        SIM_STATE['current_target_region'] = reg

    SIM_STATE['route'] = route
    SIM_STATE['route_index'] = 0
    return route


def init_fire_canvas_from_selected_image():
    global SIM_STATE
    if len(image_files) == 0 or dropdown_image.value == 'Nenhuma imagem encontrada':
        SIM_STATE['base_fire'] = None
        SIM_STATE['fire_canvas'] = None
        return
    params = coletar_params()
    img_path = DATA_DIR / dropdown_image.value
    img = cv2.imread(str(img_path))
    if img is None:
        SIM_STATE['base_fire'] = None
        SIM_STATE['fire_canvas'] = None
        return
    res = extract_hot_regions(img, params)
    filtered = res['filtered']
    Hm, Wm = filtered.shape[:2]
    SIM_STATE = {
        'base_fire': filtered.copy(),
        'fire_canvas': filtered.copy(),
        'route': [],
        'route_index': 0,
        'mode': 'borda',
        'movement_phase': 'aproximação/entre trajetórias',
        'regions': res['regions'],
        'frame': 0,
        'running': False,
        'animation_task': None,
        'x': 26,
        'y': max(18, Hm - 24),
        'pause_until_frame': -1,
        'current_target_region': None,
    }
    choose_route_from_current_fire(params)


def update_motion_sim(params):
    if SIM_STATE['frame'] < SIM_STATE['pause_until_frame']:
        return
    if SIM_STATE['route_index'] >= len(SIM_STATE['route']):
        return

    other_speed = max(1, int(params.get('aim_speed_px', 15)))
    border_speed = max(1, int(round(other_speed * 0.70)))
    if SIM_STATE['mode'] == 'borda' and SIM_STATE['route_index'] > 0:
        eff_step = border_speed
        SIM_STATE['movement_phase'] = 'offset'
    else:
        eff_step = other_speed
        SIM_STATE['movement_phase'] = 'aproximação/entre trajetórias'

    tx, ty = SIM_STATE['route'][SIM_STATE['route_index']]
    Hm, Wm = SIM_STATE['fire_canvas'].shape[:2]
    nx, ny, arrived = move_point_towards(tx, ty, SIM_STATE['x'], SIM_STATE['y'], max_step=eff_step, w_lim=Wm, h_lim=Hm)
    SIM_STATE['x'], SIM_STATE['y'] = nx, ny

    cv2.circle(SIM_STATE['fire_canvas'], (nx, ny), int(params.get('erase_radius', 28)), (0, 0, 0), -1)

    if arrived:
        SIM_STATE['route_index'] += 1
        dwell = max(0, int(params.get('centroid_hold_frames', 4)))
        if dwell > 0:
            SIM_STATE['pause_until_frame'] = max(SIM_STATE['pause_until_frame'], SIM_STATE['frame'] + dwell)
        if SIM_STATE['route_index'] >= len(SIM_STATE['route']):
            SIM_STATE['pause_until_frame'] = max(SIM_STATE['pause_until_frame'], SIM_STATE['frame'] + max(1, dwell))


def draw_simulation_panel():
    if SIM_STATE['base_fire'] is None:
        sim_image_widget.value = b''
        return

    panel = SIM_STATE['fire_canvas'].copy()
    regions = SIM_STATE['regions']
    route = SIM_STATE['route']

    for i, reg in enumerate(regions):
        x, y, w, h = reg['bbox']
        color = (0, 255, 0) if i == 0 else (180, 180, 180)
        cv2.rectangle(panel, (x, y), (x + w, y + h), color, 1)
        cx = int(round(reg['cx']))
        cy = int(round(reg['cy']))
        cv2.circle(panel, (cx, cy), 2, color, -1, lineType=cv2.LINE_AA)

    if len(route) >= 2:
        pts = np.array([[int(p[0]), int(p[1])] for p in route], dtype=np.int32)
        cv2.polylines(panel, [pts], isClosed=False, color=(68, 204, 102), thickness=1, lineType=cv2.LINE_AA)
        for p in pts:
            cv2.circle(panel, tuple(p), 2, (0, 170, 51), -1, lineType=cv2.LINE_AA)
    elif len(route) == 1:
        p = (int(route[0][0]), int(route[0][1]))
        cv2.circle(panel, p, 3, (0, 170, 51), -1, lineType=cv2.LINE_AA)

    x = int(SIM_STATE['x'])
    y = int(SIM_STATE['y'])
    r = max(4, int(s_erase.value))
    cv2.circle(panel, (x, y), r, (120, 255, 120), 1, lineType=cv2.LINE_AA)
    cv2.rectangle(panel, (x - 6, y - 6), (x + 6, y + 6), (0, 255, 0), 2, lineType=cv2.LINE_AA)

    title_h = 32
    Hm, Wm = panel.shape[:2]
    canvas = np.zeros((Hm + title_h, Wm, 3), dtype=np.uint8)
    canvas[:title_h, :, :] = np.array([18, 18, 22], dtype=np.uint8)
    canvas[title_h:, :, :] = panel

    remaining = remaining_colored_pixels(SIM_STATE['fire_canvas'])
    title = f"Simulador | modo={SIM_STATE['mode']} | pixels coloridos restantes={remaining}"
    cv2.putText(canvas, title, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (235, 235, 235), 1, cv2.LINE_AA)

    target_width = 780
    if target_width > canvas.shape[1]:
        scale = target_width / float(canvas.shape[1])
        new_h = int(round(canvas.shape[0] * scale))
        canvas = cv2.resize(canvas, (target_width, new_h), interpolation=cv2.INTER_LINEAR)

    sim_image_widget.value = encode_jpg_bgr(canvas, quality=88)

def update_status_html(message_extra=''):
    params = coletar_params()
    remaining = remaining_colored_pixels(SIM_STATE['fire_canvas']) if SIM_STATE['fire_canvas'] is not None else 0
    n_regions = len(SIM_STATE.get('regions', []))

    txt = f"""
    <pre style='font-size:13px; line-height:1.3;'>
Escala: {params['palette_name']}
Filtro: {params['temp_filter_min_c']} a {params['temp_filter_max_c']} °C
Imagem: {dropdown_image.value}
Modo atual: {SIM_STATE['mode']}
Fase de movimento: {SIM_STATE.get('movement_phase', '-')}
Frame: {SIM_STATE.get('frame', 0)}
Trajetória: {SIM_STATE.get('route_index', 0)} / {len(SIM_STATE.get('route', []))}
Regiões de trajetória atuais: {n_regions}
Pixels coloridos restantes: {remaining}
{message_extra}
    </pre>
    """
    status_html.value = txt

def maybe_rebuild_route_static(params):
    if SIM_STATE['fire_canvas'] is None:
        return
    finished = SIM_STATE['route_index'] >= len(SIM_STATE['route'])
    no_route = len(SIM_STATE['route']) == 0
    if no_route or (finished and SIM_STATE['frame'] >= SIM_STATE['pause_until_frame']):
        choose_route_from_current_fire(params)


def is_fire_extinguished(params):
    if SIM_STATE['fire_canvas'] is None:
        return True

    # Critério solicitado: parar somente quando não houver mais regiões coloridas
    # na imagem filtrada/simulada, não apenas quando a segmentação térmica ficar vazia.
    remaining = remaining_colored_pixels(SIM_STATE['fire_canvas'])
    if remaining <= 0:
        SIM_STATE['regions'] = []
        return True

    res = extract_hot_regions(SIM_STATE['fire_canvas'], params)
    regs = res['regions']
    if not regs:
        regs = fallback_regions_from_colored_pixels(SIM_STATE['fire_canvas'], min_area=1)
    SIM_STATE['regions'] = regs

    return False


def simulation_step_once():
    params = coletar_params()
    if SIM_STATE['fire_canvas'] is None:
        return False
    SIM_STATE['frame'] += 1
    maybe_rebuild_route_static(params)
    update_motion_sim(params)
    extinguished = is_fire_extinguished(params)
    draw_simulation_panel()
    if extinguished:
        update_status_html('Critério de parada: não restam regiões quentes segmentadas.')
        return False
    update_status_html('Simulação em execução.')
    return True


async def simulation_loop_async():
    try:
        while SIM_STATE['running']:
            keep = simulation_step_once()
            if not keep:
                SIM_STATE['running'] = False
                break
            await asyncio.sleep(max(0.02, float(slider_sim_delay.value)))
    finally:
        button_play_fire.disabled = False
        button_stop_fire.disabled = True
        button_play_fire.description = 'Play'
        if not SIM_STATE['running']:
            update_status_html('Simulação parada.')


def play_fire_simulation(_=None):
    if SIM_STATE['running']:
        return
    if SIM_STATE['fire_canvas'] is None:
        init_fire_canvas_from_selected_image()
    SIM_STATE['running'] = True
    button_play_fire.disabled = True
    button_stop_fire.disabled = False
    button_play_fire.description = 'Rodando...'
    try:
        loop = asyncio.get_event_loop()
        SIM_STATE['animation_task'] = loop.create_task(simulation_loop_async())
    except RuntimeError:
        SIM_STATE['animation_task'] = asyncio.ensure_future(simulation_loop_async())


def stop_fire_simulation(_=None):
    SIM_STATE['running'] = False
    button_play_fire.disabled = False
    button_stop_fire.disabled = True
    button_play_fire.description = 'Play'
    update_status_html('Simulação interrompida pelo botão Stop.')


def reset_fire_simulation(_=None):
    stop_fire_simulation()
    init_fire_canvas_from_selected_image()
    draw_simulation_panel()
    update_status_html('Simulação resetada. A imagem Filtro voltou ao estado inicial.')

# ------------------------------------------------------------
# Atualização das prévias
# ------------------------------------------------------------
def atualizar_preview(change=None):
    global CURRENT_PREPROCESS_PARAMS
    params = coletar_params()
    CURRENT_PREPROCESS_PARAMS = params.copy()

    grad_original = gerar_gradiente_lut(PALETTE_LUTS[params['palette_name']], width=520, height=48)
    temp_min = params['temp_min_c']
    temp_max = params['temp_max_c']
    fmin = params['temp_filter_min_c']
    fmax = params['temp_filter_max_c']
    idx_min = int(np.clip(255 * (fmin - temp_min) / max(1, temp_max - temp_min), 0, 255))
    idx_max = int(np.clip(255 * (fmax - temp_min) / max(1, temp_max - temp_min), 0, 255))
    if idx_max < idx_min:
        idx_min, idx_max = idx_max, idx_min
    grad_gray = np.linspace(0, 255, 520, dtype=np.uint8)
    mask_grad = ((grad_gray >= idx_min) & (grad_gray <= idx_max)).astype(np.uint8) * 255
    mask_grad = np.tile(mask_grad, (48, 1))
    grad_filtrado = grad_original.copy()
    grad_filtrado[mask_grad == 0] = (0, 0, 0)
    painel_grad = np.vstack([grad_original, np.full((16, 520, 3), 255, dtype=np.uint8), grad_filtrado])
    cv2.putText(painel_grad, 'Gradiente original', (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)
    cv2.putText(painel_grad, 'Gradiente apos filtro', (10, 96), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)
    gradient_image_widget.value = encode_png_bgr(painel_grad)

    if len(image_files) == 0 or dropdown_image.value == 'Nenhuma imagem encontrada':
        preview_image_widget.value = b''
        sim_image_widget.value = b''
        update_status_html('Nenhuma imagem encontrada na pasta data.')
        return

    img_path = DATA_DIR / dropdown_image.value
    img = cv2.imread(str(img_path))
    if img is None:
        preview_image_widget.value = b''
        sim_image_widget.value = b''
        update_status_html(f'Erro ao abrir: {img_path}')
        return

    _, _, painel = aplicar_preprocessamento(img, params)
    preview_image_widget.value = encode_jpg_bgr(painel, quality=88)

    # Reinicializa a simulação sempre que a configuração mudar.
    init_fire_canvas_from_selected_image()
    draw_simulation_panel()
    update_status_html('Prévia atualizada. A simulação foi reconstruída com os parâmetros atuais.')

# ------------------------------------------------------------
# Salvar / carregar setup
# ------------------------------------------------------------
def salvar_setup_atual(_=None):
    params = coletar_params()
    prefixo = nome_seguro(params['palette_name'])
    extra = nome_seguro(setup_extra_name.value)
    if extra == '':
        extra = 'sem_nome'
    setup_name = f"{prefixo}_{extra}"
    setups = carregar_setups()
    setups[setup_name] = params
    salvar_setups(setups)
    setup_dropdown.options = sorted(setups.keys())
    setup_dropdown.value = setup_name
    update_status_html(f'Setup salvo: {setup_name}')



def aplicar_params_nos_widgets(params):
    """Atualiza automaticamente todas as barras/widgets com os parâmetros de um setup salvo."""
    if not isinstance(params, dict):
        return

    dropdown_palette.value = normalize_palette_name(params.get('palette_name', DEFAULT_PREPROCESS['palette_name']))

    s_temp_min.value = int(params.get('temp_min_c', DEFAULT_PREPROCESS['temp_min_c']))
    s_temp_max.value = int(params.get('temp_max_c', DEFAULT_PREPROCESS['temp_max_c']))
    s_filter_min.value = int(params.get('temp_filter_min_c', DEFAULT_PREPROCESS['temp_filter_min_c']))
    s_filter_max.value = int(params.get('temp_filter_max_c', DEFAULT_PREPROCESS['temp_filter_max_c']))

    s_blur.value = int(params.get('blur_ksize', DEFAULT_PREPROCESS['blur_ksize']))
    s_open.value = int(params.get('open_iter', DEFAULT_PREPROCESS['open_iter']))
    s_close.value = int(params.get('close_iter', DEFAULT_PREPROCESS['close_iter']))
    s_dilate.value = int(params.get('dilate_iter', DEFAULT_PREPROCESS['dilate_iter']))
    s_min_area.value = int(params.get('min_area', DEFAULT_PREPROCESS['min_area']))
    s_max_area.value = int(params.get('max_area', DEFAULT_PREPROCESS['max_area']))

    c_rectify.value = bool(params.get('use_rectification', DEFAULT_PREPROCESS['use_rectification']))

    s_area_offset.value = int(params.get('area_threshold_offset', DEFAULT_PREPROCESS['area_threshold_offset']))
    s_offset_y.value = int(params.get('offset_y', DEFAULT_PREPROCESS['offset_y']))
    s_route_step.value = int(params.get('route_step_px', DEFAULT_PREPROCESS['route_step_px']))
    s_hold.value = int(params.get('centroid_hold_frames', DEFAULT_PREPROCESS['centroid_hold_frames']))
    s_aim_speed.value = int(params.get('aim_speed_px', DEFAULT_PREPROCESS['aim_speed_px']))
    s_erase.value = int(params.get('erase_radius', DEFAULT_PREPROCESS['erase_radius']))

    c_yolo.value = bool(params.get('enable_human_yolo', DEFAULT_PREPROCESS['enable_human_yolo']))
    s_human_margin.value = int(params.get('human_margin_px', DEFAULT_PREPROCESS['human_margin_px']))

    c_arduino.value = bool(params.get('enable_arduino', DEFAULT_PREPROCESS['enable_arduino']))
    serial_port_text.value = params.get('serial_port', DEFAULT_PREPROCESS['serial_port'])
    serial_baud_text.value = int(params.get('serial_baud', DEFAULT_PREPROCESS['serial_baud']))

def carregar_setup_selecionado(_=None):
    setups = carregar_setups()

    if setup_dropdown.value not in setups:
        update_status_html('Nenhum setup válido selecionado.')
        return

    params = setups[setup_dropdown.value]
    aplicar_params_nos_widgets(params)
    atualizar_preview()
    update_status_html(f"Setup carregado: {setup_dropdown.value}")



def excluir_setup_selecionado(_=None):
    """Exclui o setup selecionado e atualiza a lista."""
    setups = carregar_setups()
    selected = setup_dropdown.value

    if selected in (None, 'Nenhum setup salvo') or selected not in setups:
        update_status_html('Nenhum setup válido selecionado para exclusão.')
        return

    del setups[selected]
    salvar_setups(setups)

    if len(setups) == 0:
        setup_dropdown.options = ['Nenhum setup salvo']
        setup_dropdown.value = 'Nenhum setup salvo'
    else:
        setup_dropdown.options = sorted(setups.keys())
        setup_dropdown.value = sorted(setups.keys())[0]

    update_status_html(f"Setup excluído: {selected}")

button_save_setup.on_click(salvar_setup_atual)
button_load_setup.on_click(carregar_setup_selecionado)

def carregar_setup_ao_selecionar(change):
    """Carrega automaticamente o setup quando ele é selecionado na lista."""
    if change.get('name') != 'value':
        return

    novo_setup = change.get('new')
    if novo_setup in (None, 'Nenhum setup salvo'):
        return

    carregar_setup_selecionado()


setup_dropdown.observe(carregar_setup_ao_selecionar, names='value')

button_delete_setup.on_click(excluir_setup_selecionado)
button_play_fire.on_click(play_fire_simulation)
button_stop_fire.on_click(stop_fire_simulation)
button_reset_fire.on_click(reset_fire_simulation)

widgets_to_observe = [
    dropdown_image, dropdown_palette,
    s_temp_min, s_temp_max, s_filter_min, s_filter_max,
    s_blur, s_open, s_close, s_dilate, s_min_area, s_max_area,
    c_rectify, s_area_offset, s_offset_y, s_route_step, s_hold,
    s_aim_speed, s_erase, c_yolo, s_human_margin,
    c_arduino, serial_port_text, serial_baud_text,
]
for widget in widgets_to_observe:
    widget.observe(atualizar_preview, names='value')

# ------------------------------------------------------------
# Interface final
# ------------------------------------------------------------
interface = widgets.VBox([
    palette_reference_box,
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>Seleção de imagem e escala</h3>'),
    dropdown_image,
    dropdown_palette,
    widgets.HTML('<h3>Preview da escala e do filtro</h3>'),
    gradient_image_widget,
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>Ajustes paramétricos</h3>'),
    widgets.HTML('<p>Todos os parâmetros são ajustados sobre uma imagem estática e a lógica de combate da simulação é reconstruída conforme o simulador anexo.</p>'),
    parametros_2_colunas,
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>Salvar ou carregar setup</h3>'),
    setup_extra_name,
    widgets.HBox([button_save_setup, button_load_setup, button_delete_setup]),
    setup_dropdown,
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>Preview na imagem selecionada</h3>'),
    preview_image_widget,
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>Simulação de combate</h3>'),
    widgets.HTML('<p>A simulação usa a imagem <b>Filtro</b> como incêndio base. O jato só para quando não houver mais pixels coloridos na imagem simulada. Regiões grandes usam <b>offset da borda inferior</b>; regiões menores usam <b>centróide</b>. Regiões desconexas são atacadas uma de cada vez.</p>'),
    widgets.HBox([button_play_fire, button_stop_fire, button_reset_fire, slider_sim_delay]),
    sim_image_widget,
    widgets.HTML('<hr>'),
    widgets.HTML('<h3>Status</h3>'),
    status_html,
], layout=widgets.Layout(width='100%', max_width='1280px'))

display(interface)
atualizar_preview()

In [16]:
# ============================================================
# CÉLULA 5 — Arduino Uno + MG90S em tempo real
# Calibração dos servos pela imagem + vídeo/simulação do jato
# ============================================================
#
# Esta célula:
# - cria/abre o sketch do Arduino;
# - lista portas seriais e câmeras;
# - move os servos em tempo real durante a calibração;
# - aplica o mapeamento por 3 pontos;
# - abre as câmeras e reutiliza a simulação visual da célula 4;
# - envia a mira ou o impacto previsto ao Arduino.

import json
import time
import asyncio
import subprocess
import shutil
import sys
from pathlib import Path

import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

try:
    import serial
    from serial.tools import list_ports
    SERIAL_AVAILABLE = True
except Exception:
    serial = None
    list_ports = None
    SERIAL_AVAILABLE = False


def garantir_pyserial_importado():
    """
    Tenta importar o pyserial no mesmo kernel que está rodando o notebook.

    Importante:
    instalar com pip no terminal nem sempre instala no mesmo Python do Jupyter.
    Por isso esta função testa dentro do kernel atual.
    """
    global serial, list_ports, SERIAL_AVAILABLE

    try:
        import serial as _serial
        from serial.tools import list_ports as _list_ports

        serial = _serial
        list_ports = _list_ports
        SERIAL_AVAILABLE = True
        return True, None

    except Exception as e:
        serial = None
        list_ports = None
        SERIAL_AVAILABLE = False
        return False, e


def instalar_pyserial_no_kernel(_=None):
    """
    Instala pyserial usando o mesmo executável Python do kernel atual.
    """
    target_widget = msg_serial if "msg_serial" in globals() else None

    registrar_acao(
        "<b>Instalando pyserial no kernel atual...</b><br>"
        f"Python do kernel:<br><code>{sys.executable}</code>",
        "info",
        widget=target_widget
    )

    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "pyserial"],
            capture_output=True,
            text=True,
            timeout=120
        )

        saida = (result.stdout or "") + "\n" + (result.stderr or "")
        saida = saida.strip()

        if len(saida) > 1400:
            saida = saida[-1400:]

        ok, err = garantir_pyserial_importado()

        if result.returncode == 0 and ok:
            registrar_acao(
                "<b>pyserial instalado/importado com sucesso no kernel atual.</b><br>"
                "Agora clique em <b>Atualizar portas</b> e depois em <b>Verificar porta</b>.<br>"
                f"<b>Python do kernel:</b><br><code>{sys.executable}</code><br>"
                f"<b>Saída do pip:</b><br><pre style='white-space:pre-wrap'>{saida}</pre>",
                "ok",
                widget=target_widget
            )
            return True

        registrar_acao(
            "<b>pyserial não ficou disponível no kernel atual.</b><br>"
            f"<b>Python do kernel:</b><br><code>{sys.executable}</code><br>"
            f"<b>Erro de importação:</b><br><code>{err}</code><br>"
            f"<b>Saída do pip:</b><br><pre style='white-space:pre-wrap'>{saida}</pre>",
            "erro",
            widget=target_widget
        )
        return False

    except Exception as e:
        registrar_acao(
            "<b>Falha ao tentar instalar pyserial no kernel atual.</b><br>"
            f"<b>Python do kernel:</b><br><code>{sys.executable}</code><br>"
            f"<code>{type(e).__name__}: {e}</code>",
            "erro",
            widget=target_widget
        )
        return False


# ------------------------------------------------------------
# Caminhos
# ------------------------------------------------------------
if "THERMAL_SETUPS_FILE" in globals():
    PARAMS_DIR_5 = Path(THERMAL_SETUPS_FILE).parent
else:
    PROJECT_ROOT = localizar_raiz_blaze()
    BASE_DIR = (
        PROJECT_ROOT
        / "vision"
        / "fire_detect"
        / "jet_automation"
    )
    PARAMS_DIR_5 = BASE_DIR / "parameters_setups"

PARAMS_DIR_5.mkdir(parents=True, exist_ok=True)

ARDUINO_SETUPS_FILE = PARAMS_DIR_5 / "arduino_servo_mapping_setups.json"
ARDUINO_SKETCH_DIR = PARAMS_DIR_5 / "blaze_arduino_uno_mg90s"
ARDUINO_SKETCH_FILE = ARDUINO_SKETCH_DIR / "blaze_arduino_uno_mg90s.ino"

ARDUINO_SKETCH_CODE = '#include <Servo.h>\n\n/*\n  BLAZE - Controle de mira com dois servos MG90S\n  Placa: Arduino Uno\n\n  Serial Monitor:\n    Baud: 115200\n    Line ending: Newline\n\n  Comandos:\n    PING\n    ID?\n    STATUS?\n    ZERO\n    CENTER\n    X90.0,Y85.5,P22.0\n\n  Importante:\n    O MG90S comum não mede a posição real.\n    O status mostra a posição comandada pelo Arduino.\n\n  Segurança de montagem:\n    Na primeira utilização, deixe os atuadores/jato desacoplados.\n    Ao ligar/reiniciar, este sketch posiciona X=90 e Y=90.\n    Depois monte fisicamente o jato apontando para o centro da imagem.\n*/\n\nServo servoX;\nServo servoY;\n\nconst int SERVO_X_PIN = 9;\nconst int SERVO_Y_PIN = 10;\n\nconst float SERVO_MIN = 0.0;\nconst float SERVO_MAX = 180.0;\n\nconst char* BLAZE_ID = "UNO_MG90S_V6";\n\nfloat currentX = 90.0;\nfloat currentY = 90.0;\nfloat currentP = 22.0;\n\nString buffer = "";\n\nunsigned long lastStatusMillis = 0;\nconst unsigned long STATUS_INTERVAL_MS = 2000;\n\nfloat clampFloat(float value, float minValue, float maxValue) {\n  if (value < minValue) return minValue;\n  if (value > maxValue) return maxValue;\n  return value;\n}\n\nvoid applyServoAngles(float x, float y) {\n  currentX = clampFloat(x, SERVO_MIN, SERVO_MAX);\n  currentY = clampFloat(y, SERVO_MIN, SERVO_MAX);\n\n  servoX.write((int)round(currentX));\n  servoY.write((int)round(currentY));\n}\n\nvoid printStatusLine() {\n  Serial.print("X=");\n  Serial.print(currentX, 1);\n  Serial.print(" ; Y=");\n  Serial.print(currentY, 1);\n  Serial.print(" ; P=");\n  Serial.print(currentP, 1);\n  Serial.print(" ; ID=");\n  Serial.println(BLAZE_ID);\n}\n\nvoid printOk() {\n  Serial.print("OK ");\n  printStatusLine();\n}\n\nbool parseCommand(String cmd) {\n  cmd.trim();\n\n  if (cmd.length() == 0) {\n    return true;\n  }\n\n  if (cmd == "PING") {\n    Serial.print("BLAZE_OK ");\n    Serial.println(BLAZE_ID);\n    return true;\n  }\n\n  if (cmd == "ID?") {\n    Serial.print("BLAZE_ID ");\n    Serial.println(BLAZE_ID);\n    return true;\n  }\n\n  if (cmd == "STATUS?") {\n    printStatusLine();\n    return true;\n  }\n\n  if (cmd == "ZERO") {\n    applyServoAngles(0.0, 0.0);\n    printOk();\n    return true;\n  }\n\n  if (cmd == "CENTER") {\n    applyServoAngles(90.0, 90.0);\n    printOk();\n    return true;\n  }\n\n  int xIndex = cmd.indexOf(\'X\');\n  int yIndex = cmd.indexOf(\'Y\');\n  int pIndex = cmd.indexOf(\'P\');\n\n  int comma1 = cmd.indexOf(\',\');\n  int comma2 = cmd.indexOf(\',\', comma1 + 1);\n\n  if (xIndex < 0 || yIndex < 0 || comma1 < 0) {\n    return false;\n  }\n\n  String xStr = cmd.substring(xIndex + 1, comma1);\n  String yStr = "";\n\n  if (comma2 > comma1) {\n    yStr = cmd.substring(yIndex + 1, comma2);\n  } else {\n    yStr = cmd.substring(yIndex + 1);\n  }\n\n  if (pIndex >= 0) {\n    String pStr = cmd.substring(pIndex + 1);\n    currentP = pStr.toFloat();\n  }\n\n  float x = xStr.toFloat();\n  float y = yStr.toFloat();\n\n  applyServoAngles(x, y);\n  printOk();\n\n  return true;\n}\n\nvoid setup() {\n  Serial.begin(115200);\n\n  servoX.attach(SERVO_X_PIN);\n  servoY.attach(SERVO_Y_PIN);\n\n  applyServoAngles(90.0, 90.0);\n\n  delay(300);\n\n  Serial.println("BLAZE Arduino pronto.");\n  Serial.print("ID=");\n  Serial.println(BLAZE_ID);\n  Serial.println("Baud=115200");\n  Serial.println("Formato status: X=valor ; Y=valor ; P=valor");\n  Serial.println("Comandos: PING, ID?, STATUS?, ZERO, CENTER, X90.0,Y85.5,P22.0");\n\n  printStatusLine();\n  lastStatusMillis = millis();\n}\n\nvoid loop() {\n  while (Serial.available() > 0) {\n    char c = (char)Serial.read();\n\n    if (c == \'\\n\' || c == \'\\r\') {\n      if (buffer.length() > 0) {\n        bool ok = parseCommand(buffer);\n\n        if (!ok) {\n          Serial.print("ERRO comando: ");\n          Serial.println(buffer);\n        }\n\n        buffer = "";\n      }\n    } else {\n      buffer += c;\n\n      if (buffer.length() > 80) {\n        buffer = "";\n        Serial.println("ERRO buffer serial muito longo.");\n      }\n    }\n  }\n\n  if (millis() - lastStatusMillis >= STATUS_INTERVAL_MS) {\n    printStatusLine();\n    lastStatusMillis = millis();\n  }\n}\n'


# ------------------------------------------------------------
# Estado
# ------------------------------------------------------------
ARDUINO_STATE = {
    "serial": None,
    "connected": False,
    "camera_running": False,
    "camera_task": None,
    "cap1": None,
    "cap2": None,
    "last_angles": None,
    "last_target": None,
    "last_command": None,
    "send_count": 0,
    "calibration_applied": False,
    "ref_cap": None,
    "ref_camera_running": False,
    "ref_camera_task": None,

    # Controle de transição sem foco -> novo foco
    "focus_present_prev": None,
    "no_focus_count": 0,
    "new_focus_count": 0,
    "planner_last_reason": "aguardando",

    # Segurança por detecção de humanos
    "human_yolo_model": None,
    "human_yolo_loaded": False,
    "human_detected": False,
    "human_boxes": [],
    "human_pause_active": False,
    "human_resume_after_clear": False,
    "human_loop_id": 0,
    "human_status": "YOLO humano não carregado",

    # Sincronização/diagnóstico de câmeras
    "frame_pair_id": 0,
    "last_pair_dt_ms": 0.0,
    "last_capture_method": "read",
    "freeze_count": 0,

    # Calibração estéreo real por xadrez
    "stereo_calib": None,
    "stereo_maps": None,
    "stereo_maps_key": None,
    "stereo_status": "calibração não carregada",

    # Diagnóstico de gargalos
    "last_loop_ms": 0.0,
    "last_process_ms": 0.0,
    "last_yolo_ms": 0.0,
    "last_match_ms": 0.0,
    "last_match_count": 0,
    "stereo_last_regs2": [],
    "stereo_last_match_frame": -999,

    # Profundidade/distância do alvo em combate
    "current_target_depth": None,
    "current_target_depth_text": "",

    # Retificação visual/processamento
    "last_rectified_pair": False,
    "last_rectification_fallback": "",

    # Detecção de mudança real do cenário para atualizar canvas apagado
    "scene_change_score": 0.0,
    "scene_change_count": 0,
    "scene_change_last_reason": "",
    "scene_change_threshold": 0.22,
    "scene_change_min_pixels": 120,

    # YOLO humano
    "human_yolo_load_attempted": False,

    # Suavização/estabilidade de distância
    "last_valid_depth": None,
    "last_valid_depth_is_proxy": True,
    "last_valid_depth_frame": -999,
    "depth_hold_frames": 30,

    # Estratégia de distância por janela
    "depth_internal_every_frames": 15,
    "depth_publish_every_frames": 90,
    "depth_window_values": [],
    "depth_window_proxy_flags": [],
    "depth_window_start_frame": 0,
    "depth_display_value": None,
    "depth_display_is_proxy": True,
    "depth_display_frame": -999,
    "depth_display_text": "Dist: --",
    "depth_sample_count_window": 0,
    "depth_total_samples": 0,
    "depth_last_sample_text": "sem amostra",
    "depth_last_match_score": None,
    "depth_last_disparity": None,
    "depth_last_source": "nenhuma",
    "depth_first_publish_done": False,

    # Perfil de desempenho do loop
    "perf_capture_ms": 0.0,
    "perf_params_ms": 0.0,
    "perf_process_ms": 0.0,
    "perf_transition_ms": 0.0,
    "perf_yolo_ms": 0.0,
    "perf_panel_ms": 0.0,
    "perf_encode_ms": 0.0,
    "perf_serial_ms": 0.0,
    "perf_status_ms": 0.0,
    "perf_total_ms": 0.0,
    "last_status_text": "",
}

CALIBRATION_SERVO_STATE = {
    "active_point": None,
}

ARDUINO_MAPPING_ACTIVE = {
    "coef_x": None,
    "coef_y": None,
    "params": None,
}

ARDUINO_JET_PARAMS = {
    "pressure_calib": 22.0,
}


# ------------------------------------------------------------
# Sketch Arduino
# ------------------------------------------------------------
def verificar_pasta_sketch():
    """
    Cria a pasta do sketch se ela não existir e informa se estava vazia.
    """
    ARDUINO_SKETCH_DIR.mkdir(parents=True, exist_ok=True)

    arquivos = [
        p for p in ARDUINO_SKETCH_DIR.iterdir()
        if p.is_file() and not p.name.startswith(".")
    ]

    return len(arquivos) == 0


def criar_ou_atualizar_sketch(_=None):
    pasta_estava_vazia = verificar_pasta_sketch()
    ARDUINO_SKETCH_FILE.write_text(ARDUINO_SKETCH_CODE, encoding="utf-8")

    if pasta_estava_vazia:
        aviso = (
            "<br><b>Aviso:</b> não havia nenhum arquivo na pasta do sketch. "
            "A pasta foi criada/preenchida automaticamente."
        )
    else:
        aviso = (
            "<br><b>Observação:</b> já havia arquivo(s) na pasta. "
            "O arquivo principal foi atualizado."
        )

    registrar_acao(
        f"<b>Sketch salvo em:</b><br><code>{ARDUINO_SKETCH_FILE}</code>"
        f"{aviso}",
        "ok" if not pasta_estava_vazia else "aviso"
    )


def abrir_sketch_vscode(_=None):
    criar_ou_atualizar_sketch()

    try:
        subprocess.Popen(["code", str(ARDUINO_SKETCH_DIR)])
        registrar_acao(
            f"<b>Sketch aberto no VS Code:</b><br><code>{ARDUINO_SKETCH_DIR}</code>",
            "ok"
        )
    except Exception as e:
        registrar_acao(
            f"<b>Não consegui abrir automaticamente no VS Code.</b><br>"
            f"Abra manualmente a pasta:<br><code>{ARDUINO_SKETCH_DIR}</code><br>"
            f"Erro: {type(e).__name__}: {e}",
            "erro"
        )


def abrir_sketch_arduino_ide(_=None):
    """
    Abre o sketch especificamente no Arduino IDE.

    Esta versão não considera o comando como sucesso imediatamente.
    Ela tenta abrir, aguarda alguns segundos e verifica se o processo morreu com erro.
    Se morrer, mostra o erro real ao lado do botão.
    """
    criar_ou_atualizar_sketch()

    tentativas = []

    # Arduino IDE 2.x instalado pelo sistema.
    if shutil.which("arduino-ide"):
        tentativas.append(("Arduino IDE 2.x", ["arduino-ide", str(ARDUINO_SKETCH_FILE)]))

    # Flatpak comum para Arduino IDE 2.
    if shutil.which("flatpak"):
        tentativas.append(("Arduino IDE 2.x Flatpak", ["flatpak", "run", "cc.arduino.IDE2", str(ARDUINO_SKETCH_FILE)]))

    # Snap, quando disponível.
    if shutil.which("snap"):
        # Primeiro tenta o nome mais provável da IDE 2.x.
        tentativas.append(("Arduino IDE Snap", ["snap", "run", "arduino-ide", str(ARDUINO_SKETCH_FILE)]))
        # Depois tenta o nome antigo, mas só será aceito se o processo continuar vivo.
        tentativas.append(("Arduino clássico Snap", ["snap", "run", "arduino", str(ARDUINO_SKETCH_FILE)]))

    # Arduino IDE clássico instalado pelo sistema.
    if shutil.which("arduino"):
        tentativas.append(("Arduino IDE clássico", ["arduino", str(ARDUINO_SKETCH_FILE)]))

    # AppImage opcional em locais comuns.
    appimage_candidates = [
        Path.home() / "Applications",
        Path.home() / "Downloads",
        Path("/opt"),
    ]

    for base_app in appimage_candidates:
        try:
            if base_app.exists():
                for app in base_app.glob("*Arduino*IDE*.AppImage"):
                    tentativas.append(("Arduino IDE AppImage", [str(app), str(ARDUINO_SKETCH_FILE)]))
        except Exception:
            pass

    ambiente = []
    ambiente.append(f"DISPLAY={repr(__import__('os').environ.get('DISPLAY'))}")
    ambiente.append(f"WAYLAND_DISPLAY={repr(__import__('os').environ.get('WAYLAND_DISPLAY'))}")

    erros = []

    if not tentativas:
        registrar_acao(
            html_passo_a_passo_instalacao_arduino()
            + "<br><b>Ambiente gráfico detectado:</b><br>"
            + f"<code>{'<br>'.join(ambiente)}</code>",
            "erro"
        )
        return

    for nome, cmd in tentativas:
        try:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )

            # Dá tempo para o programa falhar se o comando estiver errado.
            time.sleep(2.5)
            ret = proc.poll()

            if ret is None:
                registrar_acao(
                    f"<b>Arduino IDE parece ter sido iniciado:</b> {nome}<br>"
                    f"<b>Comando:</b><br><code>{' '.join(cmd)}</code><br>"
                    f"<b>Sketch:</b><br><code>{ARDUINO_SKETCH_FILE}</code><br>"
                    "Se a janela não apareceu, ela pode ter aberto em outro workspace, "
                    "estar atrás do navegador, ou o Jupyter pode estar rodando sem permissão gráfica.",
                    "ok"
                )
                return

            stdout, stderr = proc.communicate(timeout=1)
            saida = (stdout or "") + "\n" + (stderr or "")
            saida = saida.strip()

            if len(saida) > 900:
                saida = saida[:900] + "..."

            erros.append(
                f"<b>{nome}</b><br>"
                f"<code>{' '.join(cmd)}</code><br>"
                f"retorno={ret}<br>"
                f"<code>{saida if saida else 'sem saída'}</code>"
            )

        except Exception as e:
            erros.append(
                f"<b>{nome}</b><br>"
                f"<code>{' '.join(cmd)}</code><br>"
                f"{type(e).__name__}: {e}"
            )

    detalhes = "<hr>".join(erros)

    registrar_acao(
        "<b>Não consegui abrir o Arduino IDE automaticamente.</b><br>"
        "O sketch foi criado corretamente, mas todos os comandos testados encerraram com erro "
        "ou não conseguiram iniciar a interface gráfica.<br><br>"
        "<b>Abra manualmente:</b><br>"
        "1. Abra o Arduino IDE pelo menu do sistema.<br>"
        "2. Vá em <b>File &gt; Open</b>.<br>"
        "3. Selecione:<br>"
        f"<code>{ARDUINO_SKETCH_FILE}</code><br><br>"
        "<b>Ambiente gráfico:</b><br>"
        f"<code>{'<br>'.join(ambiente)}</code><br><br>"
        "<b>Tentativas feitas:</b><br>"
        f"{detalhes}",
        "erro"
    )


# ------------------------------------------------------------
# Setups
# ------------------------------------------------------------
def carregar_setups_arduino():
    if ARDUINO_SETUPS_FILE.exists():
        try:
            data = json.loads(ARDUINO_SETUPS_FILE.read_text(encoding="utf-8"))

            if isinstance(data, dict):
                return data
        except Exception as e:
            print("Aviso: erro ao carregar setups Arduino:", e)

    return {}


def salvar_setups_arduino(data):
    ARDUINO_SETUPS_FILE.parent.mkdir(parents=True, exist_ok=True)
    ARDUINO_SETUPS_FILE.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


# ------------------------------------------------------------
# Setups paramétricos da célula 3
# ------------------------------------------------------------
def carregar_setups_parametricos_celula3():
    """
    Carrega os setups salvos na célula 3.
    Usa o mesmo arquivo THERMAL_SETUPS_FILE quando ele existir no notebook.
    """
    if "THERMAL_SETUPS_FILE" not in globals():
        return {}

    caminho = Path(THERMAL_SETUPS_FILE)

    if not caminho.exists():
        return {}

    try:
        data = json.loads(caminho.read_text(encoding="utf-8"))

        if isinstance(data, dict):
            return data

    except Exception as e:
        print("Aviso: erro ao carregar setups da célula 3:", e)

    return {}


def nomes_setups_parametricos_celula3():
    data = carregar_setups_parametricos_celula3()
    return ["Usar parâmetros atuais"] + sorted(data.keys())


def setup_parametrico_ativo_celula3_5():
    """
    Retorna o dicionário do setup da célula 3 selecionado.
    Se nenhum setup estiver selecionado, retorna {}.
    """
    nome = dropdown_param_setup_3.value if "dropdown_param_setup_3" in globals() else "Usar parâmetros atuais"

    if nome in (None, "Usar parâmetros atuais"):
        return {}

    data = carregar_setups_parametricos_celula3()
    params = data.get(nome)

    return dict(params) if isinstance(params, dict) else {}


def resumo_setup_celula3_5(params):
    """
    Resumo curto dos parâmetros vindos da célula 3.
    """
    if not isinstance(params, dict) or not params:
        return "Nenhum setup ativo da célula 3."

    campos = [
        ("Escala", "palette_name"),
        ("Filtro", ("temp_filter_min_c", "temp_filter_max_c")),
        ("Área mín.", "min_area"),
        ("Área offset", "area_threshold_offset"),
        ("Offset Y", "offset_y"),
        ("Passo rota", "route_step_px"),
        ("Perman. centróide", "centroid_hold_frames"),
        ("Vel. mira", "aim_speed_px"),
        ("Raio apag.", "erase_radius"),
    ]

    partes = []

    for label, key in campos:
        if isinstance(key, tuple):
            if key[0] in params and key[1] in params:
                partes.append(f"{label}: {params[key[0]]}–{params[key[1]]}")
        else:
            if key in params:
                partes.append(f"{label}: {params[key]}")

    return " | ".join(partes) if partes else "Setup carregado, mas sem campos reconhecidos."


def aplicar_setup_celula3_nos_controles_ocultos_5(params):
    """
    Sincroniza widgets internos da célula 5 com o setup da célula 3.

    Estes widgets não aparecem mais na interface principal para evitar
    reconfiguração duplicada, mas continuam existindo para compatibilidade
    com funções internas e setups antigos da célula 5.
    """
    if not isinstance(params, dict):
        return

    pares = [
        ("area_threshold_offset", "area_threshold_offset_widget_5"),
        ("offset_y", "offset_y_widget_5"),
        ("route_step_px", "route_step_widget_5"),
        ("centroid_hold_frames", "centroid_hold_widget_5"),
        ("aim_speed_px", "aim_speed_widget_5"),
        ("erase_radius", "jet_erase_radius_widget"),
    ]

    for key, widget_name in pares:
        if key in params and widget_name in globals():
            try:
                globals()[widget_name].value = int(params[key])
            except Exception:
                pass


def aplicar_setup_parametrico_celula3_por_nome(nome):
    """
    Aplica automaticamente um setup salvo na célula 3.

    A partir desta versão, o setup da célula 3 é a fonte principal dos
    parâmetros térmicos e de trajetória. A célula 5 não exige reconfigurar
    esses sliders novamente.
    """
    global CURRENT_PREPROCESS_PARAMS

    if nome in (None, "Usar parâmetros atuais"):
        registrar_acao(
            "<b>Setup da célula 3:</b> usando parâmetros atuais/fallback da célula 5.",
            "info",
            widget=msg_param_setup if "msg_param_setup" in globals() else None
        )
        return True

    data = carregar_setups_parametricos_celula3()
    params = data.get(nome)

    if not isinstance(params, dict):
        registrar_acao(
            f"<b>Setup da célula 3 não encontrado:</b> {nome}",
            "erro",
            widget=msg_param_setup if "msg_param_setup" in globals() else None
        )
        return False

    if "CURRENT_PREPROCESS_PARAMS" in globals() and isinstance(CURRENT_PREPROCESS_PARAMS, dict):
        CURRENT_PREPROCESS_PARAMS.update(params)
    else:
        CURRENT_PREPROCESS_PARAMS = dict(params)

    aplicar_setup_celula3_nos_controles_ocultos_5(params)

    # Se a célula 3 estiver carregada no kernel, também atualiza os widgets dela.
    for fn_name in [
        "aplicar_params_nos_widgets",
        "aplicar_params_aos_widgets",
        "aplicar_parametros_nos_widgets",
        "aplicar_preprocess_params_nos_widgets",
    ]:
        fn = globals().get(fn_name)

        if callable(fn):
            try:
                fn(params)
            except Exception:
                pass

    registrar_acao(
        f"<b>Setup da célula 3 carregado:</b> {nome}<br>"
        "<b>Pré-processamento e trajetória agora vêm deste setup.</b><br>"
        f"<span style='font-size:12px'>{resumo_setup_celula3_5(params)}</span>",
        "ok",
        widget=msg_param_setup if "msg_param_setup" in globals() else None
    )
    return True


def atualizar_setups_parametricos_celula3(_=None):
    valor_antigo = dropdown_param_setup_3.value if "dropdown_param_setup_3" in globals() else None
    opts = nomes_setups_parametricos_celula3()
    dropdown_param_setup_3.options = opts

    if valor_antigo in opts:
        dropdown_param_setup_3.value = valor_antigo
    elif len(opts) > 1:
        dropdown_param_setup_3.value = opts[1]
    else:
        dropdown_param_setup_3.value = opts[0]

    registrar_acao(
        f"<b>Setups da célula 3 atualizados:</b> {max(0, len(opts) - 1)} encontrado(s).",
        "ok",
        widget=msg_param_setup if "msg_param_setup" in globals() else None
    )
    return True


def carregar_setup_parametrico_celula3(_=None):
    return aplicar_setup_parametrico_celula3_por_nome(dropdown_param_setup_3.value)


# ------------------------------------------------------------
# Listagem de portas e câmeras
# ------------------------------------------------------------
def listar_portas_seriais_disponiveis():
    portas = []

    garantir_pyserial_importado()

    if SERIAL_AVAILABLE and list_ports is not None:
        try:
            for p in list_ports.comports():
                label = f"{p.device}"
                desc = getattr(p, "description", "")

                if desc:
                    label += f" — {desc}"

                portas.append((label, p.device))
        except Exception:
            pass

    fallback = ["/dev/ttyACM0", "/dev/ttyACM1", "/dev/ttyUSB0", "/dev/ttyUSB1", "COM3", "COM4", "COM5"]
    existing = {v for _, v in portas}

    for p in fallback:
        if p not in existing:
            portas.append((p, p))

    return portas


def detectar_instalacao_arduino_ide():
    """
    Detecta se há algum Arduino IDE disponível no computador.
    Retorna um dicionário com status e comandos encontrados.
    """
    encontrados = []

    if shutil.which("arduino-ide"):
        encontrados.append(("Arduino IDE 2.x", "arduino-ide"))

    if shutil.which("arduino"):
        encontrados.append(("Arduino IDE clássico", "arduino"))

    if shutil.which("flatpak"):
        try:
            result = subprocess.run(
                ["flatpak", "list"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if "arduino" in result.stdout.lower():
                encontrados.append(("Arduino IDE via Flatpak", "flatpak run cc.arduino.IDE2"))
        except Exception:
            pass

    if shutil.which("snap"):
        try:
            result = subprocess.run(
                ["snap", "list"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if "arduino" in result.stdout.lower():
                encontrados.append(("Arduino IDE via Snap", "snap run arduino"))
        except Exception:
            pass

    return {
        "installed": len(encontrados) > 0,
        "found": encontrados,
        "has_snap": shutil.which("snap") is not None,
        "has_flatpak": shutil.which("flatpak") is not None,
    }


def html_passo_a_passo_instalacao_arduino():
    snap_cmds = """
<pre style='background:#f6f6f6;padding:8px;border-radius:6px'>
sudo snap install arduino
sudo usermod -a -G dialout $USER
</pre>
<p>Depois reinicie o computador ou faça logout/login.</p>
<pre style='background:#f6f6f6;padding:8px;border-radius:6px'>
arduino --version
snap run arduino
ls /dev/ttyACM*
ls /dev/ttyUSB*
</pre>
"""

    apt_cmds = """
<pre style='background:#f6f6f6;padding:8px;border-radius:6px'>
sudo apt update
sudo apt install arduino
sudo usermod -a -G dialout $USER
</pre>
<p>Depois reinicie o computador ou faça logout/login.</p>
"""

    return f"""
<div style='border:1px solid #d99;padding:10px;border-radius:8px;background:#fff8f8'>
<b>Arduino IDE não detectado neste computador.</b><br>
A célula não encontrou <code>arduino-ide</code>, <code>arduino</code>, Flatpak com Arduino
ou Snap com Arduino instalado.

<p><b>Instalação recomendada pelo Snap:</b></p>
{snap_cmds}

<p><b>Alternativa pelo APT:</b></p>
{apt_cmds}

<p><b>Observação:</b> o comando <code>dialout</code> libera o acesso à porta USB/serial do Arduino.
Sem isso, a IDE pode abrir, mas o upload ou a comunicação serial pode falhar por permissão.</p>
</div>
"""


def verificar_instalacao_arduino_ide(_=None):
    info = detectar_instalacao_arduino_ide()

    if info["installed"]:
        itens = "<br>".join([f"<b>{nome}</b>: <code>{cmd}</code>" for nome, cmd in info["found"]])
        registrar_acao(
            f"<b>Arduino IDE detectado.</b><br>{itens}",
            "ok",
            widget=msg_sketch if "msg_sketch" in globals() else None
        )
    else:
        registrar_acao(
            html_passo_a_passo_instalacao_arduino(),
            "erro",
            widget=msg_sketch if "msg_sketch" in globals() else None
        )


def listar_cameras_disponiveis_celula5():
    """
    Lista câmeras com nomes explícitos.

    Prioridade do rótulo:
    1. v4l2-ctl --list-devices, que costuma mostrar nomes como:
       usb-HP_HP_Webcam_HD-4110-video-index0
    2. /dev/v4l/by-id e /dev/v4l/by-path
    3. v4l2-ctl --device /dev/videoX --info
    4. fallback: /dev/videoX — câmera detectada

    O valor interno continua sendo o índice OpenCV inteiro.
    """
    def mapa_v4l2_list_devices():
        mapping = {}

        try:
            result = subprocess.run(
                ["v4l2-ctl", "--list-devices"],
                capture_output=True,
                text=True,
                timeout=2.0
            )

            txt = (result.stdout or "") + "\n" + (result.stderr or "")
            current_name = None

            for raw_line in txt.splitlines():
                line = raw_line.rstrip()

                if not line.strip():
                    current_name = None
                    continue

                # Linha sem indentação = nome do dispositivo.
                if not raw_line.startswith((" ", "\t")):
                    current_name = line.strip().rstrip(":")
                    continue

                # Linha indentada = uma ou mais portas /dev/videoX.
                dev = line.strip()

                if dev.startswith("/dev/video") and current_name:
                    try:
                        idx = int(dev.replace("/dev/video", "").strip())
                        mapping[idx] = current_name
                    except Exception:
                        pass

        except Exception:
            pass

        return mapping


    def nome_por_symlink(idx):
        dev = Path(f"/dev/video{idx}")

        candidatos = []

        for folder in [Path("/dev/v4l/by-id"), Path("/dev/v4l/by-path")]:
            try:
                if not folder.exists():
                    continue

                for link in folder.iterdir():
                    try:
                        if link.resolve() == dev.resolve():
                            candidatos.append(link.name)
                    except Exception:
                        pass

            except Exception:
                pass

        if candidatos:
            # by-id costuma ser mais legível do que by-path.
            candidatos = sorted(candidatos, key=lambda s: (0 if "usb" in s.lower() else 1, s))
            return candidatos[0]

        return None


    def nome_v4l2_info(idx):
        dev = f"/dev/video{idx}"

        try:
            result = subprocess.run(
                ["v4l2-ctl", "--device", dev, "--info"],
                capture_output=True,
                text=True,
                timeout=1.5
            )

            txt = (result.stdout or "") + "\n" + (result.stderr or "")

            card_type = None
            bus_info = None
            driver = None

            for line in txt.splitlines():
                line = line.strip()
                low = line.lower()

                if low.startswith("card type"):
                    card_type = line.split(":", 1)[-1].strip()

                elif low.startswith("bus info"):
                    bus_info = line.split(":", 1)[-1].strip()

                elif low.startswith("driver name"):
                    driver = line.split(":", 1)[-1].strip()

            parts = [p for p in [card_type, bus_info, driver] if p]

            if parts:
                return " — ".join(parts)

        except Exception:
            pass

        return None


    nomes_por_idx = mapa_v4l2_list_devices()
    cams = []

    for idx in range(30):
        dev = Path(f"/dev/video{idx}")

        if not dev.exists():
            continue

        # Nome mais descritivo possível
        nome = (
            nomes_por_idx.get(idx)
            or nome_por_symlink(idx)
            or nome_v4l2_info(idx)
            or "câmera detectada"
        )

        cap = cv2.VideoCapture(idx)

        if cap.isOpened():
            ok, frame = cap.read()

            if ok and frame is not None:
                h, w = frame.shape[:2]
                cams.append((f"/dev/video{idx} — {nome} — {w}x{h}", idx))
            else:
                cams.append((f"/dev/video{idx} — {nome} — sem frame inicial", idx))

        cap.release()

    # Fallback final: se por algum motivo /dev/video não apareceu,
    # usa a função de listagem da célula 4, mas mantendo /dev/video no rótulo.
    if not cams and "listar_cameras" in globals():
        try:
            raw = listar_cameras(max_index=20)

            for label, idx in raw:
                if isinstance(idx, int) and idx >= 0:
                    cams.append((f"/dev/video{idx} — {label}", idx))

            if cams:
                return cams

        except Exception:
            pass

    if not cams:
        cams = [("Nenhuma câmera detectada", -1)]

    return cams


# ------------------------------------------------------------
# Widgets
# ------------------------------------------------------------
setups_arduino_dict = carregar_setups_arduino()
setup_options_arduino = ["Nenhum setup Arduino"] + sorted(setups_arduino_dict.keys())

dropdown_setup_arduino = widgets.Dropdown(options=setup_options_arduino, value=setup_options_arduino[0], description="Setup:", layout=widgets.Layout(width="520px"))
text_setup_arduino_nome = widgets.Text(value="", placeholder="nome do setup de calibração", description="Nome:", layout=widgets.Layout(width="420px"))
button_save_setup_arduino = widgets.Button(description="Salvar setup", button_style="success", layout=widgets.Layout(width="150px"))
button_delete_setup_arduino = widgets.Button(description="Excluir setup", button_style="danger", layout=widgets.Layout(width="150px"))

dropdown_param_setup_3 = widgets.Dropdown(
    options=nomes_setups_parametricos_celula3(),
    value=nomes_setups_parametricos_celula3()[0],
    description="Setup cél. 3:",
    layout=widgets.Layout(width="520px")
)

button_refresh_param_setup_3 = widgets.Button(
    description="Atualizar setups",
    button_style="info",
    layout=widgets.Layout(width="160px")
)

button_load_param_setup_3 = widgets.Button(
    description="Carregar setup",
    button_style="success",
    layout=widgets.Layout(width="160px")
)


button_create_sketch = widgets.Button(description="Criar/atualizar sketch", button_style="info", layout=widgets.Layout(width="190px"))
button_open_vscode = widgets.Button(description="Abrir no VS Code", layout=widgets.Layout(width="160px"))
button_check_arduino_install = widgets.Button(description="Verificar instalação IDE", button_style="warning", layout=widgets.Layout(width="210px"))
button_open_arduino_ide = widgets.Button(description="Abrir no Arduino IDE", layout=widgets.Layout(width="190px"))

serial_options = listar_portas_seriais_disponiveis()
dropdown_serial_port = widgets.Dropdown(options=serial_options, value=serial_options[0][1], description="Arduino:", layout=widgets.Layout(width="520px"))
button_refresh_serial = widgets.Button(description="Atualizar portas", button_style="info", layout=widgets.Layout(width="160px"))
button_install_pyserial = widgets.Button(description="Instalar pyserial", button_style="warning", layout=widgets.Layout(width="160px"))
serial_baud_widget = widgets.IntText(value=115200, description="Baud:", layout=widgets.Layout(width="170px"))
button_check_serial = widgets.Button(description="Verificar porta", button_style="warning", layout=widgets.Layout(width="150px"))
button_connect_serial = widgets.Button(description="Conectar", button_style="success", layout=widgets.Layout(width="120px"))
button_disconnect_serial = widgets.Button(description="Desconectar", button_style="danger", disabled=True, layout=widgets.Layout(width="130px"))
button_zero_servos = widgets.Button(description="Zerar servos", button_style="warning", layout=widgets.Layout(width="130px"))

initial_servo_x = widgets.FloatSlider(
    value=90.0,
    min=0.0,
    max=180.0,
    step=1.0,
    description="X inicial:",
    continuous_update=True,
    layout=widgets.Layout(width="260px")
)

initial_servo_y = widgets.FloatSlider(
    value=90.0,
    min=0.0,
    max=180.0,
    step=1.0,
    description="Y inicial:",
    continuous_update=True,
    layout=widgets.Layout(width="260px")
)

button_center_mount_servos = widgets.Button(
    description="Posicionar 90/90",
    button_style="warning",
    layout=widgets.Layout(width="160px")
)

button_apply_initial_to_calib = widgets.Button(
    description="Usar como base",
    button_style="info",
    layout=widgets.Layout(width="150px")
)


camera_options_5 = listar_cameras_disponiveis_celula5()
dropdown_cam1_5 = widgets.Dropdown(options=camera_options_5, value=camera_options_5[0][1], description="Câmera 1:", layout=widgets.Layout(width="440px"))
dropdown_cam2_5 = widgets.Dropdown(options=camera_options_5, value=camera_options_5[min(1, len(camera_options_5) - 1)][1], description="Câmera 2:", layout=widgets.Layout(width="440px"))
button_refresh_cameras_5 = widgets.Button(description="Atualizar câmeras", button_style="info", layout=widgets.Layout(width="180px"))

image_width_widget = widgets.IntText(value=640, description="Largura:", layout=widgets.Layout(width="180px"))
image_height_widget = widgets.IntText(value=480, description="Altura:", layout=widgets.Layout(width="180px"))
fps_widget_5 = widgets.IntText(value=30, description="FPS:", layout=widgets.Layout(width="140px"))
loop_interval_widget_5 = widgets.FloatSlider(value=0.05, min=0.02, max=0.50, step=0.01, description="Intervalo:", readout_format=".2f", continuous_update=False, layout=widgets.Layout(width="320px"))

dropdown_target_source = widgets.Dropdown(options=[("Mira do planejador", "aim_pt"), ("Impacto previsto", "impact_pt")], value="aim_pt", description="Enviar:", layout=widgets.Layout(width="310px"))
deadband_widget = widgets.FloatSlider(value=0.5, min=0.0, max=10.0, step=0.1, description="Deadband:", readout_format=".1f", continuous_update=False, layout=widgets.Layout(width="320px"))
pressure_calib_widget = widgets.FloatSlider(value=22.0, min=1.0, max=150.0, step=0.5, description="Pressão calib.:", readout_format=".1f", continuous_update=False, layout=widgets.Layout(width="360px"))

checkbox_send_servo_with_slider = widgets.Checkbox(value=True, description="Mover Arduino ao ajustar sliders", indent=False, layout=widgets.Layout(width="280px"))
checkbox_send_during_camera = widgets.Checkbox(value=True, description="Enviar mira ao Arduino durante simulação", indent=False, layout=widgets.Layout(width="330px"))

checkbox_invert_hot_mask_5 = widgets.Checkbox(
    value=False,
    description="Inverter máscara térmica",
    indent=False,
    layout=widgets.Layout(width="230px")
)

sync_flush_frames_widget = widgets.IntSlider(
    value=1,
    min=0,
    max=8,
    step=1,
    description="Flush sync:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

stereo_search_margin_widget = widgets.IntSlider(
    value=70,
    min=10,
    max=240,
    step=10,
    description="Busca Y:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

stereo_template_pad_widget = widgets.IntSlider(
    value=12,
    min=0,
    max=60,
    step=2,
    description="Pad ROI:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

jet_erase_radius_widget = widgets.IntSlider(
    value=16,
    min=2,
    max=80,
    step=1,
    description="Raio apagar:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

checkbox_draw_hot_contours_5 = widgets.Checkbox(
    value=True,
    description="Contorno quente",
    indent=False,
    layout=widgets.Layout(width="180px")
)

checkbox_draw_hot_boxes_5 = widgets.Checkbox(
    value=False,
    description="Bbox foco",
    indent=False,
    layout=widgets.Layout(width="150px")
)

checkbox_show_region_labels_5 = widgets.Checkbox(
    value=False,
    description="Labels debug",
    indent=False,
    layout=widgets.Layout(width="160px")
)

stereo_match_every_widget = widgets.IntSlider(
    value=15,
    min=1,
    max=30,
    step=1,
    description="Match a cada:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

stereo_top_regions_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=12,
    step=1,
    description="Top regiões:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

distance_correction_widget = widgets.FloatSlider(
    value=1.00,
    min=0.10,
    max=5.00,
    step=0.01,
    description="Fator dist.:",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)


area_threshold_offset_widget_5 = widgets.IntSlider(
    value=2500,
    min=0,
    max=50000,
    step=100,
    description="Área p/ offset:",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

offset_y_widget_5 = widgets.IntSlider(
    value=8,
    min=-80,
    max=80,
    step=1,
    description="Offset Y:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

route_step_widget_5 = widgets.IntSlider(
    value=18,
    min=2,
    max=80,
    step=1,
    description="Passo rota:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

centroid_hold_widget_5 = widgets.IntSlider(
    value=4,
    min=1,
    max=60,
    step=1,
    description="Perman. centróide:",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

aim_speed_widget_5 = widgets.IntSlider(
    value=15,
    min=1,
    max=80,
    step=1,
    description="Vel. mira:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)


thermal_seg_mode_widget = widgets.Dropdown(
    options=[
        ("HSV quente (paleta térmica)", "hsv_hot"),
        ("Cinza / limiar simples", "gray_threshold"),
    ],
    value="hsv_hot",
    description="Segmentação:",
    layout=widgets.Layout(width="360px")
)

stereo_calib_path_text = widgets.Text(
    value=str(BASE_DIR / "parameters_setups" / "stereo_calibration2.npz") if "BASE_DIR" in globals() else "stereo_calibration2.npz",
    description="Calibração:",
    placeholder="caminho para stereo_calibration2.npz",
    layout=widgets.Layout(width="620px")
)

checkbox_use_stereo_calib_5 = widgets.Checkbox(
    value=True,
    description="Usar calibração estéreo",
    indent=False,
    layout=widgets.Layout(width="230px")
)

checkbox_rectify_video_5 = widgets.Checkbox(
    value=False,
    description="Retificar vídeo",
    indent=False,
    layout=widgets.Layout(width="170px")
)


calib_width_widget = widgets.IntText(
    value=1920,
    description="Larg. calib:",
    layout=widgets.Layout(width="170px")
)

calib_height_widget = widgets.IntText(
    value=1080,
    description="Alt. calib:",
    layout=widgets.Layout(width="170px")
)

button_load_stereo_calib_5 = widgets.Button(
    description="Carregar calibração",
    button_style="info",
    layout=widgets.Layout(width="190px")
)

rectify_alpha_widget = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Zoom retif.:",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="300px")
)




button_apply_calibration = widgets.Button(description="Aplicar calibração do jato", button_style="primary", layout=widgets.Layout(width="230px"))
button_open_cameras_5 = widgets.Button(description="Abrir câmeras + simular", button_style="success", layout=widgets.Layout(width="210px"))
button_stop_cameras_5 = widgets.Button(description="Parar", button_style="danger", disabled=True, layout=widgets.Layout(width="120px"))
button_play_jet_5 = widgets.Button(description="Play jato", button_style="primary", layout=widgets.Layout(width="130px"))
button_stop_jet_5 = widgets.Button(description="Stop jato", layout=widgets.Layout(width="120px"))
button_reset_jet_5 = widgets.Button(description="Resetar imagem", button_style="warning", layout=widgets.Layout(width="160px"))

checkbox_auto_recover_camera = widgets.Checkbox(
    value=True,
    description="Recuperar câmera",
    indent=False,
    layout=widgets.Layout(width="170px")
)

checkbox_reset_when_no_focus = widgets.Checkbox(
    value=True,
    description="Liberar rota sem foco",
    indent=False,
    layout=widgets.Layout(width="190px")
)

checkbox_replan_when_focus_returns = widgets.Checkbox(
    value=True,
    description="Replanejar novo foco",
    indent=False,
    layout=widgets.Layout(width="190px")
)

checkbox_human_safety_yolo = widgets.Checkbox(
    value=True,
    description="Segurança humano YOLO",
    indent=False,
    layout=widgets.Layout(width="220px")
)

human_yolo_model_text = widgets.Text(
    value="yolov8n.pt",
    description="Modelo YOLO:",
    placeholder="ex.: yolov8n.pt ou caminho local .pt",
    layout=widgets.Layout(width="420px")
)

human_conf_widget = widgets.FloatSlider(
    value=0.45,
    min=0.10,
    max=0.90,
    step=0.05,
    description="Conf. humano:",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="330px")
)

human_check_every_widget = widgets.IntSlider(
    value=8,
    min=1,
    max=60,
    step=1,
    description="YOLO a cada:",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

human_yolo_width_widget = widgets.IntSlider(
    value=416,
    min=224,
    max=960,
    step=32,
    description="Larg. YOLO:",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

jpeg_quality_widget_5 = widgets.IntSlider(
    value=65,
    min=35,
    max=95,
    step=5,
    description="JPEG:",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)

panel_scale_widget_5 = widgets.FloatSlider(
    value=0.75,
    min=0.35,
    max=1.00,
    step=0.05,
    description="Escala painel:",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

status_update_every_widget_5 = widgets.IntSlider(
    value=8,
    min=1,
    max=60,
    step=1,
    description="Status a cada:",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

scene_check_every_widget_5 = widgets.IntSlider(
    value=10,
    min=1,
    max=90,
    step=1,
    description="Cena a cada:",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

button_load_human_yolo = widgets.Button(
    description="Carregar YOLO humano",
    button_style="info",
    layout=widgets.Layout(width="210px")
)





video_widget_5 = widgets.Image(format="jpg", width=1500)

reference_camera_widget = widgets.Image(format="jpg", width=760)

button_open_reference_camera = widgets.Button(
    description="Abrir câmera referência",
    button_style="success",
    layout=widgets.Layout(width="190px")
)

button_stop_reference_camera = widgets.Button(
    description="Parar câmera referência",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(width="190px")
)
status_arduino = widgets.HTML(value="<b>Status geral:</b> aguardando.")


def criar_msg_widget():
    return widgets.HTML(
        value="<span style='color:#666'>Aguardando ação.</span>",
        layout=widgets.Layout(width="520px")
    )


msg_sketch = criar_msg_widget()
msg_setup = criar_msg_widget()
msg_param_setup = criar_msg_widget()
msg_serial = criar_msg_widget()
msg_initial_position = criar_msg_widget()
msg_calib = criar_msg_widget()
msg_camera = criar_msg_widget()
msg_stereo_calib = criar_msg_widget()
msg_human_safety = criar_msg_widget()

CURRENT_MSG_WIDGET = {"widget": None}


def registrar_acao(msg, nivel="info", widget=None):
    """
    Mostra uma única mensagem atualizada para o grupo de botões em uso.
    """
    cores = {
        "info": "#1f77b4",
        "ok": "#1b8a3a",
        "aviso": "#b36b00",
        "erro": "#b00020",
    }

    cor = cores.get(nivel, "#333")
    html = f"<span style='color:{cor}'>{msg}</span>"

    target = widget or CURRENT_MSG_WIDGET.get("widget")

    if target is not None:
        target.value = html
    else:
        status_arduino.value = html

    status_arduino.value = html


def executar_com_feedback(nome_acao, func, widget=None):
    """
    Executa um botão e atualiza apenas a mensagem ao lado do grupo correspondente.
    """
    target = widget
    CURRENT_MSG_WIDGET["widget"] = target

    try:
        registrar_acao(f"<b>{nome_acao}:</b> executando...", "info", widget=target)
        result = func()

        # Se a função não substituiu a mensagem, conclui de forma genérica.
        # Se a função retornou False, não mostra "concluído" como sucesso.
        if target is not None and "executando..." in target.value:
            if result is False:
                registrar_acao(f"<b>{nome_acao}:</b> não confirmado.", "erro", widget=target)
            else:
                registrar_acao(f"<b>{nome_acao}:</b> concluído.", "ok", widget=target)

        return result

    except Exception as e:
        registrar_acao(
            f"<b>{nome_acao}:</b> não pôde ser concluído.<br>"
            f"<code>{type(e).__name__}: {e}</code>",
            "erro",
            widget=target
        )
        return None

    finally:
        CURRENT_MSG_WIDGET["widget"] = None


def servo_slider(value, description):
    return widgets.FloatSlider(value=float(value), min=0.0, max=180.0, step=0.5, description=description, readout_format=".1f", continuous_update=True, layout=widgets.Layout(width="350px"))


sx_tl = servo_slider(90, "Servo X:")
sy_tl = servo_slider(90, "Servo Y:")
sx_tr = servo_slider(90, "Servo X:")
sy_tr = servo_slider(90, "Servo Y:")
sx_bl = servo_slider(90, "Servo X:")
sy_bl = servo_slider(90, "Servo Y:")

button_start_tl = widgets.Button(description="Iniciar sup. esq.", button_style="info", layout=widgets.Layout(width="150px"))
button_send_tl = widgets.Button(description="Enviar sup. esq.", button_style="success", layout=widgets.Layout(width="150px"))
button_start_tr = widgets.Button(description="Iniciar sup. dir.", button_style="info", layout=widgets.Layout(width="150px"))
button_send_tr = widgets.Button(description="Enviar sup. dir.", button_style="success", layout=widgets.Layout(width="150px"))
button_start_bl = widgets.Button(description="Iniciar inf. esq.", button_style="info", layout=widgets.Layout(width="150px"))
button_send_bl = widgets.Button(description="Enviar inf. esq.", button_style="success", layout=widgets.Layout(width="150px"))


# ------------------------------------------------------------
# Serial
# ------------------------------------------------------------
def serial_is_open():
    ser = ARDUINO_STATE.get("serial")
    return ser is not None and getattr(ser, "is_open", False)


def refresh_serial_ports(_=None):
    opts = listar_portas_seriais_disponiveis()
    dropdown_serial_port.options = opts

    if opts:
        dropdown_serial_port.value = opts[0][1]

    registrar_acao(f"<b>Portas seriais encontradas:</b> {len(opts)}", "ok")


def verificar_porta_arduino(_=None):
    """
    Verifica se a porta selecionada é a porta correta do Arduino com o sketch BLAZE.

    Critério principal:
    - envia PING;
    - espera BLAZE_OK.

    Critério secundário:
    - envia X90.0,Y90.0,P22.0;
    - espera OK.
    """
    target_widget = msg_serial if "msg_serial" in globals() else None

    ok_serial, import_error = garantir_pyserial_importado()

    if not ok_serial:
        registrar_acao(
            "<b>Porta não verificada:</b> pyserial não está disponível neste kernel do Jupyter.<br>"
            f"<b>Python do kernel atual:</b><br><code>{sys.executable}</code><br>"
            f"<b>Erro de importação:</b><br><code>{type(import_error).__name__}: {import_error}</code><br>"
            "Clique em <b>Instalar pyserial</b> nesta própria célula.",
            "erro",
            widget=target_widget
        )
        return False

    if serial_is_open():
        registrar_acao(
            "<b>Porta não verificada:</b> a serial já está conectada.<br>"
            "Clique em <b>Desconectar</b> antes de testar outra porta.",
            "aviso",
            widget=target_widget
        )
        return False

    porta = dropdown_serial_port.value
    baud = int(serial_baud_widget.value)

    registrar_acao(
        f"<b>Verificando porta:</b> <code>{porta}</code><br>"
        f"Baud: <code>{baud}</code>. Enviando <code>PING</code>...",
        "info",
        widget=target_widget
    )

    try:
        ser = serial.Serial(port=porta, baudrate=baud, timeout=0.25)

        # Arduino Uno reinicia ao abrir a serial.
        time.sleep(2.2)

        linhas = []

        # Lê mensagens iniciais do boot/reset.
        t0 = time.time()
        while time.time() - t0 < 1.0:
            line = ser.readline().decode("utf-8", errors="ignore").strip()
            if line:
                linhas.append(line)

        ser.write(b"PING\n")
        time.sleep(0.25)

        blaze_ok = None
        t_ping = time.time()

        while time.time() - t_ping < 1.5:
            line = ser.readline().decode("utf-8", errors="ignore").strip()
            if line:
                linhas.append(line)
                if line.startswith("BLAZE_OK"):
                    blaze_ok = line
                    break

        teste_cmd = "X90.0,Y90.0,P22.0"
        ser.write((teste_cmd + "\n").encode("utf-8"))
        time.sleep(0.25)

        resposta_ok = None
        t_cmd = time.time()

        while time.time() - t_cmd < 1.5:
            line = ser.readline().decode("utf-8", errors="ignore").strip()
            if line:
                linhas.append(line)
                if line.startswith("OK"):
                    resposta_ok = line
                    break

        ser.close()

        ultimas = "<br>".join([f"<code>{l}</code>" for l in linhas[-12:]])
        linhas_texto = "\n".join(linhas).lower()

        if blaze_ok or resposta_ok:
            registrar_acao(
                f"<b>PORTA CONFIRMADA:</b> <code>{porta}</code><br>"
                "O Arduino respondeu ao protocolo BLAZE.<br>"
                f"<b>Resposta PING:</b> <code>{blaze_ok if blaze_ok else 'não recebida'}</code><br>"
                f"<b>Resposta comando:</b> <code>{resposta_ok if resposta_ok else 'não recebida'}</code><br>"
                f"<b>Últimas leituras:</b><br>{ultimas}",
                "ok",
                widget=target_widget
            )
            return True

        if "pot" in linhas_texto or "pot 1x" in linhas_texto or "pot 1y" in linhas_texto:
            registrar_acao(
                f"<b>PORTA RESPONDE, MAS NÃO COM O SKETCH BLAZE:</b> <code>{porta}</code><br>"
                "A resposta contém textos de outro programa, por exemplo <code>Pot 1X</code> / <code>Pot 1Y</code>.<br><br>"
                "<b>Isso indica que o Arduino está rodando outro sketch na porta selecionada.</b><br>"
                "Mesmo que o arquivo aberto no editor seja o BLAZE, ele pode não ter sido enviado para esta placa/porta.<br><br>"
                "<b>Teste manual:</b><br>"
                "No Arduino IDE, abra o Serial Monitor em <code>115200</code>, envie <code>PING</code> e pressione Enter.<br>"
                "O sketch correto deve responder <code>BLAZE_OK UNO_MG90S_V2</code>.<br><br>"
                f"<b>Leituras recebidas:</b><br>{ultimas}",
                "aviso",
                widget=target_widget
            )
            return False

        if len(linhas) > 0:
            registrar_acao(
                f"<b>ARDUINO ENCONTRADO, MAS SKETCH BLAZE NÃO CONFIRMADO:</b> <code>{porta}</code><br>"
                "A porta está respondendo, mas não retornou <code>BLAZE_OK</code> nem <code>OK</code>.<br><br>"
                f"<b>Leituras recebidas:</b><br>{ultimas}<br><br>"
                "<b>Verifique no Arduino IDE:</b> placa Arduino Uno, porta correta e upload concluído sem erro.",
                "aviso",
                widget=target_widget
            )
            return False

        registrar_acao(
            f"<b>PORTA NÃO CONFIRMADA:</b> <code>{porta}</code><br>"
            "A porta abriu, mas não retornou nenhuma resposta útil.<br>"
            "Verifique se a placa está conectada, se o sketch foi enviado e se o baud é 115200.",
            "erro",
            widget=target_widget
        )
        return False

    except Exception as e:
        registrar_acao(
            f"<b>PORTA NÃO CONFIRMADA:</b> <code>{porta}</code><br>"
            "Não foi possível abrir/testar esta porta.<br>"
            f"<code>{type(e).__name__}: {e}</code>",
            "erro",
            widget=target_widget
        )
        return False


def connect_serial(_=None):
    ok_serial, import_error = garantir_pyserial_importado()

    if not ok_serial:
        registrar_acao(
            "<b>Erro:</b> pyserial não está disponível neste kernel do Jupyter.<br>"
            f"<b>Python do kernel atual:</b><br><code>{sys.executable}</code><br>"
            f"<b>Erro de importação:</b><br><code>{type(import_error).__name__}: {import_error}</code><br>"
            "Clique em <b>Instalar pyserial</b> nesta própria célula.",
            "erro",
            widget=msg_serial if "msg_serial" in globals() else None
        )
        return False

    if serial_is_open():
        registrar_acao("<b>Status:</b> Arduino já conectado.", "aviso")
        return

    try:
        ser = serial.Serial(port=dropdown_serial_port.value, baudrate=int(serial_baud_widget.value), timeout=0.05)
        time.sleep(2.0)

        ARDUINO_STATE["serial"] = ser
        ARDUINO_STATE["connected"] = True
        button_connect_serial.disabled = True
        button_disconnect_serial.disabled = False
        status_arduino.value = f"<b>Arduino conectado:</b> <code>{dropdown_serial_port.value}</code> @ {int(serial_baud_widget.value)} baud."

    except Exception as e:
        ARDUINO_STATE["serial"] = None
        ARDUINO_STATE["connected"] = False
        registrar_acao(f"<b>Erro ao conectar:</b> {type(e).__name__}: {e}", "erro")


def disconnect_serial(_=None):
    ARDUINO_STATE["camera_running"] = False

    ser = ARDUINO_STATE.get("serial")
    if ser is not None:
        try:
            ser.close()
        except Exception:
            pass

    ARDUINO_STATE["serial"] = None
    ARDUINO_STATE["connected"] = False
    button_connect_serial.disabled = False
    button_disconnect_serial.disabled = True
    registrar_acao("<b>Arduino desconectado.</b>", "ok")


def format_command(servo_x, servo_y, pressure=None):
    if pressure is None:
        pressure = float(pressure_calib_widget.value)

    return f"X{float(servo_x):.1f},Y{float(servo_y):.1f},P{float(pressure):.1f}\n"


def send_servo_command(servo_x, servo_y, force=False):
    if not serial_is_open():
        status_arduino.value = "<b>Arduino não conectado.</b>"
        return False

    servo_x = float(np.clip(float(servo_x), 0.0, 180.0))
    servo_y = float(np.clip(float(servo_y), 0.0, 180.0))

    last = ARDUINO_STATE.get("last_angles")
    deadband = float(deadband_widget.value)

    if not force and last is not None:
        if abs(servo_x - float(last[0])) < deadband and abs(servo_y - float(last[1])) < deadband:
            return False

    cmd = format_command(servo_x, servo_y)

    try:
        ser = ARDUINO_STATE["serial"]
        ser.write(cmd.encode("utf-8"))
        ser.flush()

        ARDUINO_STATE["last_angles"] = (servo_x, servo_y)
        ARDUINO_STATE["last_command"] = cmd.strip()
        ARDUINO_STATE["send_count"] += 1
        return True

    except Exception as e:
        registrar_acao(
            f"<b>Erro ao enviar comando serial:</b> {type(e).__name__}: {e}<br>"
            f"<b>Último comando:</b> <code>{cmd.strip()}</code>",
            "erro",
            widget=msg_serial if "msg_serial" in globals() else None
        )
        return False




def posicionar_servos_montagem_central(_=None):
    """
    Move os servos para a posição inicial de montagem.
    Por padrão: X=90 e Y=90.
    """
    if not serial_is_open():
        registrar_acao(
            "<b>Conecte o Arduino antes de posicionar os servos.</b>",
            "erro",
            widget=msg_initial_position if "msg_initial_position" in globals() else None
        )
        return False

    x0 = float(initial_servo_x.value)
    y0 = float(initial_servo_y.value)

    ok = send_servo_command(x0, y0, force=True)

    if ok:
        registrar_acao(
            f"<b>Servos posicionados para montagem:</b> X={x0:.1f}, Y={y0:.1f}.<br>"
            "Agora monte fisicamente o jato apontando para o centro da imagem.",
            "ok",
            widget=msg_initial_position if "msg_initial_position" in globals() else None
        )
    else:
        registrar_acao(
            "<b>Falha ao posicionar servos.</b>",
            "erro",
            widget=msg_initial_position if "msg_initial_position" in globals() else None
        )

    return ok


def aplicar_posicao_inicial_nas_barras(_=None):
    """
    Copia a posição inicial para todos os sliders de calibração.
    Isso faz cada ponto começar do centro mecânico escolhido.
    """
    x0 = float(initial_servo_x.value)
    y0 = float(initial_servo_y.value)

    for sx, sy in [(sx_tl, sy_tl), (sx_tr, sy_tr), (sx_bl, sy_bl)]:
        sx.value = x0
        sy.value = y0

    registrar_acao(
        f"<b>Base aplicada à calibração:</b> todos os pontos começam em X={x0:.1f}, Y={y0:.1f}.",
        "ok",
        widget=msg_initial_position if "msg_initial_position" in globals() else None
    )

    return True


def zerar_servos_e_sliders(_=None):
    """
    Coloca todos os sliders de calibração em 0 e manda X=0/Y=0 para o Arduino.
    Deve ser usado antes de iniciar a calibração.
    """
    CALIBRATION_SERVO_STATE["active_point"] = None

    ok = send_servo_command(0.0, 0.0, force=True)

    registrar_acao(
        "<b>Servos enviados para zero:</b> X=0, Y=0.<br>"
        "Use apenas se a montagem permitir movimento até 0° sem colisão.",
        "ok" if ok else "erro",
        widget=msg_calib if "msg_calib" in globals() else None
    )

    return ok


def ponto_ativo_label():
    mapa = {"tl": "superior esquerdo", "tr": "superior direito", "bl": "inferior esquerdo", None: "nenhum"}
    return mapa.get(CALIBRATION_SERVO_STATE.get("active_point"), "nenhum")


def set_active_calibration_point(point_key):
    if not serial_is_open():
        registrar_acao("<b>Conecte o Arduino antes de iniciar a calibração.</b>", "erro", widget=msg_calib if "msg_calib" in globals() else None)
        return False

    CALIBRATION_SERVO_STATE["active_point"] = point_key

    # Inicia o ajuste do ponto usando o valor atual dos sliders.
    # Recomenda-se aplicar a posição inicial 90/90 como base antes.
    if point_key == "tl":
        send_servo_command(sx_tl.value, sy_tl.value, force=True)
    elif point_key == "tr":
        send_servo_command(sx_tr.value, sy_tr.value, force=True)
    elif point_key == "bl":
        send_servo_command(sx_bl.value, sy_bl.value, force=True)

    registrar_acao(
        f"<b>Calibração iniciada:</b> ponto {ponto_ativo_label()}.<br>"
        "Mova os sliders X/Y deste ponto. Os servos acompanharão em tempo real.<br>"
        "Quando a mira coincidir com o canto desejado na imagem da câmera de referência, clique em <b>Enviar</b>.",
        "info",
        widget=msg_calib if "msg_calib" in globals() else None
    )
    return True


def finalizar_calibracao_ponto(point_key, sx, sy, nome):
    estava_ativo = CALIBRATION_SERVO_STATE.get("active_point") == point_key
    CALIBRATION_SERVO_STATE["active_point"] = None
    apply_jet_calibration()

    if estava_ativo:
        registrar_acao(
            f"<b>{nome} calibrado:</b> X={float(sx):.1f}, Y={float(sy):.1f}.<br>"
            "Envio em tempo real deste ponto interrompido.",
            "ok",
            widget=msg_calib if "msg_calib" in globals() else None
        )
    else:
        registrar_acao(
            f"<b>{nome} salvo:</b> X={float(sx):.1f}, Y={float(sy):.1f}.<br>"
            "Para mover o servo em tempo real, use <b>Iniciar</b> antes de ajustar.",
            "aviso",
            widget=msg_calib if "msg_calib" in globals() else None
        )
    return True


def slider_calib_changed(change=None):
    if not checkbox_send_servo_with_slider.value:
        return

    active = CALIBRATION_SERVO_STATE.get("active_point")
    if active is None:
        return

    owner = change.get("owner", None) if isinstance(change, dict) else None

    if active == "tl" and owner in (sx_tl, sy_tl):
        send_servo_command(sx_tl.value, sy_tl.value, force=True)
    elif active == "tr" and owner in (sx_tr, sy_tr):
        send_servo_command(sx_tr.value, sy_tr.value, force=True)
    elif active == "bl" and owner in (sx_bl, sy_bl):
        send_servo_command(sx_bl.value, sy_bl.value, force=True)


# ------------------------------------------------------------
# Mapeamento imagem -> servo
# ------------------------------------------------------------
def get_mapping_params_from_widgets():
    return {
        "serial_port": dropdown_serial_port.value,
        "serial_baud": int(serial_baud_widget.value),
        "image_width": int(image_width_widget.value),
        "image_height": int(image_height_widget.value),
        "fps": int(fps_widget_5.value),
        "loop_interval": float(loop_interval_widget_5.value),
        "target_source": dropdown_target_source.value,
        "deadband": float(deadband_widget.value),
        "pressure_calib": float(pressure_calib_widget.value),
        "send_during_camera": bool(checkbox_send_during_camera.value),
        "invert_hot_mask": bool(checkbox_invert_hot_mask_5.value),
        "thermal_seg_mode": thermal_seg_mode_widget.value,
        "sync_flush_frames": int(sync_flush_frames_widget.value),
        "stereo_search_margin": int(stereo_search_margin_widget.value),
        "stereo_template_pad": int(stereo_template_pad_widget.value),
        "jet_erase_radius": int(jet_erase_radius_widget.value),
        "draw_hot_contours": bool(checkbox_draw_hot_contours_5.value),
        "draw_hot_boxes": bool(checkbox_draw_hot_boxes_5.value),
        "show_region_labels": bool(checkbox_show_region_labels_5.value),
        "stereo_match_every": int(stereo_match_every_widget.value),
        "stereo_top_regions": int(stereo_top_regions_widget.value),
        "distance_correction_factor": float(distance_correction_widget.value),
        "jpeg_quality": int(jpeg_quality_widget_5.value),
        "panel_scale": float(panel_scale_widget_5.value),
        "status_update_every": int(status_update_every_widget_5.value),
        "scene_check_every": int(scene_check_every_widget_5.value),
        "human_yolo_width": int(human_yolo_width_widget.value),

        "stereo_calibration": {
            "enabled": bool(checkbox_use_stereo_calib_5.value),
            "rectify_video": bool(checkbox_rectify_video_5.value),
            "path": stereo_calib_path_text.value,
            "calib_width": int(calib_width_widget.value),
            "calib_height": int(calib_height_widget.value),
            "rectify_alpha": float(rectify_alpha_widget.value),
        },

        # Setup paramétrico da célula 3 selecionado nesta célula
        "cell3_parametric_setup": dropdown_param_setup_3.value if "dropdown_param_setup_3" in globals() else "Usar parâmetros atuais",

        # Posição inicial de montagem
        "initial_position": {
            "servo": [float(initial_servo_x.value), float(initial_servo_y.value)]
        },

        "camera_recovery": {
            "auto_recover": bool(checkbox_auto_recover_camera.value),
            "reset_when_no_focus": bool(checkbox_reset_when_no_focus.value),
            "replan_when_focus_returns": bool(checkbox_replan_when_focus_returns.value),
        },

        "human_safety": {
            "enabled": bool(checkbox_human_safety_yolo.value),
            "model": human_yolo_model_text.value,
            "confidence": float(human_conf_widget.value),
            "check_every": int(human_check_every_widget.value),
        },

        # Pontos de calibração imagem -> servo
        "points": {
            "top_left": {"image": [0.0, 0.0], "servo": [float(sx_tl.value), float(sy_tl.value)]},
            "top_right": {"image": [float(image_width_widget.value), 0.0], "servo": [float(sx_tr.value), float(sy_tr.value)]},
            "bottom_left": {"image": [0.0, float(image_height_widget.value)], "servo": [float(sx_bl.value), float(sy_bl.value)]},
        }
    }


def apply_mapping_params_to_widgets(params):
    if not isinstance(params, dict):
        return

    port = params.get("serial_port", dropdown_serial_port.value)
    available_values = [v for _, v in dropdown_serial_port.options]

    if port in available_values:
        dropdown_serial_port.value = port

    serial_baud_widget.value = int(params.get("serial_baud", serial_baud_widget.value))
    image_width_widget.value = int(params.get("image_width", image_width_widget.value))
    image_height_widget.value = int(params.get("image_height", image_height_widget.value))
    fps_widget_5.value = int(params.get("fps", fps_widget_5.value))
    loop_interval_widget_5.value = float(params.get("loop_interval", loop_interval_widget_5.value))
    dropdown_target_source.value = params.get("target_source", dropdown_target_source.value)
    deadband_widget.value = float(params.get("deadband", deadband_widget.value))
    pressure_calib_widget.value = float(params.get("pressure_calib", pressure_calib_widget.value))
    checkbox_send_during_camera.value = bool(params.get("send_during_camera", checkbox_send_during_camera.value))
    checkbox_invert_hot_mask_5.value = bool(params.get("invert_hot_mask", checkbox_invert_hot_mask_5.value))
    thermal_seg_mode_widget.value = params.get("thermal_seg_mode", thermal_seg_mode_widget.value)
    sync_flush_frames_widget.value = int(params.get("sync_flush_frames", sync_flush_frames_widget.value))
    stereo_search_margin_widget.value = int(params.get("stereo_search_margin", stereo_search_margin_widget.value))
    stereo_template_pad_widget.value = int(params.get("stereo_template_pad", stereo_template_pad_widget.value))
    jet_erase_radius_widget.value = int(params.get("jet_erase_radius", jet_erase_radius_widget.value))
    checkbox_draw_hot_contours_5.value = bool(params.get("draw_hot_contours", checkbox_draw_hot_contours_5.value))
    checkbox_draw_hot_boxes_5.value = bool(params.get("draw_hot_boxes", checkbox_draw_hot_boxes_5.value))
    checkbox_show_region_labels_5.value = bool(params.get("show_region_labels", checkbox_show_region_labels_5.value))
    stereo_match_every_widget.value = int(params.get("stereo_match_every", stereo_match_every_widget.value))
    stereo_top_regions_widget.value = int(params.get("stereo_top_regions", stereo_top_regions_widget.value))
    distance_correction_widget.value = float(params.get("distance_correction_factor", distance_correction_widget.value))
    jpeg_quality_widget_5.value = int(params.get("jpeg_quality", jpeg_quality_widget_5.value))
    panel_scale_widget_5.value = float(params.get("panel_scale", panel_scale_widget_5.value))
    status_update_every_widget_5.value = int(params.get("status_update_every", status_update_every_widget_5.value))
    scene_check_every_widget_5.value = int(params.get("scene_check_every", scene_check_every_widget_5.value))
    human_yolo_width_widget.value = int(params.get("human_yolo_width", human_yolo_width_widget.value))
    if not setup_parametrico_ativo_celula3_5():
        area_threshold_offset_widget_5.value = int(params.get("area_threshold_offset", area_threshold_offset_widget_5.value))
        offset_y_widget_5.value = int(params.get("offset_y", offset_y_widget_5.value))
        route_step_widget_5.value = int(params.get("route_step", route_step_widget_5.value))
        centroid_hold_widget_5.value = int(params.get("centroid_hold", centroid_hold_widget_5.value))
        aim_speed_widget_5.value = int(params.get("aim_speed", aim_speed_widget_5.value))

    stereo_cfg = params.get("stereo_calibration", {})
    checkbox_use_stereo_calib_5.value = bool(stereo_cfg.get("enabled", checkbox_use_stereo_calib_5.value))
    checkbox_rectify_video_5.value = bool(stereo_cfg.get("rectify_video", checkbox_rectify_video_5.value))
    stereo_calib_path_text.value = stereo_cfg.get("path", stereo_calib_path_text.value)
    calib_width_widget.value = int(stereo_cfg.get("calib_width", calib_width_widget.value))
    calib_height_widget.value = int(stereo_cfg.get("calib_height", calib_height_widget.value))
    rectify_alpha_widget.value = float(stereo_cfg.get("rectify_alpha", rectify_alpha_widget.value))

    initial = params.get("initial_position", {}).get("servo", [initial_servo_x.value, initial_servo_y.value])
    initial_servo_x.value = float(initial[0])
    initial_servo_y.value = float(initial[1])

    cam_recovery = params.get("camera_recovery", {})
    checkbox_auto_recover_camera.value = bool(cam_recovery.get("auto_recover", checkbox_auto_recover_camera.value))
    checkbox_reset_when_no_focus.value = bool(cam_recovery.get("reset_when_no_focus", checkbox_reset_when_no_focus.value))
    checkbox_replan_when_focus_returns.value = bool(cam_recovery.get("replan_when_focus_returns", checkbox_replan_when_focus_returns.value))

    human = params.get("human_safety", {})
    checkbox_human_safety_yolo.value = bool(human.get("enabled", checkbox_human_safety_yolo.value))
    human_yolo_model_text.value = human.get("model", human_yolo_model_text.value)
    human_conf_widget.value = float(human.get("confidence", human_conf_widget.value))
    human_check_every_widget.value = int(human.get("check_every", human_check_every_widget.value))

    setup3 = params.get("cell3_parametric_setup", None)
    if setup3 is not None and "dropdown_param_setup_3" in globals():
        opts = list(dropdown_param_setup_3.options)
        if setup3 in opts:
            dropdown_param_setup_3.value = setup3

    pts = params.get("points", {})
    tl = pts.get("top_left", {}).get("servo", [sx_tl.value, sy_tl.value])
    tr = pts.get("top_right", {}).get("servo", [sx_tr.value, sy_tr.value])
    bl = pts.get("bottom_left", {}).get("servo", [sx_bl.value, sy_bl.value])

    sx_tl.value = float(tl[0])
    sy_tl.value = float(tl[1])
    sx_tr.value = float(tr[0])
    sy_tr.value = float(tr[1])
    sx_bl.value = float(bl[0])
    sy_bl.value = float(bl[1])


def compute_affine_mapping(params=None):
    if params is None:
        params = get_mapping_params_from_widgets()

    pts = params["points"]

    img = np.array([
        [pts["top_left"]["image"][0], pts["top_left"]["image"][1], 1.0],
        [pts["top_right"]["image"][0], pts["top_right"]["image"][1], 1.0],
        [pts["bottom_left"]["image"][0], pts["bottom_left"]["image"][1], 1.0],
    ], dtype=float)

    sx = np.array([pts["top_left"]["servo"][0], pts["top_right"]["servo"][0], pts["bottom_left"]["servo"][0]], dtype=float)
    sy = np.array([pts["top_left"]["servo"][1], pts["top_right"]["servo"][1], pts["bottom_left"]["servo"][1]], dtype=float)

    coef_x = np.linalg.solve(img, sx)
    coef_y = np.linalg.solve(img, sy)

    return coef_x, coef_y


def apply_jet_calibration(_=None):
    params = get_mapping_params_from_widgets()
    coef_x, coef_y = compute_affine_mapping(params)

    ARDUINO_MAPPING_ACTIVE["coef_x"] = coef_x
    ARDUINO_MAPPING_ACTIVE["coef_y"] = coef_y
    ARDUINO_MAPPING_ACTIVE["params"] = params

    ARDUINO_JET_PARAMS["pressure_calib"] = float(pressure_calib_widget.value)
    ARDUINO_STATE["calibration_applied"] = True

    status_arduino.value = (
        "<b>Calibração aplicada.</b><br>"
        f"Servo X = {coef_x[0]:.4f}*x + {coef_x[1]:.4f}*y + {coef_x[2]:.2f}<br>"
        f"Servo Y = {coef_y[0]:.4f}*x + {coef_y[1]:.4f}*y + {coef_y[2]:.2f}"
    )


def image_point_to_servo(point_xy):
    if point_xy is None:
        return None

    if ARDUINO_MAPPING_ACTIVE["coef_x"] is None or ARDUINO_MAPPING_ACTIVE["coef_y"] is None:
        apply_jet_calibration()

    params = ARDUINO_MAPPING_ACTIVE["params"] or get_mapping_params_from_widgets()

    x, y = float(point_xy[0]), float(point_xy[1])
    w = max(1.0, float(params.get("image_width", 640)))
    h = max(1.0, float(params.get("image_height", 480)))

    x = float(np.clip(x, 0, w))
    y = float(np.clip(y, 0, h))

    v = np.array([x, y, 1.0], dtype=float)

    sx = float(np.dot(ARDUINO_MAPPING_ACTIVE["coef_x"], v))
    sy = float(np.dot(ARDUINO_MAPPING_ACTIVE["coef_y"], v))

    return float(np.clip(sx, 0.0, 180.0)), float(np.clip(sy, 0.0, 180.0))


# ------------------------------------------------------------
# Pressão do jato: sobrescreve função da célula 4
# ------------------------------------------------------------
def atualizar_pressao_global(change=None):
    ARDUINO_JET_PARAMS["pressure_calib"] = float(pressure_calib_widget.value)


pressure_calib_widget.observe(atualizar_pressao_global, names="value")
atualizar_pressao_global()


def estimar_ponto_impacto(aim_pt, region, params, canvas_shape):
    H, W = canvas_shape[:2]

    if aim_pt is None:
        return None

    x, y = int(aim_pt[0]), int(aim_pt[1])
    z = region.get("depth_z") if isinstance(region, dict) else None

    if z is None or not np.isfinite(float(z)):
        dy = 0
    else:
        calib = max(1.0, float(ARDUINO_JET_PARAMS.get("pressure_calib", 22.0)))
        dy = int(np.clip(float(z) / calib, -80, 160))

    ix = int(np.clip(x, 0, W - 1))
    iy = int(np.clip(y + dy, 0, H - 1))
    return (ix, iy)


# ------------------------------------------------------------
# Simulação e envio
# ------------------------------------------------------------
def get_target_from_jet_state():
    if "JET_SIM_STATE" not in globals():
        return None

    source = dropdown_target_source.value
    pt = JET_SIM_STATE.get(source)

    if pt is None and source == "impact_pt":
        pt = JET_SIM_STATE.get("aim_pt")

    return pt


def maybe_send_target_to_arduino():
    if ARDUINO_STATE.get("human_detected", False):
        return False

    if not checkbox_send_during_camera.value or not serial_is_open():
        return False

    pt = get_target_from_jet_state()

    if pt is None:
        return False

    angles = image_point_to_servo(pt)

    if angles is None:
        return False

    sx, sy = angles
    sent = send_servo_command(sx, sy, force=False)
    ARDUINO_STATE["last_target"] = (float(pt[0]), float(pt[1]))
    return sent




# ------------------------------------------------------------
# Fallbacks independentes da célula 4
# ------------------------------------------------------------
def encode_jpg_bgr_local_5(img_bgr, quality=85):
    """
    Codifica uma imagem BGR para JPEG.
    Usada quando encode_jpg_bgr da célula 4 não existe.
    """
    ok, buf = cv2.imencode(
        ".jpg",
        img_bgr,
        [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)]
    )

    if not ok:
        return b""

    return buf.tobytes()


if "encode_jpg_bgr" not in globals():
    encode_jpg_bgr = encode_jpg_bgr_local_5


def parametros_processamento_independente_5(params=None):
    """
    Seleciona parâmetros básicos de segmentação.

    Prioriza o setup da célula 3, quando selecionado.
    """
    p = {}
    p.update(dict(params or {}))
    p.update(setup_parametrico_ativo_celula3_5())

    return {
        "temp_min": float(p.get("temp_filter_min_c", p.get("temp_min_c", p.get("temp_min", p.get("filter_min", 120))))),
        "temp_max": float(p.get("temp_filter_max_c", p.get("temp_max_c", p.get("temp_max", p.get("filter_max", 255))))),
        "blur": int(p.get("blur_ksize", p.get("blur", 3))),
        "open": int(p.get("open_iter", p.get("open", 1))),
        "close": int(p.get("close_iter", p.get("close", 1))),
        "dilate": int(p.get("dilate_iter", p.get("dilate", 0))),
        "min_area": int(p.get("min_area", 80)),
        "max_area": int(p.get("max_area", 999999)),
    }


def extrair_regioes_independente_5(frame_bgr, params=None):
    """
    Extrai focos quentes como componentes com máscara própria.

    A lógica agora volta a ficar compatível com a célula 3:
    - cada região possui máscara, área, centróide e bbox auxiliar;
    - regiões grandes usam rota pela borda inferior com offset;
    - regiões pequenas usam permanência no centróide.
    """
    p = parametros_processamento_independente_5(params)

    if thermal_seg_mode_widget.value == "hsv_hot":
        hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
        H, S, V = cv2.split(hsv)
        mask1 = ((H <= 45) | (H >= 165)) & (S >= 35) & (V >= 45)
        mask2 = (H <= 65) & (S >= 20) & (V >= 120)
        mask = ((mask1 | mask2).astype(np.uint8)) * 255
    else:
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        blur = int(p["blur"])
        if blur > 1:
            if blur % 2 == 0:
                blur += 1
            gray = cv2.GaussianBlur(gray, (blur, blur), 0)
        lo = int(np.clip(p["temp_min"], 0, 255))
        hi = int(np.clip(p["temp_max"], 0, 255))
        if hi <= lo:
            hi = 255
        mask = cv2.inRange(gray, lo, hi)

    if checkbox_invert_hot_mask_5.value:
        mask = cv2.bitwise_not(mask)

    kernel = np.ones((3, 3), np.uint8)
    if int(p["open"]) > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=int(p["open"]))
    if int(p["close"]) > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=int(p["close"]))
    if int(p["dilate"]) > 0:
        mask = cv2.dilate(mask, kernel, iterations=int(p["dilate"]))

    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), connectivity=8)
    regs = []
    min_area = int(p["min_area"])
    max_area = int(p["max_area"])

    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area or area > max_area:
            continue
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        bw = int(stats[i, cv2.CC_STAT_WIDTH])
        bh = int(stats[i, cv2.CC_STAT_HEIGHT])
        comp = labels == i
        comp_u8 = comp.astype(np.uint8) * 255
        contours, _ = cv2.findContours(comp_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contour = max(contours, key=cv2.contourArea) if contours else None
        cx = float(centroids[i][0])
        cy = float(centroids[i][1])
        regs.append({
            "label": i,
            "mask": comp,
            "contour": contour,
            "bbox": [x, y, x + bw, y + bh],
            "bbox_wh": (x, y, bw, bh),
            "bbox_prod": int(bw * bh),
            "centroid": [cx, cy],
            "cx": cx,
            "cy": cy,
            "area": area,
            "depth_z": None,
            "depth_proxy": None,
            "disparity_px": None,
            "match_score": None,
            "matched_bbox_cam2": None,
            "track_id": i,
        })

    regs.sort(key=lambda r: int(r.get("area", 0)), reverse=True)
    regs = regs[:max(1, int(stereo_top_regions_widget.value))]

    filtered = np.zeros_like(frame_bgr)
    filtered[mask > 0] = frame_bgr[mask > 0]
    return filtered, mask, regs


def remaining_colored_mask_5(img_bgr):
    if img_bgr is None:
        return None
    return (np.max(img_bgr, axis=2) > 12).astype(np.uint8)


def remaining_colored_pixels_5(img_bgr):
    mask = remaining_colored_mask_5(img_bgr)
    if mask is None:
        return 0
    return int(np.count_nonzero(mask))


def region_components_from_mask_5(mask, min_area=1):
    mask_u8 = (mask > 0).astype(np.uint8)
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    regs = []
    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        bw = int(stats[i, cv2.CC_STAT_WIDTH])
        bh = int(stats[i, cv2.CC_STAT_HEIGHT])
        comp = labels == i
        comp_u8 = comp.astype(np.uint8) * 255
        contours, _ = cv2.findContours(comp_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contour = max(contours, key=cv2.contourArea) if contours else None
        cx = float(centroids[i][0])
        cy = float(centroids[i][1])
        regs.append({
            "label": i,
            "mask": comp,
            "contour": contour,
            "bbox": [x, y, x + bw, y + bh],
            "bbox_wh": (x, y, bw, bh),
            "bbox_prod": int(bw * bh),
            "area": area,
            "cx": cx,
            "cy": cy,
            "centroid": [cx, cy],
        })
    regs.sort(key=lambda r: r["area"], reverse=True)
    return regs


def fallback_regions_from_colored_pixels_5(img_bgr, min_area=1):
    mask = remaining_colored_mask_5(img_bgr)
    if mask is None:
        return []
    return region_components_from_mask_5(mask, min_area=min_area)


def move_point_towards_5(tx, ty, x, y, max_step=10, w_lim=None, h_lim=None):
    dx = tx - x
    dy = ty - y
    dist = float(np.hypot(dx, dy))
    if dist <= max_step or dist == 0:
        nx = int(round(tx))
        ny = int(round(ty))
        if w_lim is not None:
            nx = int(np.clip(nx, 0, w_lim - 1))
        if h_lim is not None:
            ny = int(np.clip(ny, 0, h_lim - 1))
        return nx, ny, True
    s = max_step / dist
    nx = int(round(x + dx * s))
    ny = int(round(y + dy * s))
    if w_lim is not None:
        nx = int(np.clip(nx, 0, w_lim - 1))
    if h_lim is not None:
        ny = int(np.clip(ny, 0, h_lim - 1))
    return nx, ny, False


def build_bottom_route_5(mask, start_xy, step_px=18, y_offset=4):
    Hm, Wm = mask.shape[:2]
    ys, xs = np.where(mask)
    if xs.size == 0:
        return []
    x_left, x_right = int(xs.min()), int(xs.max())
    step_px = max(2, int(step_px))
    anchors_x = list(range(x_left, x_right + 1, step_px))
    if len(anchors_x) == 0 or anchors_x[-1] != x_right:
        anchors_x.append(x_right)
    route = []
    used = set()
    for xq in anchors_x:
        best = None
        best_key = None
        for dx in range(-12, 13):
            xc = int(np.clip(xq + dx, 0, Wm - 1))
            ys_col = np.where(mask[:, xc])[0]
            if ys_col.size == 0:
                continue
            y_bottom = int(ys_col.max())
            y_target = y_bottom
            if y_offset > 0:
                ys_valid = ys_col[ys_col <= y_bottom - int(y_offset)]
                if ys_valid.size > 0:
                    y_target = int(ys_valid.max())
            elif y_offset < 0:
                y_target = int(np.clip(y_bottom - int(y_offset), 0, Hm - 1))
            key = (abs(dx), -y_bottom)
            if best_key is None or key < best_key:
                best = (xc, y_target)
                best_key = key
        if best is not None and best not in used:
            route.append(best)
            used.add(best)
    compact = []
    for p in route:
        if not compact or np.hypot(p[0] - compact[-1][0], p[1] - compact[-1][1]) > 5:
            compact.append(p)
    if len(compact) <= 1:
        return compact
    p_first, p_last = compact[0], compact[-1]
    d_first = np.hypot(start_xy[0] - p_first[0], start_xy[1] - p_first[1])
    d_last = np.hypot(start_xy[0] - p_last[0], start_xy[1] - p_last[1])
    return compact if d_first <= d_last else list(reversed(compact))


def choose_nearest_offset_route_5(large_regions, start_xy, step_px=18, y_offset=4):
    candidates = []
    for reg in large_regions:
        route = build_bottom_route_5(reg["mask"], start_xy, step_px=step_px, y_offset=y_offset)
        if not route:
            continue
        d0 = float(np.hypot(start_xy[0] - route[0][0], start_xy[1] - route[0][1]))
        candidates.append((d0, reg, route))
    if not candidates:
        return None, []
    candidates.sort(key=lambda item: item[0])
    return candidates[0][1], candidates[0][2]


def params_trajetoria_5():
    """
    Parâmetros de trajetória.

    Prioridade:
    1. setup da célula 3 selecionado;
    2. widgets internos da célula 5 como fallback.

    Assim, não é necessário ajustar novamente na célula 5 aquilo que já foi
    ajustado e salvo na célula 3.
    """
    setup3 = setup_parametrico_ativo_celula3_5()

    return {
        "area_threshold_offset": int(setup3.get("area_threshold_offset", area_threshold_offset_widget_5.value)),
        "offset_y": int(setup3.get("offset_y", offset_y_widget_5.value)),
        "route_step_px": int(setup3.get("route_step_px", route_step_widget_5.value)),
        "centroid_hold_frames": int(setup3.get("centroid_hold_frames", centroid_hold_widget_5.value)),
        "aim_speed_px": int(setup3.get("aim_speed_px", aim_speed_widget_5.value)),
        "erase_radius": int(setup3.get("erase_radius", jet_erase_radius_widget.value)),
    }


def escolher_rota_jato_celula3_5(regs, start_xy):
    """
    Replica a estratégia da célula 3:
    - região grande: rota pela borda inferior com offset Y;
    - caso contrário: região mais próxima e permanência no centróide.
    """
    if not regs:
        return None, [], "sem foco"

    p = params_trajetoria_5()
    area_thr = int(p["area_threshold_offset"])
    large_regions = [r for r in regs if int(r.get("area", 0)) >= area_thr]
    chosen, route = choose_nearest_offset_route_5(
        large_regions,
        start_xy,
        step_px=int(p["route_step_px"]),
        y_offset=int(p["offset_y"]),
    )
    if route:
        return chosen, route, "borda"

    idx_near = int(np.argmin([np.hypot(start_xy[0] - r["cx"], start_xy[1] - r["cy"]) for r in regs]))
    reg = regs[idx_near]
    hold = max(1, int(p["centroid_hold_frames"]))
    c = (int(round(reg["cx"])), int(round(reg["cy"])))
    route = [c for _ in range(hold)]
    return reg, route, "centroides"


def expandir_bbox_5(bbox, shape, pad):
    h, w = shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    return [max(0, x1 - pad), max(0, y1 - pad), min(w, x2 + pad), min(h, y2 + pad)]



def profundidade_por_triangulacao_raw_5(pt1, pt2):
    """
    Calcula profundidade usando a calibração estéreo SEM precisar retificar a imagem exibida.

    Ideia:
    - pt1 e pt2 são pontos correspondentes nas imagens raw;
    - remove a distorção com K/D;
    - triangula usando P1=[I|0] e P2=[R|T] em coordenadas normalizadas.

    A unidade de Z segue a unidade usada em T na calibração do xadrez.
    """
    if not checkbox_use_stereo_calib_5.value:
        return None

    calib = ARDUINO_STATE.get("stereo_calib")

    if calib is None:
        carregar_calibracao_stereo_5()
        calib = ARDUINO_STATE.get("stereo_calib")

    if calib is None:
        return None

    try:
        # Escala K para a resolução atual.
        # Usa a resolução atual de captura informada pelos frames mais recentes.
        # A calibração original pode ter sido, por exemplo, 1920x1080.
        calib_w = int(calib_width_widget.value)
        calib_h = int(calib_height_widget.value)

        cur_w = int(image_width_widget.value)
        cur_h = int(image_height_widget.value)

        if cur_w <= 0 or cur_h <= 0:
            return None

        if calib_w <= 0 or calib_h <= 0:
            calib_w, calib_h = cur_w, cur_h

        K1s = escalar_matriz_camera_5(calib["K1"], calib_w, calib_h, cur_w, cur_h)
        K2s = escalar_matriz_camera_5(calib["K2"], calib_w, calib_h, cur_w, cur_h)

        D1 = calib["D1"]
        D2 = calib["D2"]
        R = calib["R"]
        T = calib["T"].reshape(3, 1)

        pts1 = np.array([[pt1]], dtype=np.float64)
        pts2 = np.array([[pt2]], dtype=np.float64)

        # Pontos normalizados sem distorção.
        p1n = cv2.undistortPoints(pts1, K1s, D1)
        p2n = cv2.undistortPoints(pts2, K2s, D2)

        # Projeções em coordenadas normalizadas.
        P1 = np.hstack([np.eye(3), np.zeros((3, 1))]).astype(np.float64)
        P2 = np.hstack([R, T]).astype(np.float64)

        X_h = cv2.triangulatePoints(P1, P2, p1n.reshape(2, 1), p2n.reshape(2, 1))

        if abs(float(X_h[3, 0])) < 1e-9:
            return None

        X = (X_h[:3, 0] / X_h[3, 0]).astype(float)

        # Z no sistema da câmera 1. Usa módulo para evitar sinal negativo por convenção.
        Z = float(abs(X[2]))

        if not np.isfinite(Z):
            return None

        return Z

    except Exception:
        return None


def buscar_correspondencia_bbox_raw_5(frame1, frame2, bbox1):
    pad = int(stereo_template_pad_widget.value)
    margin_y = int(stereo_search_margin_widget.value)
    b1 = expandir_bbox_5(bbox1, frame1.shape, pad)
    x1, y1, x2, y2 = b1
    if x2 <= x1 + 6 or y2 <= y1 + 6:
        return None
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    template = gray1[y1:y2, x1:x2]
    th, tw = template.shape[:2]
    if th < 8 or tw < 8:
        return None
    sy1 = max(0, y1 - margin_y)
    sy2 = min(gray2.shape[0], y2 + margin_y)
    if sy2 - sy1 < th + 2:
        return None
    search = gray2[sy1:sy2, :]
    if search.shape[0] < th or search.shape[1] < tw:
        return None
    try:
        res = cv2.matchTemplate(search, template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(res)
        mx, my = max_loc
        x1m = int(mx)
        y1m = int(sy1 + my)
        x2m = int(x1m + tw)
        y2m = int(y1m + th)
        c1x = (x1 + x2) / 2.0
        c1y = (y1 + y2) / 2.0
        c2x = (x1m + x2m) / 2.0
        disparity = float(c1x - c2x)
        depth_proxy = 1.0 / max(abs(disparity), 1.0)

        # Se o par estiver retificado, Q é válido.
        depth_q = profundidade_por_Q_5(c1x, c1y, disparity)

        # Mesmo sem retificar a imagem exibida, podemos calcular profundidade
        # por triangulação usando calibração + pontos raw correspondentes.
        depth_tri = None
        if depth_q is None:
            depth_tri = profundidade_por_triangulacao_raw_5((c1x, c1y), (c2x, (y1m + y2m) / 2.0))

        return {
            "bbox": [x1m, y1m, x2m, y2m],
            "score": float(max_val),
            "disparity_px": disparity,
            "depth_proxy": depth_proxy,
            "depth_q": depth_q,
            "depth_tri": depth_tri,
        }
    except Exception:
        return None


def associar_regioes_cam2_por_raw_matching_5(frame1, frame2, regs1):
    t0 = time.time()
    pair_id = int(ARDUINO_STATE.get("frame_pair_id", 0))
    every = max(1, int(stereo_match_every_widget.value))
    ARDUINO_STATE["depth_internal_every_frames"] = every
    if pair_id - int(ARDUINO_STATE.get("stereo_last_match_frame", -999)) < every:
        return ARDUINO_STATE.get("stereo_last_regs2", [])
    regs2 = []
    regs_for_match = sorted(regs1 or [], key=lambda r: int(r.get("area", 0)), reverse=True)
    regs_for_match = regs_for_match[:max(1, int(stereo_top_regions_widget.value))]
    for r in regs_for_match:
        match = buscar_correspondencia_bbox_raw_5(frame1, frame2, r.get("bbox", [0, 0, 0, 0]))
        if match is None:
            continue
        r["matched_bbox_cam2"] = match["bbox"]
        r["match_score"] = match["score"]
        r["disparity_px"] = match["disparity_px"]
        r["depth_proxy"] = match["depth_proxy"]
        r["depth_q"] = match["depth_q"]
        r["depth_tri"] = match.get("depth_tri", None)

        if match["depth_q"] is not None:
            r["depth_z"] = match["depth_q"]
            r["depth_is_proxy"] = False
            depth_source = "Q"
        elif match.get("depth_tri", None) is not None:
            r["depth_z"] = match["depth_tri"]
            r["depth_is_proxy"] = False
            depth_source = "triangulacao_raw"
        else:
            r["depth_z"] = match["depth_proxy"]
            r["depth_is_proxy"] = True
            depth_source = "proxy_disparidade"

        # Registro direto da amostra: não depende mais de a rota ser reconstruída.
        registrar_amostra_distancia_5(
            r["depth_z"],
            is_proxy=bool(r["depth_is_proxy"]),
            source=depth_source,
            score=match.get("score"),
            disparity=match.get("disparity_px"),
        )
        x1, y1, x2, y2 = match["bbox"]
        regs2.append({
            "bbox": [x1, y1, x2, y2],
            "centroid": [(x1 + x2) / 2.0, (y1 + y2) / 2.0],
            "area": int(max(1, (x2 - x1) * (y2 - y1))),
            "match_score": match["score"],
            "disparity_px": match["disparity_px"],
            "depth_proxy": match["depth_proxy"],
            "depth_q": match["depth_q"],
            "depth_tri": match.get("depth_tri", None),
            "depth_z": (
                match["depth_q"]
                if match["depth_q"] is not None
                else (
                    match.get("depth_tri", None)
                    if match.get("depth_tri", None) is not None
                    else match["depth_proxy"]
                )
            ),
            "depth_is_proxy": match["depth_q"] is None and match.get("depth_tri", None) is None,
        })
    ARDUINO_STATE["stereo_last_regs2"] = regs2
    ARDUINO_STATE["stereo_last_match_frame"] = pair_id
    ARDUINO_STATE["last_match_count"] = len(regs2)
    ARDUINO_STATE["last_match_ms"] = (time.time() - t0) * 1000.0
    return regs2



def calcular_mudanca_cenario_5(mask_ref, mask_atual):
    """
    Mede mudança entre a máscara de referência do cenário real e a máscara atual.
    """
    if mask_ref is None or mask_atual is None:
        return 0.0

    try:
        a = (mask_ref > 0)
        b = (mask_atual > 0)

        if a.shape != b.shape:
            return 1.0

        union = np.logical_or(a, b)
        union_count = int(np.count_nonzero(union))

        if union_count <= 0:
            return 0.0

        diff = np.logical_xor(a, b)
        return float(np.count_nonzero(diff)) / float(union_count)

    except Exception:
        return 0.0


def verificar_e_atualizar_canvas_por_mudanca_real_5(jet, filtered1):
    """
    Atualiza a quarta imagem quando a cena real muda de forma relevante.

    Compara o frame filtrado real atual com uma referência do cenário real,
    não com o canvas apagado. Isso evita repetir o mesmo foco já apagado,
    mas permite reconstruir o canvas quando a cena realmente muda.
    """
    if not isinstance(jet, dict) or filtered1 is None:
        return False

    cur_mask = remaining_colored_mask_5(filtered1)
    cur_pixels = int(np.count_nonzero(cur_mask)) if cur_mask is not None else 0

    if cur_mask is None:
        return False

    if jet.get("live_reference_mask") is None:
        jet["live_reference_mask"] = cur_mask.copy()
        jet["live_reference_pixels"] = cur_pixels
        ARDUINO_STATE["scene_change_score"] = 0.0
        ARDUINO_STATE["scene_change_count"] = 0
        return False

    min_pix = int(ARDUINO_STATE.get("scene_change_min_pixels", 120))
    threshold = float(ARDUINO_STATE.get("scene_change_threshold", 0.22))

    ref_mask = jet.get("live_reference_mask")
    ref_pixels = int(jet.get("live_reference_pixels", 0))

    score_shape = calcular_mudanca_cenario_5(ref_mask, cur_mask)
    score_area = abs(cur_pixels - ref_pixels) / max(1, max(cur_pixels, ref_pixels))
    score = max(score_shape, score_area)

    ARDUINO_STATE["scene_change_score"] = float(score)

    if cur_pixels < min_pix and ref_pixels < min_pix:
        ARDUINO_STATE["scene_change_count"] = 0
        return False

    if score >= threshold:
        ARDUINO_STATE["scene_change_count"] = int(ARDUINO_STATE.get("scene_change_count", 0)) + 1
    else:
        ARDUINO_STATE["scene_change_count"] = 0

    # Duas leituras consecutivas reduzem falso reset por ruído.
    if int(ARDUINO_STATE.get("scene_change_count", 0)) < 2:
        return False

    jet["canvas"] = filtered1.copy()
    jet["live_reference_mask"] = cur_mask.copy()
    jet["live_reference_pixels"] = cur_pixels
    jet["route"] = []
    jet["route_index"] = 0
    jet["pause_until_frame"] = -1
    jet["current_target_region"] = None
    jet["target_region"] = None
    jet["impact_pt"] = None

    ARDUINO_STATE["scene_change_count"] = 0
    ARDUINO_STATE["scene_change_last_reason"] = f"canvas atualizado por mudança real score={score:.2f}"

    return True


def jato_bloqueado_por_humano_5():
    return bool(ARDUINO_STATE.get("human_detected", False))


def atualizar_estado_jato_independente_5(regs1, frame_shape, filtered1=None):
    """
    Atualiza estado do jato com otimização de latência.

    Otimização principal:
    - não calcula componentes conectados do canvas em todo frame;
    - só recalcula regiões do canvas quando precisa criar uma rota nova;
    - checa mudança real de cena a cada N frames, não em todo frame.
    """
    if "JET_SIM_STATE" not in globals():
        globals()["JET_SIM_STATE"] = {}

    jet = globals()["JET_SIM_STATE"]

    if not isinstance(jet, dict):
        return

    h, w = frame_shape[:2]
    jet.setdefault("running", False)
    jet.setdefault("route", [])
    jet.setdefault("route_index", 0)
    jet.setdefault("mode", "borda")
    jet.setdefault("movement_phase", "aproximação/entre trajetórias")
    jet.setdefault("frame", 0)
    jet.setdefault("pause_until_frame", -1)
    jet.setdefault("x", 26)
    jet.setdefault("y", max(18, h - 24))
    jet.setdefault("current_target_region", None)

    pair_id = int(ARDUINO_STATE.get("frame_pair_id", 0))

    if jet.get("canvas") is None and filtered1 is not None:
        jet["canvas"] = filtered1.copy()
        ref_mask = remaining_colored_mask_5(filtered1)
        jet["live_reference_mask"] = ref_mask.copy() if ref_mask is not None else None
        jet["live_reference_pixels"] = int(np.count_nonzero(ref_mask)) if ref_mask is not None else 0

    # Checagem de mudança de cena é cara; faz a cada N frames.
    if filtered1 is not None:
        scene_every = max(1, int(scene_check_every_widget_5.value)) if "scene_check_every_widget_5" in globals() else 10
        if pair_id % scene_every == 0:
            verificar_e_atualizar_canvas_por_mudanca_real_5(jet, filtered1)

    # Segurança humana: congela também o cursor/planejador visual da imagem 4.
    if jato_bloqueado_por_humano_5():
        jet["running"] = False
        jet["mode"] = "pausado por humano"
        jet["movement_phase"] = "congelado por humano"
        return

    jet["frame"] = int(jet.get("frame", 0)) + 1

    # Se já existe rota em andamento, evita recomputar regiões do canvas.
    route = jet.get("route", []) or []
    idx_now = int(jet.get("route_index", 0))
    route_finished = idx_now >= len(route)
    no_route = len(route) == 0
    can_replan_now = no_route or (route_finished and jet["frame"] >= int(jet.get("pause_until_frame", -1)))

    if can_replan_now:
        regs_for_route = []

        if remaining_colored_pixels_5(jet.get("canvas")) > 0:
            regs_for_route = fallback_regions_from_colored_pixels_5(jet.get("canvas"), min_area=1)

        if not regs_for_route and regs1:
            regs_for_route = regs1

        if not regs_for_route:
            jet["route"] = []
            jet["route_index"] = 0
            jet["aim_pt"] = None
            jet["impact_pt"] = None
            jet["target_region"] = None
            jet["current_target_region"] = None
            ARDUINO_STATE["current_target_depth"] = ARDUINO_STATE.get("depth_display_value", None)
            ARDUINO_STATE["current_target_depth_text"] = ARDUINO_STATE.get("depth_display_text", "Dist: --")
            jet["mode"] = "sem foco"

            if filtered1 is not None and not jet.get("running", False):
                jet["canvas"] = filtered1.copy()
                ref_mask = remaining_colored_mask_5(filtered1)
                jet["live_reference_mask"] = ref_mask.copy() if ref_mask is not None else None
                jet["live_reference_pixels"] = int(np.count_nonzero(ref_mask)) if ref_mask is not None else 0

            return

        start_xy = (int(jet.get("x", 26)), int(jet.get("y", max(18, h - 24))))
        chosen, route, mode = escolher_rota_jato_celula3_5(regs_for_route, start_xy)

        jet["route"] = route
        jet["route_index"] = 0
        jet["mode"] = mode
        chosen = enriquecer_alvo_com_profundidade_5(chosen, regs1)
        jet["current_target_region"] = chosen
        jet["target_region"] = chosen
        jet["impact_pt"] = route[-1] if route else None

    if not jet.get("route"):
        return

    # Congela o cursor quando o jato não está ativo.
    if not jet.get("running", False):
        jet["movement_phase"] = "pausado/congelado"
        return

    if jet["frame"] < int(jet.get("pause_until_frame", -1)):
        return

    idx = int(jet.get("route_index", 0))

    if idx >= len(jet["route"]):
        return

    p = params_trajetoria_5()
    other_speed = max(1, int(p["aim_speed_px"]))
    border_speed = max(1, int(round(other_speed * 0.70)))

    if jet.get("mode") == "borda" and idx > 0:
        eff_step = border_speed
        jet["movement_phase"] = "offset"
    else:
        eff_step = other_speed
        jet["movement_phase"] = "aproximação/entre trajetórias"

    tx, ty = jet["route"][idx]
    nx, ny, arrived = move_point_towards_5(
        tx,
        ty,
        int(jet.get("x", 26)),
        int(jet.get("y", max(18, h - 24))),
        max_step=eff_step,
        w_lim=w,
        h_lim=h
    )

    jet["x"], jet["y"] = nx, ny
    jet["aim_pt"] = (nx, ny)

    if jet.get("canvas") is not None:
        cv2.circle(jet["canvas"], (nx, ny), int(p["erase_radius"]), (0, 0, 0), -1)

    if arrived:
        jet["route_index"] = idx + 1
        dwell = max(0, int(p["centroid_hold_frames"]))

        if dwell > 0:
            jet["pause_until_frame"] = max(int(jet.get("pause_until_frame", -1)), jet["frame"] + dwell)

        if jet["route_index"] >= len(jet["route"]):
            jet["pause_until_frame"] = max(int(jet.get("pause_until_frame", -1)), jet["frame"] + max(1, dwell))


def processar_frame_stereo_independente_5(frame1, frame2, params=None):
    frame1r, frame2r, rectified = retificar_par_stereo_5(frame1, frame2)
    filtered1, mask1, regs1 = extrair_regioes_independente_5(frame1r, params)
    regs2 = associar_regioes_cam2_por_raw_matching_5(frame1r, frame2r, regs1)
    filtered2 = frame2r.copy()
    atualizar_estado_jato_independente_5(regs1, frame1r.shape, filtered1=filtered1)
    return frame1r, frame2r, filtered1, filtered2, regs1, regs2




def registrar_amostra_distancia_5(valor, is_proxy=False, source="alvo", score=None, disparity=None):
    """
    Registra uma amostra válida de distância na janela atual.

    A janela acumula valores por 90 frames e publica a média.
    Agora as amostras podem vir diretamente do matching, não apenas do alvo/rota.
    """
    if valor is None:
        return False

    try:
        v = float(valor)
    except Exception:
        return False

    if not np.isfinite(v):
        return False

    ARDUINO_STATE.setdefault("depth_window_values", [])
    ARDUINO_STATE.setdefault("depth_window_proxy_flags", [])

    ARDUINO_STATE["depth_window_values"].append(v)
    ARDUINO_STATE["depth_window_proxy_flags"].append(bool(is_proxy))

    ARDUINO_STATE["last_valid_depth"] = v
    ARDUINO_STATE["last_valid_depth_is_proxy"] = bool(is_proxy)
    ARDUINO_STATE["last_valid_depth_frame"] = int(ARDUINO_STATE.get("frame_pair_id", 0))

    ARDUINO_STATE["depth_sample_count_window"] = len(ARDUINO_STATE.get("depth_window_values", []))
    ARDUINO_STATE["depth_total_samples"] = int(ARDUINO_STATE.get("depth_total_samples", 0)) + 1
    ARDUINO_STATE["depth_last_sample_text"] = formatar_distancia_alvo_5(v, is_proxy=bool(is_proxy))
    ARDUINO_STATE["depth_last_match_score"] = score
    ARDUINO_STATE["depth_last_disparity"] = disparity
    ARDUINO_STATE["depth_last_source"] = str(source)

    return True


def publicar_distancia_se_necessario_5(force=False):
    """
    Publica a média da janela de distância a cada 90 frames.

    Se a janela não tiver nenhum valor válido, mantém a última distância publicada.
    """
    frame_id = int(ARDUINO_STATE.get("frame_pair_id", 0))
    publish_every = int(ARDUINO_STATE.get("depth_publish_every_frames", 90))
    window_start = int(ARDUINO_STATE.get("depth_window_start_frame", 0))

    if window_start <= 0:
        ARDUINO_STATE["depth_window_start_frame"] = frame_id
        window_start = frame_id

    # Se ainda nunca houve publicação e já existe pelo menos uma amostra,
    # publica imediatamente a primeira média para evitar ficar preso em Dist: --.
    if (
        not ARDUINO_STATE.get("depth_first_publish_done", False)
        and len(ARDUINO_STATE.get("depth_window_values", []) or []) > 0
    ):
        force = True

    if not force and (frame_id - window_start) < publish_every:
        return ARDUINO_STATE.get("depth_display_text", "Dist: --")

    vals = ARDUINO_STATE.get("depth_window_values", []) or []
    flags = ARDUINO_STATE.get("depth_window_proxy_flags", []) or []

    if vals:
        media = float(np.mean(vals))
        is_proxy = bool(flags) and all(bool(x) for x in flags)

        ARDUINO_STATE["depth_display_value"] = media
        ARDUINO_STATE["depth_display_is_proxy"] = is_proxy
        ARDUINO_STATE["depth_display_frame"] = frame_id
        ARDUINO_STATE["depth_display_text"] = formatar_distancia_alvo_5(media, is_proxy=is_proxy)
        ARDUINO_STATE["depth_first_publish_done"] = True
    else:
        # Sem valor válido na janela: mantém o último valor publicado.
        if ARDUINO_STATE.get("depth_display_value", None) is None:
            ARDUINO_STATE["depth_display_text"] = "Dist: --"
        else:
            ARDUINO_STATE["depth_display_text"] = formatar_distancia_alvo_5(
                ARDUINO_STATE.get("depth_display_value"),
                is_proxy=bool(ARDUINO_STATE.get("depth_display_is_proxy", True))
            )

    ARDUINO_STATE["depth_window_values"] = []
    ARDUINO_STATE["depth_window_proxy_flags"] = []
    ARDUINO_STATE["depth_sample_count_window"] = 0
    ARDUINO_STATE["depth_window_start_frame"] = frame_id

    return ARDUINO_STATE.get("depth_display_text", "Dist: --")


def distancia_publicada_atual_5():
    """
    Valor que deve aparecer na imagem/status.
    Não pisca para -- entre janelas.
    """
    txt = publicar_distancia_se_necessario_5(force=False)
    return txt if txt else "Dist: --"


def formatar_distancia_alvo_5(valor, is_proxy=False):
    """
    Formata a distância/profundidade do alvo usando apenas ASCII.
    Isso evita que símbolos especiais apareçam como pontos de interrogação.
    """
    if valor is None:
        return "Dist: --"

    try:
        v = float(valor)
    except Exception:
        return "Dist: --"

    if not np.isfinite(v):
        return "Dist: --"

    try:
        fator = float(distance_correction_widget.value) if "distance_correction_widget" in globals() else 1.0
    except Exception:
        fator = 1.0

    v_corrigido = v * fator
    prefix = "Dist proxy: " if is_proxy else "Dist: "

    if abs(v_corrigido) >= 100:
        return f"{prefix}{v_corrigido:.1f}"
    if abs(v_corrigido) >= 10:
        return f"{prefix}{v_corrigido:.2f}"
    return f"{prefix}{v_corrigido:.3f}"


def enriquecer_alvo_com_profundidade_5(alvo, regs1):
    """
    Garante que a região em combate tenha profundidade quando algum match
    estéreo já calculou depth_q/depth_z para a mesma região ou uma região próxima.
    """
    if not isinstance(alvo, dict):
        ARDUINO_STATE["current_target_depth"] = None
        ARDUINO_STATE["current_target_depth_text"] = "Dist: --"
        return alvo

    profundidade = alvo.get("depth_q", None)
    profundidade_proxy = False

    if profundidade is None:
        profundidade = alvo.get("depth_tri", None)

    if profundidade is None:
        profundidade = alvo.get("depth_z", None)
        profundidade_proxy = alvo.get("depth_q", None) is None and alvo.get("depth_tri", None) is None

    # Se o alvo atual veio do canvas/fallback e não tem profundidade,
    # procura uma região detectada no frame atual com centróide próximo.
    if profundidade is None and regs1:
        cx, cy = alvo.get("centroid", [alvo.get("cx", None), alvo.get("cy", None)])

        if cx is None and "cx" in alvo:
            cx = alvo.get("cx")
        if cy is None and "cy" in alvo:
            cy = alvo.get("cy")

        if cx is not None and cy is not None:
            melhor = None
            melhor_d = None

            for r in regs1:
                rcx, rcy = r.get("centroid", [r.get("cx", None), r.get("cy", None)])
                if rcx is None or rcy is None:
                    continue

                d = float(np.hypot(float(cx) - float(rcx), float(cy) - float(rcy)))
                if melhor is None or d < melhor_d:
                    melhor = r
                    melhor_d = d

            if melhor is not None:
                profundidade = melhor.get("depth_q", None)
                profundidade_proxy = False

                if profundidade is None:
                    profundidade = melhor.get("depth_tri", None)

                if profundidade is None:
                    profundidade = melhor.get("depth_z", None)
                    profundidade_proxy = True

                if profundidade is not None:
                    if not profundidade_proxy:
                        if melhor.get("depth_q", None) is not None:
                            alvo["depth_q"] = profundidade
                        else:
                            alvo["depth_tri"] = profundidade
                    alvo["depth_z"] = profundidade
                    alvo["depth_is_proxy"] = bool(profundidade_proxy)
                    alvo["match_score"] = melhor.get("match_score", alvo.get("match_score", None))
                    alvo["disparity_px"] = melhor.get("disparity_px", alvo.get("disparity_px", None))

    alvo["depth_is_proxy"] = bool(alvo.get("depth_is_proxy", profundidade_proxy))

    is_proxy_now = bool(alvo.get("depth_is_proxy", False))

    if profundidade is not None:
        registrar_amostra_distancia_5(profundidade, is_proxy=is_proxy_now, source="alvo_rota")

    # O valor exibido é atualizado apenas a cada 90 frames, usando a média
    # das amostras válidas coletadas a cada 15 frames.
    ARDUINO_STATE["current_target_depth"] = ARDUINO_STATE.get("depth_display_value", None)
    ARDUINO_STATE["current_target_depth_text"] = distancia_publicada_atual_5()

    return alvo


def desenhar_distancia_sobre_alvo_5(out, alvo):
    """
    Mostra a distância calculada sobre o centróide da região quente sendo combatida.
    """
    if not isinstance(alvo, dict):
        return out

    txt = distancia_publicada_atual_5()

    cx, cy = alvo.get("centroid", [alvo.get("cx", None), alvo.get("cy", None)])

    if cx is None and "cx" in alvo:
        cx = alvo.get("cx")
    if cy is None and "cy" in alvo:
        cy = alvo.get("cy")

    if cx is None or cy is None:
        return out

    x = int(round(cx))
    y = int(round(cy))

    # Caixa de texto com fundo escuro para legibilidade.
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.58
    thick = 2
    (tw, th), base = cv2.getTextSize(txt, font, scale, thick)

    x0 = int(np.clip(x - tw // 2 - 5, 0, max(0, out.shape[1] - tw - 10)))
    y0 = int(np.clip(y - 30, th + 8, out.shape[0] - 8))

    cv2.rectangle(out, (x0, y0 - th - 6), (x0 + tw + 10, y0 + base + 4), (0, 0, 0), -1)
    cv2.rectangle(out, (x0, y0 - th - 6), (x0 + tw + 10, y0 + base + 4), (0, 255, 255), 1)
    cv2.putText(out, txt, (x0 + 5, y0), font, scale, (0, 255, 255), thick, cv2.LINE_AA)
    cv2.drawMarker(out, (x, y), (0, 255, 255), cv2.MARKER_CROSS, 18, 2)

    return out


def desenhar_regioes_independente_5(img, regs, titulo=""):
    out = img.copy()
    if titulo:
        cv2.putText(out, titulo, (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    alvo_atual = globals().get("JET_SIM_STATE", {}).get("current_target_region", None)

    for r in regs or []:
        contour = r.get("contour")
        x1, y1, x2, y2 = [int(v) for v in r.get("bbox", [0, 0, 0, 0])]
        cx, cy = r.get("centroid", [(x1 + x2) / 2, (y1 + y2) / 2])
        if checkbox_draw_hot_contours_5.value and contour is not None:
            cv2.drawContours(out, [contour], -1, (0, 255, 255), 2)
        if checkbox_draw_hot_boxes_5.value:
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 255), 1)
        cv2.circle(out, (int(cx), int(cy)), 4, (0, 255, 0), -1)
        if checkbox_show_region_labels_5.value:
            label = f"A={int(r.get('area', 0))}"
            if r.get("match_score") is not None:
                label += f" M={float(r.get('match_score', 0)):.2f}"
            if r.get("disparity_px") is not None:
                label += f" d={float(r.get('disparity_px', 0)):.1f}px"
            cv2.putText(out, label, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (0, 255, 255), 1)

    # A distância não é desenhada aqui.
    # Ela aparece somente na última imagem: "Filtro 1 + Jato".
    return out


def desenhar_match_cam2_5(img, regs2, titulo="Camera 2"):
    out = img.copy()
    cv2.putText(out, titulo, (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    for r in regs2 or []:
        cx, cy = r.get("centroid", [None, None])
        if cx is None or cy is None:
            continue
        cv2.drawMarker(out, (int(cx), int(cy)), (0, 255, 255), cv2.MARKER_CROSS, 18, 2)
        if checkbox_show_region_labels_5.value:
            txt = f"M={float(r.get('match_score', 0)):.2f}"
            if r.get("depth_q") is not None:
                txt += f" Z={float(r.get('depth_q')):.3f}"
            cv2.putText(out, txt, (int(cx) + 8, int(cy)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
    return out


def desenhar_jato_independente_5(base_img):
    jet = globals().get("JET_SIM_STATE", {})
    out = jet["canvas"].copy() if isinstance(jet, dict) and jet.get("canvas") is not None else base_img.copy()
    route = jet.get("route", []) if isinstance(jet, dict) else []
    idx = int(jet.get("route_index", 0)) if isinstance(jet, dict) else 0
    if len(route) >= 2:
        pts = np.array(route, dtype=np.int32).reshape((-1, 1, 2))
        cv2.polylines(out, [pts], False, (68, 204, 102), 2)
        for k, p in enumerate(route):
            c = (0, 170, 51) if k >= idx else (80, 80, 80)
            cv2.circle(out, tuple(p), 2, c, -1)
    elif len(route) == 1:
        cv2.circle(out, tuple(route[0]), 3, (0, 170, 51), -1)
    x = int(jet.get("x", 26)) if isinstance(jet, dict) else 26
    y = int(jet.get("y", 26)) if isinstance(jet, dict) else 26
    cv2.circle(out, (x, y), max(4, int(jet_erase_radius_widget.value)), (120, 255, 120), 1, lineType=cv2.LINE_AA)
    cv2.rectangle(out, (x - 6, y - 6), (x + 6, y + 6), (0, 255, 0), 2, lineType=cv2.LINE_AA)
    desenhar_distancia_sobre_alvo_5(out, jet.get("current_target_region", None) if isinstance(jet, dict) else None)
    cv2.putText(out, "Filtro 1 + Jato", (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (255, 255, 255), 2)
    return out


def montar_painel_camera_independente_5(frame1, frame2, filtered1, filtered2, regs1, regs2, status="", params=None):
    """
    Painel limpo:
    Camera 1 | Filtro 1 | Filtro 1 + Jato

    A câmera 2 continua sendo usada internamente para matching/profundidade,
    mas não é mais exibida no painel principal.
    """
    f1 = desenhar_regioes_independente_5(frame1, regs1, "Camera 1")
    g1 = desenhar_regioes_independente_5(filtered1, regs1, "Filtro 1")
    g2 = desenhar_jato_independente_5(filtered1)

    h = min(f1.shape[0], g1.shape[0], g2.shape[0])
    w = min(f1.shape[1], g1.shape[1], g2.shape[1])

    imgs = [cv2.resize(im, (w, h)) for im in [f1, g1, g2]]
    panel = np.hstack(imgs)

    try:
        scale = float(panel_scale_widget_5.value) if "panel_scale_widget_5" in globals() else 1.0
    except Exception:
        scale = 1.0

    if scale < 0.999:
        new_w = max(1, int(panel.shape[1] * scale))
        new_h = max(1, int(panel.shape[0] * scale))
        panel = cv2.resize(panel, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return panel


# A célula 5 agora usa o pipeline independente para evitar conflitos com versões antigas da célula 4.
processar_frame_stereo = processar_frame_stereo_independente_5
montar_painel_camera = montar_painel_camera_independente_5


# ------------------------------------------------------------
# Câmera de referência para calibração
# ------------------------------------------------------------
async def reference_camera_loop_5():
    ARDUINO_STATE["ref_camera_running"] = True

    try:
        while ARDUINO_STATE["ref_camera_running"]:
            cap = ARDUINO_STATE.get("ref_cap")
            if cap is None:
                await asyncio.sleep(0.1)
                continue

            ok, frame = cap.read()
            if not ok or frame is None:
                registrar_acao("<b>Erro:</b> falha ao ler a câmera de referência.", "erro", widget=msg_camera if "msg_camera" in globals() else None)
                await asyncio.sleep(0.2)
                continue

            h, w = frame.shape[:2]
            cv2.circle(frame, (0, 0), 14, (0, 255, 255), 2)
            cv2.putText(frame, "sup. esq.", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            cv2.circle(frame, (w - 1, 0), 14, (0, 255, 255), 2)
            cv2.putText(frame, "sup. dir.", (max(10, w - 145), 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            cv2.circle(frame, (0, h - 1), 14, (0, 255, 255), 2)
            cv2.putText(frame, "inf. esq.", (10, max(25, h - 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

            active = CALIBRATION_SERVO_STATE.get("active_point")
            if active is not None:
                cv2.putText(frame, f"Calibrando: {ponto_ativo_label()}", (20, max(45, h - 45)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            reference_camera_widget.value = encode_jpg_bgr(frame, quality=85)
            await asyncio.sleep(float(loop_interval_widget_5.value))

    finally:
        cap = ARDUINO_STATE.get("ref_cap")
        if cap is not None:
            try:
                cap.release()
            except Exception:
                pass
        ARDUINO_STATE["ref_cap"] = None
        ARDUINO_STATE["ref_camera_task"] = None
        ARDUINO_STATE["ref_camera_running"] = False
        button_open_reference_camera.disabled = False
        button_stop_reference_camera.disabled = True


def abrir_camera_referencia_5(_=None):
    if ARDUINO_STATE.get("camera_running", False):
        registrar_acao(
            "<b>Não é possível abrir a câmera de referência enquanto a simulação está ativa.</b><br>"
            "Pare as câmeras da simulação primeiro.",
            "aviso",
            widget=msg_camera if "msg_camera" in globals() else None
        )
        return False

    if ARDUINO_STATE.get("ref_camera_running"):
        registrar_acao("<b>Câmera de referência já está aberta.</b>", "aviso", widget=msg_camera if "msg_camera" in globals() else None)
        return False

    cam1 = int(dropdown_cam1_5.value)
    if cam1 < 0:
        registrar_acao("<b>Selecione uma câmera de referência válida.</b>", "erro", widget=msg_camera if "msg_camera" in globals() else None)
        return False

    try:
        ARDUINO_STATE["ref_cap"] = abrir_camera_celula5(cam1, int(image_width_widget.value), int(image_height_widget.value), int(fps_widget_5.value))
        button_open_reference_camera.disabled = True
        button_stop_reference_camera.disabled = False
        ARDUINO_STATE["ref_camera_task"] = asyncio.create_task(reference_camera_loop_5())

        registrar_acao(f"<b>Câmera de referência aberta:</b> {cam1}.<br>Use esta imagem para ajustar os 3 pontos dos servos.", "ok", widget=msg_camera if "msg_camera" in globals() else None)
        return True
    except Exception as e:
        registrar_acao(f"<b>Erro ao abrir câmera:</b> {type(e).__name__}: {e}", "erro", widget=msg_camera if "msg_camera" in globals() else None)
        return False


def parar_camera_referencia_5(_=None):
    """
    Para e libera a câmera de referência imediatamente.
    """
    ARDUINO_STATE["ref_camera_running"] = False

    task = ARDUINO_STATE.get("ref_camera_task")
    if task is not None:
        try:
            task.cancel()
        except Exception:
            pass
        ARDUINO_STATE["ref_camera_task"] = None

    cap = ARDUINO_STATE.get("ref_cap")
    if cap is not None:
        try:
            cap.release()
        except Exception:
            pass

    ARDUINO_STATE["ref_cap"] = None
    button_open_reference_camera.disabled = False
    button_stop_reference_camera.disabled = True

    try:
        reference_camera_widget.value = b""
    except Exception:
        pass

    registrar_acao(
        "<b>Câmera de referência parada e liberada.</b>",
        "ok",
        widget=msg_camera if "msg_camera" in globals() else None
    )
    return True



def refresh_cameras_5(_=None):
    cams = listar_cameras_disponiveis_celula5()
    dropdown_cam1_5.options = cams
    dropdown_cam2_5.options = cams
    dropdown_cam1_5.value = cams[0][1]
    dropdown_cam2_5.value = cams[min(1, len(cams) - 1)][1]
    registrar_acao(f"<b>Câmeras detectadas:</b> {len(cams) if cams[0][1] != -1 else 0}", "ok")


def abrir_camera_celula5(cam_index, width=None, height=None, fps=None):
    """
    Abre câmera para a célula 5 sem depender da função abrir_camera() da célula 4.

    Motivo:
    algumas versões da célula 4 possuem abrir_camera(idx) com apenas 1 argumento.
    A célula 5 precisa abrir_camera_celula5(idx, largura, altura, fps).
    """
    cam_index = int(cam_index)

    if cam_index < 0:
        raise ValueError("Índice de câmera inválido.")

    cap = cv2.VideoCapture(cam_index, cv2.CAP_V4L2)

    if not cap.isOpened():
        try:
            cap.release()
        except Exception:
            pass

        cap = cv2.VideoCapture(cam_index)

    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir a câmera {cam_index}.")

    # Reduz atraso de buffer. Nem todo backend respeita, mas quando respeita ajuda muito.
    try:
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    except Exception:
        pass

    # Força MJPG quando disponível para reduzir carga USB/CPU.
    try:
        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
    except Exception:
        pass

    if width is not None:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(width))

    if height is not None:
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(height))

    if fps is not None:
        cap.set(cv2.CAP_PROP_FPS, int(fps))

    # Lê alguns frames para aquecer a câmera.
    ok = False
    frame = None

    for _ in range(5):
        ok, frame = cap.read()

        if ok and frame is not None:
            break

        time.sleep(0.05)

    if not ok or frame is None:
        try:
            cap.release()
        except Exception:
            pass

        raise RuntimeError(f"Câmera {cam_index} abriu, mas não retornou frame válido.")

    return cap





# ------------------------------------------------------------
# Segurança por detecção de humanos com YOLO
# ------------------------------------------------------------
def carregar_yolo_humano_5(_=None):
    """
    Carrega um modelo YOLO para detectar humanos.
    Esperado: ultralytics + modelo COCO, por exemplo yolov8n.pt.
    Classe COCO usada: person = 0.
    """
    try:
        from ultralytics import YOLO

        model_path = human_yolo_model_text.value.strip() or "yolov8n.pt"
        ARDUINO_STATE["human_yolo_model"] = YOLO(model_path)
        ARDUINO_STATE["human_yolo_loaded"] = True
        ARDUINO_STATE["human_status"] = f"YOLO humano carregado: {model_path}"

        registrar_acao(
            f"<b>YOLO humano carregado:</b><br><code>{model_path}</code><br>"
            "Classe usada: <code>person</code>.",
            "ok",
            widget=msg_human_safety if "msg_human_safety" in globals() else None
        )
        return True

    except Exception as e:
        ARDUINO_STATE["human_yolo_model"] = None
        ARDUINO_STATE["human_yolo_loaded"] = False
        ARDUINO_STATE["human_status"] = f"falha ao carregar YOLO humano: {type(e).__name__}: {e}"

        registrar_acao(
            "<b>Não foi possível carregar o YOLO humano.</b><br>"
            "Verifique se <code>ultralytics</code> está instalado e se o modelo existe localmente.<br>"
            f"<code>{type(e).__name__}: {e}</code>",
            "erro",
            widget=msg_human_safety if "msg_human_safety" in globals() else None
        )
        return False


def detectar_humanos_yolo_5(frame_bgr):
    if not checkbox_human_safety_yolo.value:
        return []
    if ARDUINO_STATE.get("human_yolo_model") is None:
        return []
    if frame_bgr is None:
        return []

    try:
        model = ARDUINO_STATE["human_yolo_model"]
        conf_min = float(human_conf_widget.value)

        h0, w0 = frame_bgr.shape[:2]
        target_w = int(human_yolo_width_widget.value) if "human_yolo_width_widget" in globals() else w0
        target_w = max(224, min(target_w, w0))

        if target_w < w0:
            scale = target_w / float(w0)
            target_h = max(1, int(round(h0 * scale)))
            frame_in = cv2.resize(frame_bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
        else:
            scale = 1.0
            frame_in = frame_bgr

        results = model.predict(frame_in, conf=conf_min, classes=[0], verbose=False)
        boxes_out = []

        if not results:
            return boxes_out

        r = results[0]
        if r.boxes is None:
            return boxes_out

        inv_scale = 1.0 / max(scale, 1e-6)

        for b in r.boxes:
            cls = int(b.cls[0].item()) if b.cls is not None else -1
            conf = float(b.conf[0].item()) if b.conf is not None else 0.0

            if cls != 0 or conf < conf_min:
                continue

            xyxy = b.xyxy[0].detach().cpu().numpy().tolist()
            x1, y1, x2, y2 = [int(round(v * inv_scale)) for v in xyxy]
            x1 = int(np.clip(x1, 0, w0 - 1))
            x2 = int(np.clip(x2, 0, w0 - 1))
            y1 = int(np.clip(y1, 0, h0 - 1))
            y2 = int(np.clip(y2, 0, h0 - 1))
            boxes_out.append({"bbox": [x1, y1, x2, y2], "conf": conf})

        return boxes_out

    except Exception as e:
        ARDUINO_STATE["human_status"] = f"erro YOLO humano: {type(e).__name__}: {e}"
        return []


def desenhar_humanos_5(frame_bgr, boxes):
    if frame_bgr is None:
        return frame_bgr

    out = frame_bgr.copy()

    for item in boxes or []:
        x1, y1, x2, y2 = item["bbox"]
        conf = float(item.get("conf", 0.0))
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 3)
        cv2.putText(out, f"HUMANO {conf:.2f}", (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)

    if boxes:
        cv2.putText(out, "JATO INTERROMPIDO - HUMANO DETECTADO", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 0, 255), 3)

    return out


def atualizar_seguranca_humana_5(frame_bgr):
    """
    Pausa o jato quando há humano e retoma quando o humano sai da cena.

    Se a trava estiver ativada, tenta carregar o YOLO automaticamente uma única vez.
    Se falhar, o status mostra o erro e o botão manual continua disponível.
    """
    t_yolo0 = time.time()

    if not checkbox_human_safety_yolo.value:
        ARDUINO_STATE["human_detected"] = False
        ARDUINO_STATE["human_boxes"] = []
        ARDUINO_STATE["human_status"] = "segurança humana desativada"
        ARDUINO_STATE["last_yolo_ms"] = 0.0
        return []

    if not ARDUINO_STATE.get("human_yolo_loaded", False):
        if not ARDUINO_STATE.get("human_yolo_load_attempted", False):
            ARDUINO_STATE["human_yolo_load_attempted"] = True
            carregar_yolo_humano_5()

        if not ARDUINO_STATE.get("human_yolo_loaded", False):
            ARDUINO_STATE["human_detected"] = False
            ARDUINO_STATE["human_boxes"] = []
            ARDUINO_STATE["human_status"] = (
                "YOLO humano não carregado; use Carregar YOLO humano "
                "ou verifique ultralytics/modelo local."
            )
            ARDUINO_STATE["last_yolo_ms"] = (time.time() - t_yolo0) * 1000.0
            return []

    loop_id = int(ARDUINO_STATE.get("human_loop_id", 0)) + 1
    ARDUINO_STATE["human_loop_id"] = loop_id
    every = max(1, int(human_check_every_widget.value))

    if loop_id % every != 0:
        boxes = ARDUINO_STATE.get("human_boxes", [])
    else:
        boxes = detectar_humanos_yolo_5(frame_bgr)
        ARDUINO_STATE["human_boxes"] = boxes

    ARDUINO_STATE["last_yolo_ms"] = (time.time() - t_yolo0) * 1000.0

    human_now = len(boxes) > 0
    human_before = bool(ARDUINO_STATE.get("human_detected", False))
    jet = globals().get("JET_SIM_STATE", None)

    if human_now and not human_before:
        was_running = bool(isinstance(jet, dict) and jet.get("running", False))
        ARDUINO_STATE["human_resume_after_clear"] = was_running

        if isinstance(jet, dict):
            jet["running"] = False
            # Mantém rota/aim_pt como estavam para a imagem 4 ficar congelada,
            # sem andar nem redesenhar rota durante a presença humana.
            jet["mode"] = "pausado por humano"
            jet["movement_phase"] = "congelado por humano"

        # Bloqueia imediatamente novos comandos de mira.
        ARDUINO_STATE["last_target"] = None
        ARDUINO_STATE["human_status"] = "HUMANO DETECTADO - jato interrompido imediatamente"
        registrar_acao(
            "<b>HUMANO DETECTADO.</b><br>"
            "Jato interrompido imediatamente. Ao remover o humano da cena, a operação será retomada.",
            "erro",
            widget=msg_human_safety if "msg_human_safety" in globals() else None
        )

    elif (not human_now) and human_before:
        if isinstance(jet, dict) and ARDUINO_STATE.get("human_resume_after_clear", False):
            jet["running"] = True
            jet["route"] = []
            jet["route_index"] = 0
            jet["pause_until_frame"] = -1
            jet["refresh_boxes_requested"] = True
            jet["mode"] = "retomado após humano sair"

        ARDUINO_STATE["human_resume_after_clear"] = False
        ARDUINO_STATE["human_status"] = "humano removido - operação retomada"
        registrar_acao(
            "<b>Humano removido da cena.</b><br>Operação retomada.",
            "ok",
            widget=msg_human_safety if "msg_human_safety" in globals() else None
        )
    else:
        ARDUINO_STATE["human_status"] = "HUMANO DETECTADO - jato pausado" if human_now else "sem humano detectado"

    ARDUINO_STATE["human_detected"] = human_now
    return boxes


def status_planejador_5(regs1=None):
    jet = globals().get("JET_SIM_STATE", {})
    if not isinstance(jet, dict):
        return "Planejador: indisponível"

    route_len = len(jet.get("route", []) or [])
    route_index = int(jet.get("route_index", 0) or 0)
    regs_len = len(regs1 or [])
    humano_txt = "sim" if ARDUINO_STATE.get("human_detected", False) else "não"

    return (
        f"Planejador | Regiões: {regs_len} | "
        f"Rota: {route_index}/{route_len} | "
        f"Modo: {jet.get('mode', 'n/d')} | "
        f"Humano: {humano_txt} | "
        f"Sem foco: {ARDUINO_STATE.get('no_focus_count', 0)} | "
        f"Novo foco: {ARDUINO_STATE.get('new_focus_count', 0)} | "
        f"Último motivo: {ARDUINO_STATE.get('planner_last_reason', 'n/d')}"
    )


# ------------------------------------------------------------
# Tratamento robusto de ausência de foco e recuperação de câmera
# ------------------------------------------------------------
def painel_fallback_cameras_5(frame1, frame2, status_txt=""):
    """
    Painel simples para manter a visualização atualizando mesmo quando
    o processamento térmico/planejamento falhar ou quando não houver foco.
    """
    try:
        f1 = frame1.copy() if frame1 is not None else np.zeros((240, 320, 3), dtype=np.uint8)
        f2 = frame2.copy() if frame2 is not None else np.zeros_like(f1)

        h = min(f1.shape[0], f2.shape[0])
        w = min(f1.shape[1], f2.shape[1])

        f1 = cv2.resize(f1, (w, h))
        f2 = cv2.resize(f2, (w, h))

        cv2.putText(f1, "Camera 1 - referencia", (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        cv2.putText(f2, "Camera 2", (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        panel = np.hstack([f1, f2])

        if status_txt:
            linhas = str(status_txt).split("\n")[:6]
            y0 = max(35, panel.shape[0] - 22 * len(linhas) - 12)
            for i, linha in enumerate(linhas):
                cv2.putText(panel, linha[:120], (18, y0 + 22 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

        return panel

    except Exception:
        return np.zeros((480, 960, 3), dtype=np.uint8)


def liberar_estado_sem_foco_5():
    """
    Quando nenhum foco é detectado, libera rota/alvo antigos sem travar a câmera.
    Isso permite que novos focos futuros gerem novas trajetórias.
    """
    jet = globals().get("JET_SIM_STATE", None)

    if isinstance(jet, dict):
        jet["route"] = []
        jet["route_index"] = 0
        jet["aim_pt"] = None
        jet["impact_pt"] = None
        jet["target_region"] = None
        jet["pause_until"] = -1
        jet["refresh_boxes_requested"] = True
        jet["last_signature"] = None
        jet["mode"] = "sem foco - aguardando novo foco"

        # Mantém o canvas liberado para a próxima imagem real.
        # Isso evita ficar preso no canvas apagado quando aparecer novo foco.
        jet["canvas"] = None

    box = globals().get("BOX_STATE", None)
    if isinstance(box, dict):
        box["locked"] = False
        box["display_regs1"] = []
        box["display_regs2"] = []
        box["depth_samples"] = {}

    ARDUINO_STATE["planner_last_signature"] = None
    ARDUINO_STATE["planner_stuck_count"] = 0
    ARDUINO_STATE["planner_last_reason"] = "sem foco - aguardando"



def tratar_transicao_foco_5(regs1, raw_frame1, raw_frame2, params):
    """
    Trata a transição:
    havia foco -> sem foco -> novo foco.

    A versão anterior liberava a rota quando não havia foco, mas quando o foco
    reaparecia o planejador podia continuar preso no estado/canvas vazio.
    Aqui, quando o foco reaparece, fazemos um reset controlado do estado e
    reprocessamos o frame atual uma vez.
    """
    foco_agora = len(regs1 or []) > 0
    foco_antes = ARDUINO_STATE.get("focus_present_prev")

    if foco_agora:
        ARDUINO_STATE["new_focus_count"] += 1
        ARDUINO_STATE["no_focus_count"] = 0
    else:
        ARDUINO_STATE["no_focus_count"] += 1

    # Primeira iteração: apenas registra o estado.
    if foco_antes is None:
        ARDUINO_STATE["focus_present_prev"] = foco_agora
        return False, None

    # Transição: tinha foco e agora não tem.
    if foco_antes and not foco_agora:
        if checkbox_reset_when_no_focus.value:
            liberar_estado_sem_foco_5()

        ARDUINO_STATE["focus_present_prev"] = False
        ARDUINO_STATE["planner_last_reason"] = "foco removido - aguardando reaparecer"
        return False, None

    # Transição: estava sem foco e agora apareceu foco novo.
    if (not foco_antes) and foco_agora:
        ARDUINO_STATE["focus_present_prev"] = True

        if checkbox_replan_when_focus_returns.value:
            # Reseta apenas quando o foco reaparece, não em loop.
            jet = globals().get("JET_SIM_STATE", None)
            if isinstance(jet, dict):
                jet["canvas"] = None
                jet["last_signature"] = None
                jet["route"] = []
                jet["route_index"] = 0
                jet["aim_pt"] = None
                jet["impact_pt"] = None
                jet["target_region"] = None
                jet["pause_until"] = -1
                jet["refresh_boxes_requested"] = True
                jet["mode"] = "replanejar: foco reapareceu"

            box = globals().get("BOX_STATE", None)
            if isinstance(box, dict):
                box["locked"] = False
                box["display_regs1"] = []
                box["display_regs2"] = []
                box["depth_samples"] = {}

            # Reprocessa o frame atual uma única vez depois do reset.
            # Isso evita esperar uma próxima iteração e ajuda a rota nascer
            # no mesmo ciclo em que o foco reapareceu.
            try:
                rf1 = raw_frame1.copy() if raw_frame1 is not None else raw_frame1
                rf2 = raw_frame2.copy() if raw_frame2 is not None else raw_frame2

                out = processar_frame_stereo(rf1, rf2, params)
                ARDUINO_STATE["planner_last_reason"] = "foco reapareceu - frame reprocessado"
                return True, out

            except Exception as e:
                ARDUINO_STATE["planner_last_reason"] = f"foco reapareceu, mas reprocessamento falhou: {type(e).__name__}"
                return False, None

        ARDUINO_STATE["planner_last_reason"] = "foco reapareceu - replanejamento automático desligado"
        return False, None

    ARDUINO_STATE["focus_present_prev"] = foco_agora
    return False, None


def reabrir_camera_individual_5(key, cam_index):
    """
    Tenta reabrir uma câmera individual quando ela falha.
    key: "cap1" ou "cap2"
    """
    if not checkbox_auto_recover_camera.value:
        return False

    try:
        cap_old = ARDUINO_STATE.get(key)
        if cap_old is not None:
            try:
                cap_old.release()
            except Exception:
                pass

        ARDUINO_STATE[key] = abrir_camera_celula5(
            int(cam_index),
            int(image_width_widget.value),
            int(image_height_widget.value),
            int(fps_widget_5.value)
        )

        ARDUINO_STATE["planner_last_reason"] = f"{key} reaberta"
        return True

    except Exception as e:
        ARDUINO_STATE["planner_last_reason"] = f"falha ao reabrir {key}: {type(e).__name__}"
        return False


def ler_frame_camera_robusto_5(key, cam_index):
    """
    Lê frame e tenta recuperar a câmera se a leitura falhar.
    """
    cap = ARDUINO_STATE.get(key)

    if cap is None:
        reabrir_camera_individual_5(key, cam_index)
        cap = ARDUINO_STATE.get(key)

    if cap is None:
        return False, None

    ok, frame = cap.read()

    if ok and frame is not None:
        return True, frame

    # Segunda tentativa: reabrir câmera.
    if reabrir_camera_individual_5(key, cam_index):
        cap = ARDUINO_STATE.get(key)

        if cap is not None:
            ok, frame = cap.read()

            if ok and frame is not None:
                return True, frame

    return False, None




# ------------------------------------------------------------
# Calibração estéreo real por xadrez
# ------------------------------------------------------------
def localizar_arquivo_calibracao_stereo_5():
    """
    Procura o arquivo de calibração informado no widget.
    """
    raw = stereo_calib_path_text.value.strip()

    candidatos = []

    if raw:
        candidatos.append(Path(raw).expanduser())

    if "BASE_DIR" in globals():
        candidatos.append(BASE_DIR / "parameters_setups" / "stereo_calibration2.npz")
        candidatos.append(BASE_DIR / "stereo_calibration2.npz")

    candidatos.append(Path.cwd() / "stereo_calibration2.npz")
    candidatos.append(Path("/mnt/data/stereo_calibration2.npz"))

    for c in candidatos:
        try:
            if c.exists():
                return c
        except Exception:
            pass

    return Path(raw) if raw else None


def carregar_calibracao_stereo_5(_=None):
    """
    Carrega calibração estéreo salva em .npz.

    Espera chaves:
    K1, D1, K2, D2, R, T
    Opcionalmente:
    R1, R2, P1, P2, Q
    """
    caminho = localizar_arquivo_calibracao_stereo_5()

    if caminho is None or not caminho.exists():
        ARDUINO_STATE["stereo_calib"] = None
        ARDUINO_STATE["stereo_maps"] = None
        ARDUINO_STATE["stereo_status"] = "arquivo de calibração não encontrado"

        registrar_acao(
            "<b>Arquivo de calibração não encontrado.</b><br>"
            f"Caminho informado: <code>{stereo_calib_path_text.value}</code>",
            "erro",
            widget=msg_stereo_calib if "msg_stereo_calib" in globals() else None
        )
        return False

    try:
        data = np.load(str(caminho), allow_pickle=True)

        required = ["K1", "D1", "K2", "D2", "R", "T"]
        missing = [k for k in required if k not in data.files]

        if missing:
            raise ValueError(f"Chaves ausentes no .npz: {missing}")

        calib = {
            "path": str(caminho),
            "K1": data["K1"].astype(np.float64),
            "D1": data["D1"].astype(np.float64),
            "K2": data["K2"].astype(np.float64),
            "D2": data["D2"].astype(np.float64),
            "R": data["R"].astype(np.float64),
            "T": data["T"].astype(np.float64),
            "keys": list(data.files),
        }

        for k in ["E", "F", "R1", "R2", "P1", "P2", "Q"]:
            if k in data.files:
                calib[k] = data[k].astype(np.float64)

        ARDUINO_STATE["stereo_calib"] = calib
        ARDUINO_STATE["stereo_maps"] = None
        ARDUINO_STATE["stereo_maps_key"] = None

        t_norm = float(np.linalg.norm(calib["T"]))
        ARDUINO_STATE["stereo_status"] = (
            f"calibração carregada | baseline≈{t_norm:.5f} unidade(s) do xadrez"
        )

        registrar_acao(
            f"<b>Calibração estéreo carregada.</b><br>"
            f"Arquivo: <code>{caminho}</code><br>"
            f"Chaves: <code>{', '.join(calib['keys'])}</code><br>"
            f"Baseline aproximado: <code>{t_norm:.5f}</code> na unidade usada no xadrez.<br>"
            "<b>Atenção:</b> se a calibração foi feita em outra resolução, ajuste Larg./Alt. calib.",
            "ok",
            widget=msg_stereo_calib if "msg_stereo_calib" in globals() else None
        )

        return True

    except Exception as e:
        ARDUINO_STATE["stereo_calib"] = None
        ARDUINO_STATE["stereo_maps"] = None
        ARDUINO_STATE["stereo_status"] = f"erro ao carregar calibração: {type(e).__name__}: {e}"

        registrar_acao(
            f"<b>Erro ao carregar calibração estéreo:</b><br><code>{type(e).__name__}: {e}</code>",
            "erro",
            widget=msg_stereo_calib if "msg_stereo_calib" in globals() else None
        )
        return False


def escalar_matriz_camera_5(K, src_w, src_h, dst_w, dst_h):
    """
    Escala matriz intrínseca quando a captura está em resolução diferente da calibração.
    """
    sx = float(dst_w) / float(src_w)
    sy = float(dst_h) / float(src_h)

    Ks = K.copy().astype(np.float64)
    Ks[0, 0] *= sx
    Ks[0, 2] *= sx
    Ks[1, 1] *= sy
    Ks[1, 2] *= sy

    return Ks


def obter_mapas_retificacao_stereo_5(frame_shape):
    """
    Cria ou reaproveita mapas de retificação para a resolução atual.
    """
    if not checkbox_use_stereo_calib_5.value:
        return None

    calib = ARDUINO_STATE.get("stereo_calib")

    if calib is None:
        # tenta carregar automaticamente uma vez
        carregar_calibracao_stereo_5()
        calib = ARDUINO_STATE.get("stereo_calib")

    if calib is None:
        return None

    h, w = frame_shape[:2]
    calib_w = int(calib_width_widget.value)
    calib_h = int(calib_height_widget.value)

    if calib_w <= 0 or calib_h <= 0:
        calib_w, calib_h = w, h

    alpha_rect = float(rectify_alpha_widget.value)
    key = (w, h, calib_w, calib_h, calib.get("path", ""), round(alpha_rect, 3))

    if ARDUINO_STATE.get("stereo_maps") is not None and ARDUINO_STATE.get("stereo_maps_key") == key:
        return ARDUINO_STATE["stereo_maps"]

    K1s = escalar_matriz_camera_5(calib["K1"], calib_w, calib_h, w, h)
    K2s = escalar_matriz_camera_5(calib["K2"], calib_w, calib_h, w, h)
    D1 = calib["D1"]
    D2 = calib["D2"]
    R = calib["R"]
    T = calib["T"]

    try:
        R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
            K1s, D1,
            K2s, D2,
            (w, h),
            R, T,
            flags=cv2.CALIB_ZERO_DISPARITY,
            alpha=alpha_rect
        )

        map1x, map1y = cv2.initUndistortRectifyMap(K1s, D1, R1, P1, (w, h), cv2.CV_32FC1)
        map2x, map2y = cv2.initUndistortRectifyMap(K2s, D2, R2, P2, (w, h), cv2.CV_32FC1)

        maps = {
            "map1x": map1x,
            "map1y": map1y,
            "map2x": map2x,
            "map2y": map2y,
            "Q": Q,
            "P1": P1,
            "P2": P2,
            "R1": R1,
            "R2": R2,
            "roi1": roi1,
            "roi2": roi2,
            "resolution": (w, h),
            "calibration_resolution": (calib_w, calib_h),
        }

        ARDUINO_STATE["stereo_maps"] = maps
        ARDUINO_STATE["stereo_maps_key"] = key
        ARDUINO_STATE["stereo_status"] = f"retificado {w}x{h} usando calib {calib_w}x{calib_h} | alpha={alpha_rect:.2f}"

        return maps

    except Exception as e:
        ARDUINO_STATE["stereo_status"] = f"erro retificação: {type(e).__name__}: {e}"
        return None


def imagem_retificada_invalida_5(img):
    """
    Detecta caso típico de mapa de retificação ruim: imagem quase toda preta.
    """
    if img is None:
        return True

    try:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        black_ratio = float(np.mean(gray < 6))
        mean_v = float(np.mean(gray))
        return black_ratio > 0.82 or mean_v < 8.0
    except Exception:
        return True


def retificar_par_stereo_5(frame1, frame2):
    """
    Retifica o par apenas quando "Retificar vídeo" estiver marcado.

    Por padrão fica desligado porque, com calibração/resolução/orientação incorreta,
    o remap pode recortar muito a imagem, dando impressão de zoom, ou deixar a
    câmera 2 preta.
    """
    ARDUINO_STATE["last_rectified_pair"] = False
    ARDUINO_STATE["last_rectification_fallback"] = ""

    if not checkbox_rectify_video_5.value:
        return frame1, frame2, False

    maps = obter_mapas_retificacao_stereo_5(frame1.shape)

    if maps is None:
        return frame1, frame2, False

    try:
        r1 = cv2.remap(frame1, maps["map1x"], maps["map1y"], cv2.INTER_LINEAR)
        r2 = cv2.remap(frame2, maps["map2x"], maps["map2y"], cv2.INTER_LINEAR)

        if imagem_retificada_invalida_5(r1) or imagem_retificada_invalida_5(r2):
            ARDUINO_STATE["last_rectification_fallback"] = "retificação descartada: imagem preta/cortada"
            ARDUINO_STATE["stereo_status"] = ARDUINO_STATE.get("stereo_status", "") + " | fallback raw"
            return frame1, frame2, False

        ARDUINO_STATE["last_rectified_pair"] = True
        return r1, r2, True

    except Exception as e:
        ARDUINO_STATE["last_rectification_fallback"] = f"falha remap: {type(e).__name__}"
        return frame1, frame2, False



def profundidade_por_Q_5(x, y, disparity):
    """
    Reprojeta um ponto usando a matriz Q.

    Retorna Z na unidade usada na calibração do xadrez, quando a calibração tiver escala real.
    """
    if not ARDUINO_STATE.get("last_rectified_pair", False):
        return None

    maps = ARDUINO_STATE.get("stereo_maps")

    if not maps or "Q" not in maps:
        return None

    try:
        Q = maps["Q"]
        vec = np.array([float(x), float(y), float(disparity), 1.0], dtype=np.float64)
        X, Y, Z, W = Q @ vec

        if abs(W) < 1e-9:
            return None

        X /= W
        Y /= W
        Z /= W

        return float(abs(Z))

    except Exception:
        return None


def ler_par_cameras_sincronizado_5():
    """
    Lê o par estéreo tentando reduzir atraso entre câmeras.

    Usa grab() nas duas câmeras antes de retrieve(), em vez de read() sequencial.
    Isso não é sincronismo de hardware, mas reduz diferença temporal e esvazia buffers.
    """
    cap1 = ARDUINO_STATE.get("cap1")
    cap2 = ARDUINO_STATE.get("cap2")

    if cap1 is None or cap2 is None:
        return False, None, False, None

    flush_n = int(sync_flush_frames_widget.value)

    try:
        # Descarta frames antigos dos buffers.
        for _ in range(max(0, flush_n)):
            cap1.grab()
            cap2.grab()

        t1 = time.time()
        okg1 = cap1.grab()
        okg2 = cap2.grab()
        tm = time.time()
        ok1, frame1 = cap1.retrieve()
        ok2, frame2 = cap2.retrieve()
        t2 = time.time()

        ARDUINO_STATE["frame_pair_id"] += 1
        ARDUINO_STATE["last_pair_dt_ms"] = abs(t2 - t1) * 1000.0
        ARDUINO_STATE["last_capture_method"] = "grab/retrieve"

        if okg1 and okg2 and ok1 and ok2 and frame1 is not None and frame2 is not None:
            return True, frame1, True, frame2

    except Exception:
        pass

    # Fallback: leitura robusta antiga.
    ok1, frame1 = ler_frame_camera_robusto_5("cap1", dropdown_cam1_5.value)
    ok2, frame2 = ler_frame_camera_robusto_5("cap2", dropdown_cam2_5.value)

    ARDUINO_STATE["frame_pair_id"] += 1
    ARDUINO_STATE["last_capture_method"] = "fallback read"

    return ok1, frame1, ok2, frame2


async def camera_simulation_loop_5():
    ARDUINO_STATE["camera_running"] = True

    try:
        while ARDUINO_STATE["camera_running"]:
            loop_t0 = time.time()

            capture_t0 = time.time()
            ok1, frame1, ok2, frame2 = ler_par_cameras_sincronizado_5()
            ARDUINO_STATE["perf_capture_ms"] = (time.time() - capture_t0) * 1000.0

            if not ok1 or frame1 is None or not ok2 or frame2 is None:
                status = (
                    f"Célula 5 | Falha de leitura de câmera\n"
                    f"Cam1 ok: {ok1} | Cam2 ok: {ok2}\n"
                    f"Recuperação automática: {'ligada' if checkbox_auto_recover_camera.value else 'desligada'}\n"
                    f"Último motivo: {ARDUINO_STATE.get('planner_last_reason')}"
                )
                panel = painel_fallback_cameras_5(frame1, frame2, status)
                video_widget_5.value = encode_jpg_bgr(panel, quality=int(jpeg_quality_widget_5.value))
                status_arduino.value = f"<pre>{status}</pre>"
                await asyncio.sleep(0.2)
                continue

            # Evita cópias completas de frame por padrão.
            raw_frame1_5 = frame1
            raw_frame2_5 = frame2

            params_t0 = time.time()

            if "params_selecionados_celula4" in globals():
                params = params_selecionados_celula4()
            elif "CURRENT_PREPROCESS_PARAMS" in globals():
                params = CURRENT_PREPROCESS_PARAMS.copy()
            else:
                params = {}

            setup3_nome = dropdown_param_setup_3.value if "dropdown_param_setup_3" in globals() else "Usar parâmetros atuais"

            if setup3_nome not in (None, "Usar parâmetros atuais"):
                setup3_data = carregar_setups_parametricos_celula3()
                setup3_params = setup3_data.get(setup3_nome)

                if isinstance(setup3_params, dict):
                    params.update(setup3_params)

            params["pressure_calib"] = float(pressure_calib_widget.value)
            ARDUINO_STATE["perf_params_ms"] = (time.time() - params_t0) * 1000.0

            try:
                process_t0 = time.time()
                frame1, frame2, filtered1, filtered2, regs1, regs2 = processar_frame_stereo(frame1, frame2, params)
                ARDUINO_STATE["last_process_ms"] = (time.time() - process_t0) * 1000.0
                ARDUINO_STATE["perf_process_ms"] = ARDUINO_STATE["last_process_ms"]

                transition_t0 = time.time()
                reprocessou_foco, out_reprocessado = tratar_transicao_foco_5(
                    regs1,
                    raw_frame1_5,
                    raw_frame2_5,
                    params
                )

                if reprocessou_foco and out_reprocessado is not None:
                    frame1, frame2, filtered1, filtered2, regs1, regs2 = out_reprocessado

                ARDUINO_STATE["perf_transition_ms"] = (time.time() - transition_t0) * 1000.0

                yolo_t0 = time.time()
                human_boxes = atualizar_seguranca_humana_5(raw_frame1_5)
                ARDUINO_STATE["perf_yolo_ms"] = (time.time() - yolo_t0) * 1000.0

                frame1 = desenhar_humanos_5(frame1, human_boxes)

                ARDUINO_STATE["last_loop_ms"] = (time.time() - loop_t0) * 1000.0

                # Status detalhado apenas a cada N frames.
                status_t0 = time.time()
                frame_id = int(ARDUINO_STATE.get("frame_pair_id", 0))
                status_every = max(1, int(status_update_every_widget_5.value)) if "status_update_every_widget_5" in globals() else 8

                if frame_id % status_every == 0:
                    status_planner = status_planejador_5(regs1)
                    status = (
                        f"Célula 5 | Cam1: {dropdown_cam1_5.value} | Cam2: {dropdown_cam2_5.value}\n"
                        f"Regiões cam1: {len(regs1)} | Matches cam2: {len(regs2)} | "
                        f"Jato: {'rodando' if globals().get('JET_SIM_STATE', {}).get('running', False) else 'parado'}\n"
                        f"{status_planner}\n"
                        f"Captura: {ARDUINO_STATE.get('last_capture_method')} | "
                        f"dt par≈{ARDUINO_STATE.get('last_pair_dt_ms', 0):.1f} ms | "
                        f"pair={ARDUINO_STATE.get('frame_pair_id', 0)}\n"
                        f"TEMPOS ms | total={ARDUINO_STATE.get('perf_total_ms', 0):.1f} | "
                        f"captura={ARDUINO_STATE.get('perf_capture_ms', 0):.1f} | "
                        f"params={ARDUINO_STATE.get('perf_params_ms', 0):.1f} | "
                        f"proc={ARDUINO_STATE.get('perf_process_ms', 0):.1f} | "
                        f"trans={ARDUINO_STATE.get('perf_transition_ms', 0):.1f} | "
                        f"yolo={ARDUINO_STATE.get('perf_yolo_ms', 0):.1f} | "
                        f"painel={ARDUINO_STATE.get('perf_panel_ms', 0):.1f} | "
                        f"jpeg={ARDUINO_STATE.get('perf_encode_ms', 0):.1f} | "
                        f"serial={ARDUINO_STATE.get('perf_serial_ms', 0):.1f}\n"
                        f"match≈{ARDUINO_STATE.get('last_match_ms', 0):.1f} ms/{ARDUINO_STATE.get('last_match_count', 0)} | "
                        f"YOLO cada {human_check_every_widget.value}f em {human_yolo_width_widget.value}px | "
                        f"Painel escala={panel_scale_widget_5.value:.2f}, JPEG={jpeg_quality_widget_5.value}\n"
                        f"Alvo: {ARDUINO_STATE.get('current_target_depth_text', 'Dist: --')} "
                        f"(media {ARDUINO_STATE.get('depth_publish_every_frames', 90)}f; tentativa {ARDUINO_STATE.get('depth_internal_every_frames', 15)}f)\n"
                        f"Dist dbg: amostras janela={ARDUINO_STATE.get('depth_sample_count_window', 0)} | "
                        f"total={ARDUINO_STATE.get('depth_total_samples', 0)} | "
                        f"ultima={ARDUINO_STATE.get('depth_last_sample_text', 'sem amostra')} | "
                        f"fonte={ARDUINO_STATE.get('depth_last_source', 'nenhuma')} | "
                        f"score={ARDUINO_STATE.get('depth_last_match_score', None)} | "
                        f"disp={ARDUINO_STATE.get('depth_last_disparity', None)}\n"
                        f"Humano: {ARDUINO_STATE.get('human_status', '')} | "
                        f"Mudança cena={ARDUINO_STATE.get('scene_change_score', 0):.2f} "
                        f"{ARDUINO_STATE.get('scene_change_last_reason', '')}\n"
                        f"Arduino: {'conectado' if serial_is_open() else 'desconectado'} | "
                        f"Envios: {ARDUINO_STATE['send_count']} | "
                        f"Último cmd: {ARDUINO_STATE.get('last_command')}"
                    )
                    ARDUINO_STATE["last_status_text"] = status
                    status_arduino.value = f"<pre>{status}</pre>"
                else:
                    status = ARDUINO_STATE.get("last_status_text", "")

                ARDUINO_STATE["perf_status_ms"] = (time.time() - status_t0) * 1000.0

                panel_t0 = time.time()
                try:
                    panel = montar_painel_camera(frame1, frame2, filtered1, filtered2, regs1, regs2, status, params)
                except Exception as e_panel:
                    status += f"\nPainel fallback: {type(e_panel).__name__}: {e_panel}"
                    panel = painel_fallback_cameras_5(frame1, frame2, status)

                ARDUINO_STATE["perf_panel_ms"] = (time.time() - panel_t0) * 1000.0

                encode_t0 = time.time()
                video_widget_5.value = encode_jpg_bgr(panel, quality=int(jpeg_quality_widget_5.value))
                ARDUINO_STATE["perf_encode_ms"] = (time.time() - encode_t0) * 1000.0

                serial_t0 = time.time()
                if len(regs1) > 0 and not ARDUINO_STATE.get("human_detected", False):
                    maybe_send_target_to_arduino()

                ARDUINO_STATE["perf_serial_ms"] = (time.time() - serial_t0) * 1000.0
                ARDUINO_STATE["perf_total_ms"] = (time.time() - loop_t0) * 1000.0
                ARDUINO_STATE["last_loop_ms"] = ARDUINO_STATE["perf_total_ms"]

            except Exception as e:
                status = (
                    f"Erro no processamento: {type(e).__name__}: {e}\n"
                    f"Cam1: {dropdown_cam1_5.value} | Cam2: {dropdown_cam2_5.value}\n"
                    f"A imagem bruta continua sendo exibida para evitar congelamento visual."
                )
                panel = painel_fallback_cameras_5(frame1 if 'frame1' in locals() else None, frame2 if 'frame2' in locals() else None, status)
                video_widget_5.value = encode_jpg_bgr(panel, quality=int(jpeg_quality_widget_5.value))
                status_arduino.value = f"<pre>{status}</pre>"

            await asyncio.sleep(float(loop_interval_widget_5.value))

    finally:
        ARDUINO_STATE["camera_running"] = False
        ARDUINO_STATE["camera_task"] = None
        liberar_cameras_simulacao_5()


def liberar_cameras_simulacao_5():
    """
    Libera cap1/cap2 imediatamente.
    """
    for key in ["cap1", "cap2"]:
        cap = ARDUINO_STATE.get(key)
        if cap is not None:
            try:
                cap.release()
            except Exception:
                pass
        ARDUINO_STATE[key] = None

    button_open_cameras_5.disabled = False
    button_stop_cameras_5.disabled = True


def abrir_cameras_e_simular(_=None):
    if ARDUINO_STATE["camera_running"]:
        registrar_acao(
            "<b>As câmeras da simulação já estão abertas.</b>",
            "aviso",
            widget=msg_camera if "msg_camera" in globals() else None
        )
        return False

    # A câmera de referência pode ocupar a mesma porta da simulação.
    # Libera antes de abrir cap1/cap2.
    if ARDUINO_STATE.get("ref_camera_running", False) or ARDUINO_STATE.get("ref_cap") is not None:
        parar_camera_referencia_5()

    liberar_cameras_simulacao_5()

    cam1 = int(dropdown_cam1_5.value)
    cam2 = int(dropdown_cam2_5.value)

    if cam1 < 0 or cam2 < 0:
        registrar_acao(
            "<b>Erro:</b> selecione duas câmeras válidas.",
            "erro",
            widget=msg_camera if "msg_camera" in globals() else None
        )
        return False

    if cam1 == cam2:
        registrar_acao(
            "<b>Erro:</b> câmera 1 e câmera 2 não podem ser iguais.",
            "erro",
            widget=msg_camera if "msg_camera" in globals() else None
        )
        return False

    apply_jet_calibration()

    try:
        ARDUINO_STATE["cap1"] = abrir_camera_celula5(
            cam1,
            int(image_width_widget.value),
            int(image_height_widget.value),
            int(fps_widget_5.value)
        )

        ARDUINO_STATE["cap2"] = abrir_camera_celula5(
            cam2,
            int(image_width_widget.value),
            int(image_height_widget.value),
            int(fps_widget_5.value)
        )

        button_open_cameras_5.disabled = True
        button_stop_cameras_5.disabled = False

        ARDUINO_STATE["camera_task"] = asyncio.create_task(camera_simulation_loop_5())

        registrar_acao(
            f"<b>Câmeras abertas.</b><br>Cam1: /dev/video{cam1}<br>Cam2: /dev/video{cam2}",
            "ok",
            widget=msg_camera if "msg_camera" in globals() else None
        )
        return True

    except Exception as e:
        liberar_cameras_simulacao_5()
        registrar_acao(
            f"<b>Erro ao abrir câmeras:</b> {type(e).__name__}: {e}<br>"
            "Verifique se a câmera de referência está parada e se nenhum outro programa está usando /dev/video.",
            "erro",
            widget=msg_camera if "msg_camera" in globals() else None
        )
        return False



def parar_cameras_5(_=None):
    """
    Para a simulação e libera cap1/cap2 imediatamente.
    """
    ARDUINO_STATE["camera_running"] = False

    task = ARDUINO_STATE.get("camera_task")
    if task is not None:
        try:
            task.cancel()
        except Exception:
            pass
        ARDUINO_STATE["camera_task"] = None

    liberar_cameras_simulacao_5()

    registrar_acao(
        "<b>Câmeras da simulação paradas e liberadas.</b>",
        "ok",
        widget=msg_camera if "msg_camera" in globals() else None
    )
    return True



def play_jet_5(_=None):
    if "JET_SIM_STATE" not in globals():
        globals()["JET_SIM_STATE"] = {}
    JET_SIM_STATE["running"] = True
    JET_SIM_STATE["route"] = []
    JET_SIM_STATE["route_index"] = 0
    JET_SIM_STATE["canvas"] = None
    JET_SIM_STATE["live_reference_mask"] = None
    JET_SIM_STATE["live_reference_pixels"] = 0
    JET_SIM_STATE["pause_until_frame"] = -1
    JET_SIM_STATE["mode"] = "borda"
    JET_SIM_STATE["movement_phase"] = "aproximação/entre trajetórias"
    ARDUINO_STATE["depth_window_values"] = []
    ARDUINO_STATE["depth_window_proxy_flags"] = []
    ARDUINO_STATE["depth_sample_count_window"] = 0
    ARDUINO_STATE["depth_window_start_frame"] = int(ARDUINO_STATE.get("frame_pair_id", 0))
    ARDUINO_STATE["depth_first_publish_done"] = False
    registrar_acao("<b>Jato iniciado.</b> Estratégia ativa: regiões grandes por borda inferior; regiões pequenas por centróide.", "ok", widget=msg_camera if "msg_camera" in globals() else None)


def stop_jet_5(_=None):
    if "JET_SIM_STATE" in globals():
        JET_SIM_STATE["running"] = False
    registrar_acao("<b>Jato pausado.</b>", "ok")


def reset_jet_5(_=None):
    if "resetar_imagem_jato" in globals():
        resetar_imagem_jato()
    elif "JET_SIM_STATE" in globals():
        JET_SIM_STATE["canvas"] = None
        JET_SIM_STATE["route"] = []
        JET_SIM_STATE["route_index"] = 0
        JET_SIM_STATE["target_region"] = None
        JET_SIM_STATE["live_reference_mask"] = None
        JET_SIM_STATE["live_reference_pixels"] = 0
        JET_SIM_STATE["refresh_boxes_requested"] = True

    registrar_acao("<b>Imagem do simulador resetada.</b>", "ok", widget=msg_camera if "msg_camera" in globals() else None)
    return True


# ------------------------------------------------------------
# Setup save/load/delete
# ------------------------------------------------------------
def atualizar_dropdown_setups_arduino(selecionar=None):
    global setups_arduino_dict
    setups_arduino_dict = carregar_setups_arduino()
    opts = ["Nenhum setup Arduino"] + sorted(setups_arduino_dict.keys())
    dropdown_setup_arduino.options = opts

    if selecionar in opts:
        dropdown_setup_arduino.value = selecionar
    else:
        dropdown_setup_arduino.value = opts[0]


def salvar_setup_arduino(_=None):
    nome = text_setup_arduino_nome.value.strip()

    if not nome:
        status_arduino.value = "<b>Informe um nome para salvar o setup.</b>"
        return

    data = carregar_setups_arduino()
    data[nome] = get_mapping_params_from_widgets()
    salvar_setups_arduino(data)
    atualizar_dropdown_setups_arduino(selecionar=nome)
    status_arduino.value = (
        f"<b>Setup salvo:</b> {nome}<br>"
        "Foram salvos os parâmetros editáveis da célula 5: serial, imagem, envio, deadband, "
        "pressão, posição inicial, pontos de calibração, setup da célula 3 selecionado, recuperação de câmera, segurança humana YOLO, sincronização de câmeras e calibração estéreo. Os parâmetros térmicos e de trajetória ficam centralizados no setup da célula 3."
    )


def excluir_setup_arduino(_=None):
    nome = dropdown_setup_arduino.value

    if nome in (None, "Nenhum setup Arduino"):
        status_arduino.value = "<b>Nenhum setup selecionado para excluir.</b>"
        return

    data = carregar_setups_arduino()

    if nome in data:
        del data[nome]
        salvar_setups_arduino(data)

    atualizar_dropdown_setups_arduino()
    status_arduino.value = f"<b>Setup excluído:</b> {nome}"


def carregar_setup_arduino_por_dropdown(change=None):
    nome = dropdown_setup_arduino.value

    if nome in (None, "Nenhum setup Arduino"):
        return

    data = carregar_setups_arduino()
    params = data.get(nome)

    if params is None:
        return

    apply_mapping_params_to_widgets(params)
    apply_jet_calibration()
    status_arduino.value = f"<b>Setup carregado:</b> {nome}"


# ------------------------------------------------------------
# Callbacks
# ------------------------------------------------------------
button_create_sketch.on_click(lambda _: executar_com_feedback("Criar/atualizar sketch", lambda: criar_ou_atualizar_sketch(), msg_sketch))
button_open_vscode.on_click(lambda _: executar_com_feedback("Abrir sketch no VS Code", lambda: abrir_sketch_vscode(), msg_sketch))
button_check_arduino_install.on_click(lambda _: executar_com_feedback("Verificar instalação Arduino IDE", lambda: verificar_instalacao_arduino_ide(), msg_sketch))
button_open_arduino_ide.on_click(lambda _: executar_com_feedback("Abrir sketch no Arduino IDE", lambda: abrir_sketch_arduino_ide(), msg_sketch))

button_refresh_serial.on_click(lambda _: executar_com_feedback("Atualizar portas seriais", lambda: refresh_serial_ports(), msg_serial))
button_install_pyserial.on_click(lambda _: executar_com_feedback("Instalar pyserial", lambda: instalar_pyserial_no_kernel(), msg_serial))
button_check_serial.on_click(lambda _: executar_com_feedback("Verificar porta Arduino", lambda: verificar_porta_arduino(), msg_serial))
button_connect_serial.on_click(lambda _: executar_com_feedback("Conectar Arduino", lambda: connect_serial(), msg_serial))
button_disconnect_serial.on_click(lambda _: executar_com_feedback("Desconectar Arduino", lambda: disconnect_serial(), msg_serial))
button_zero_servos.on_click(lambda _: executar_com_feedback("Zerar servos", lambda: zerar_servos_e_sliders(), msg_calib))
button_center_mount_servos.on_click(lambda _: executar_com_feedback("Posicionar servos 90/90", lambda: posicionar_servos_montagem_central(), msg_initial_position))
button_apply_initial_to_calib.on_click(lambda _: executar_com_feedback("Aplicar base inicial", lambda: aplicar_posicao_inicial_nas_barras(), msg_initial_position))

button_refresh_cameras_5.on_click(lambda _: executar_com_feedback("Atualizar câmeras", lambda: refresh_cameras_5(), msg_camera))
button_open_reference_camera.on_click(lambda _: executar_com_feedback("Abrir câmera de referência", lambda: abrir_camera_referencia_5(), msg_camera))
button_stop_reference_camera.on_click(lambda _: executar_com_feedback("Parar câmera de referência", lambda: parar_camera_referencia_5(), msg_camera))
button_open_cameras_5.on_click(lambda _: executar_com_feedback("Abrir câmeras e simular", lambda: abrir_cameras_e_simular(), msg_camera))
button_stop_cameras_5.on_click(lambda _: executar_com_feedback("Parar câmeras", lambda: parar_cameras_5(), msg_camera))

button_play_jet_5.on_click(lambda _: executar_com_feedback("Play jato", lambda: play_jet_5(), msg_camera))
button_stop_jet_5.on_click(lambda _: executar_com_feedback("Stop jato", lambda: stop_jet_5(), msg_camera))
button_reset_jet_5.on_click(lambda _: executar_com_feedback("Resetar imagem", lambda: reset_jet_5(), msg_camera))
button_load_human_yolo.on_click(lambda _: executar_com_feedback("Carregar YOLO humano", lambda: carregar_yolo_humano_5(), msg_human_safety))
button_load_stereo_calib_5.on_click(lambda _: executar_com_feedback("Carregar calibração estéreo", lambda: carregar_calibracao_stereo_5(), msg_stereo_calib))

def invalidar_mapas_retificacao_5(change=None):
    ARDUINO_STATE["stereo_maps"] = None
    ARDUINO_STATE["stereo_maps_key"] = None

rectify_alpha_widget.observe(invalidar_mapas_retificacao_5, names="value")
checkbox_rectify_video_5.observe(invalidar_mapas_retificacao_5, names="value")
calib_width_widget.observe(invalidar_mapas_retificacao_5, names="value")
calib_height_widget.observe(invalidar_mapas_retificacao_5, names="value")

def atualizar_texto_distancia_por_fator_5(change=None):
    valor = ARDUINO_STATE.get("depth_display_value", None)
    if valor is not None:
        ARDUINO_STATE["depth_display_text"] = formatar_distancia_alvo_5(
            valor,
            is_proxy=bool(ARDUINO_STATE.get("depth_display_is_proxy", True))
        )
        ARDUINO_STATE["current_target_depth_text"] = ARDUINO_STATE["depth_display_text"]

distance_correction_widget.observe(atualizar_texto_distancia_por_fator_5, names="value")

button_start_tl.on_click(lambda _: executar_com_feedback("Iniciar calibração superior esquerdo", lambda: set_active_calibration_point("tl"), msg_calib))
button_start_tr.on_click(lambda _: executar_com_feedback("Iniciar calibração superior direito", lambda: set_active_calibration_point("tr"), msg_calib))
button_start_bl.on_click(lambda _: executar_com_feedback("Iniciar calibração inferior esquerdo", lambda: set_active_calibration_point("bl"), msg_calib))

button_send_tl.on_click(lambda _: executar_com_feedback("Enviar calibração superior esquerdo", lambda: finalizar_calibracao_ponto("tl", sx_tl.value, sy_tl.value, "Superior esquerdo"), msg_calib))
button_send_tr.on_click(lambda _: executar_com_feedback("Enviar calibração superior direito", lambda: finalizar_calibracao_ponto("tr", sx_tr.value, sy_tr.value, "Superior direito"), msg_calib))
button_send_bl.on_click(lambda _: executar_com_feedback("Enviar calibração inferior esquerdo", lambda: finalizar_calibracao_ponto("bl", sx_bl.value, sy_bl.value, "Inferior esquerdo"), msg_calib))

for s in [sx_tl, sy_tl, sx_tr, sy_tr, sx_bl, sy_bl]:
    s.observe(slider_calib_changed, names="value")

button_apply_calibration.on_click(lambda _: executar_com_feedback("Aplicar calibração do jato", lambda: apply_jet_calibration(), msg_calib))

button_save_setup_arduino.on_click(lambda _: executar_com_feedback("Salvar setup Arduino", lambda: salvar_setup_arduino(), msg_setup))
button_delete_setup_arduino.on_click(lambda _: executar_com_feedback("Excluir setup Arduino", lambda: excluir_setup_arduino(), msg_setup))
dropdown_setup_arduino.observe(carregar_setup_arduino_por_dropdown, names="value")

button_refresh_param_setup_3.on_click(lambda _: executar_com_feedback("Atualizar setups da célula 3", lambda: atualizar_setups_parametricos_celula3(), msg_param_setup))
button_load_param_setup_3.on_click(lambda _: executar_com_feedback("Carregar setup da célula 3", lambda: carregar_setup_parametrico_celula3(), msg_param_setup))
dropdown_param_setup_3.observe(lambda change: aplicar_setup_parametrico_celula3_por_nome(change["new"]), names="value")



# ------------------------------------------------------------
# Ajustes avançados compactados
# ------------------------------------------------------------
advanced_params_accordion = widgets.Accordion(children=[
    widgets.VBox([
        widgets.HTML("<b>Visualização e debug</b>"),
        widgets.HBox([checkbox_invert_hot_mask_5, thermal_seg_mode_widget]),
        widgets.HBox([checkbox_draw_hot_contours_5, checkbox_draw_hot_boxes_5, checkbox_show_region_labels_5]),

        widgets.HTML("<b>Desempenho / sincronização / matching</b>"),
        widgets.HBox([sync_flush_frames_widget, stereo_match_every_widget, stereo_top_regions_widget]),
        widgets.HBox([jpeg_quality_widget_5, panel_scale_widget_5, status_update_every_widget_5]),
        widgets.HBox([scene_check_every_widget_5, human_yolo_width_widget]),
        widgets.HBox([stereo_search_margin_widget, stereo_template_pad_widget]),

        widgets.HTML("<b>Calibração estéreo por xadrez</b>"),
        widgets.HTML(
            "Use aqui o arquivo <code>stereo_calibration2.npz</code>. "
            "Se a calibração foi feita em resolução diferente da captura, ajuste Larg./Alt. calib."
        ),
        widgets.HBox([checkbox_use_stereo_calib_5, checkbox_rectify_video_5, button_load_stereo_calib_5, rectify_alpha_widget]),
        widgets.HBox([stereo_calib_path_text]),
        widgets.HBox([calib_width_widget, calib_height_widget]),
        widgets.HTML("<span style='font-size:12px; color:#666;'>Retificar vídeo fica desligado por padrão para evitar zoom/imagem preta. Mesmo sem retificar visualmente, a célula tenta calcular profundidade por triangulação raw usando K, D, R e T. Ligue Retificar vídeo apenas se a imagem retificada estiver visualmente correta.</span>"),
        msg_stereo_calib,

        widgets.HTML("<b>Recuperação de câmera / novo foco</b>"),
        widgets.HBox([checkbox_auto_recover_camera, checkbox_reset_when_no_focus, checkbox_replan_when_focus_returns]),

        widgets.HTML("<b>Segurança humana com YOLO</b>"),
        widgets.HTML(
            "Quando um humano é detectado na câmera 1, o jato é interrompido imediatamente. "
            "A célula tenta carregar o YOLO automaticamente uma vez; se falhar, use o botão abaixo e confira o status."
        ),
        widgets.HBox([checkbox_human_safety_yolo, human_yolo_model_text, button_load_human_yolo]),
        widgets.HBox([human_conf_widget, human_check_every_widget]),
        msg_human_safety,
    ])
])
advanced_params_accordion.set_title(0, "Ajustes avançados opcionais")
advanced_params_accordion.selected_index = None


status_debug_accordion = widgets.Accordion(children=[
    widgets.VBox([
        widgets.HTML("<b>Status e diagnóstico interno</b>"),
        status_arduino,
    ])
])
status_debug_accordion.set_title(0, "Status / diagnóstico")
status_debug_accordion.selected_index = None

# ------------------------------------------------------------
# Interface
# ------------------------------------------------------------
ui_arduino = widgets.VBox([
    widgets.HTML("<h3>Célula 5 — Arduino Uno + MG90S + simulação de jato</h3>"),

    widgets.HTML("<h4>1) Sketch do Arduino</h4>"),
    widgets.HTML("Crie o sketch na pasta do projeto. Depois faça upload para o Arduino Uno pelo Arduino IDE ou pela extensão Arduino do VS Code."),
    widgets.HBox([
        widgets.VBox([button_create_sketch, button_open_vscode, button_check_arduino_install, button_open_arduino_ide]),
        msg_sketch,
    ]),

    widgets.HTML("<h4>2) Setup de calibração dos servos</h4>"),
    widgets.HBox([dropdown_setup_arduino, text_setup_arduino_nome]),
    widgets.HBox([
        widgets.HBox([button_save_setup_arduino, button_delete_setup_arduino]),
        msg_setup,
    ]),

    widgets.HTML("<h4>2.1) Setup paramétrico da célula 3</h4>"),
    widgets.HTML(
        "Selecione aqui um dos setups salvos na célula 3. "
        "Ao selecionar, a célula 5 carrega automaticamente o pré-processamento térmico "
        "e todos os parâmetros de trajetória já ajustados."
    ),
    widgets.HBox([dropdown_param_setup_3, button_refresh_param_setup_3, button_load_param_setup_3]),
    msg_param_setup,

    widgets.HTML("<h4>3) Arduino / Serial</h4>"),
    widgets.HTML(
        "Depois de enviar o sketch ao Arduino, use <b>Verificar porta</b>. "
        "O teste envia <code>X90.0,Y90.0,P22.0</code> e espera uma resposta <code>OK</code>."
    ),
    widgets.HBox([dropdown_serial_port, button_refresh_serial, button_install_pyserial]),
    widgets.HBox([
        widgets.HBox([serial_baud_widget, button_check_serial, button_connect_serial, button_disconnect_serial, button_zero_servos]),
        msg_serial,
    ]),

    widgets.HTML("<h4>3.1) Posição inicial de montagem dos servos</h4>"),
    widgets.HTML(
        "<b>Antes da calibração:</b> posicione os servos em aproximadamente <b>90° em X</b> e <b>90° em Y</b>. "
        "Se for a primeira vez utilizando estes motores, mantenha os atuadores/jato <b>desacoplados</b> nesta etapa "
        "para evitar colisão ou esforço mecânico. Depois que os servos estiverem em 90/90, monte fisicamente o jato "
        "na posição central, apontando para o <b>centro da imagem</b>."
    ),
    widgets.HBox([initial_servo_x, initial_servo_y, button_center_mount_servos, button_apply_initial_to_calib]),
    msg_initial_position,

    widgets.HTML("<h4>4) Calibração dos servos por 3 pontos</h4>"),
    widgets.HTML(
        "Abra a <b>câmera de referência</b> na seção 6. "
        "Depois ajuste um ponto por vez: clique em <b>Iniciar</b>, mova X/Y com os sliders, "
        "e clique em <b>Enviar</b> para salvar o ponto e interromper o envio em tempo real."
    ),
    widgets.HBox([checkbox_send_servo_with_slider]),
    widgets.HTML("<b>Canto superior esquerdo da imagem</b>"),
    widgets.HBox([sx_tl, sy_tl, button_start_tl, button_send_tl]),
    widgets.HTML("<b>Canto superior direito da imagem</b>"),
    widgets.HBox([sx_tr, sy_tr, button_start_tr, button_send_tr]),
    widgets.HTML("<b>Canto inferior esquerdo da imagem</b>"),
    widgets.HBox([sx_bl, sy_bl, button_start_bl, button_send_bl]),
    widgets.HBox([
        button_apply_calibration,
        msg_calib,
    ]),

    widgets.HTML("<h4>5) Parâmetros mínimos do jato e envio</h4>"),
    widgets.HBox([image_width_widget, image_height_widget, fps_widget_5]),
    widgets.HBox([loop_interval_widget_5, deadband_widget]),
    widgets.HBox([pressure_calib_widget, dropdown_target_source, checkbox_send_during_camera]),
    widgets.HBox([distance_correction_widget]),
    widgets.HTML(
        "<b>Pré-processamento térmico e trajetória:</b> vêm automaticamente do "
        "<b>setup da célula 3</b> selecionado na seção 2.1. "
        "Não é necessário ajustar Área p/ offset, Offset Y, Passo rota, Permanência, Vel. mira ou Raio de apagamento aqui."
    ),
    advanced_params_accordion,

    widgets.HTML("<h4>6) Câmeras + simulação + envio ao Arduino</h4>"),
    widgets.HBox([dropdown_cam1_5, dropdown_cam2_5, button_refresh_cameras_5]),
    widgets.HTML("<b>Câmera de referência para calibração</b>"),
    widgets.HBox([
        widgets.HBox([button_open_reference_camera, button_stop_reference_camera]),
        msg_camera,
    ]),
    reference_camera_widget,
    widgets.HTML("<b>Simulação do jato e envio ao Arduino</b>"),
    widgets.HBox([
        widgets.HBox([button_open_cameras_5, button_stop_cameras_5, button_play_jet_5, button_stop_jet_5, button_reset_jet_5]),
        msg_camera,
    ]),
    video_widget_5,
    status_debug_accordion,
])

display(ui_arduino)

# Verificação inicial da IDE. Se não estiver instalada, já mostra o passo a passo ao lado dos botões do sketch.
_info_arduino_ide = detectar_instalacao_arduino_ide()
if not _info_arduino_ide["installed"]:
    msg_sketch.value = html_passo_a_passo_instalacao_arduino()
else:
    encontrados_txt = "<br>".join([f"<b>{n}</b>: <code>{c}</code>" for n, c in _info_arduino_ide["found"]])
    msg_sketch.value = f"<span style='color:#1b8a3a'><b>Arduino IDE detectado.</b><br>{encontrados_txt}</span>"

_setups_celula3_iniciais = nomes_setups_parametricos_celula3()

if len(_setups_celula3_iniciais) > 1:
    # Seleciona automaticamente o primeiro setup salvo, evitando que a célula 5
    # comece com parâmetros duplicados/fallback quando já existe setup da célula 3.
    if dropdown_param_setup_3.value in (None, "Usar parâmetros atuais"):
        dropdown_param_setup_3.value = _setups_celula3_iniciais[1]
    aplicar_setup_parametrico_celula3_por_nome(dropdown_param_setup_3.value)
else:
    msg_param_setup.value = (
        "<span style='color:#b36b00'><b>Nenhum setup da célula 3 encontrado.</b><br>"
        "Salve um setup na célula 3 ou clique em <b>Atualizar setups</b> depois.</span>"
    )

ok_pyserial_inicial, erro_pyserial_inicial = garantir_pyserial_importado()
if not ok_pyserial_inicial:
    msg_serial.value = (
        "<span style='color:#b00020'>"
        "<b>pyserial não está disponível no kernel atual.</b><br>"
        "Clique em <b>Instalar pyserial</b> nesta célula.<br>"
        f"<b>Python do kernel:</b><br><code>{sys.executable}</code>"
        "</span>"
    )
else:
    msg_serial.value = (
        "<span style='color:#1b8a3a'>"
        "<b>pyserial disponível no kernel atual.</b><br>"
        "Clique em <b>Atualizar portas</b> para listar o Arduino."
        "</span>"
    )

print("Célula 5 carregada. Modo independente disponível mesmo sem célula 4.")
print(f"Sketch Arduino: {ARDUINO_SKETCH_FILE}")
print(f"Python do kernel: {sys.executable}")
print("Formato serial enviado: X90.0,Y85.5,P22.0")

[ WARN:0@14624.298] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
ioctl(VIDIOC_QBUF): Bad file descriptor
[ WARN:0@14625.716] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video3): can't open camera by index
ioctl(VIDIOC_QBUF): Bad file descriptor
[ WARN:0@14626.599] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video5): can't open camera by index
ioctl(VIDIOC_QBUF): Bad file descriptor


Célula 5 carregada. Modo independente disponível mesmo sem célula 4.
Sketch Arduino: vision/fire_detect/jet_automation/parameters_setups/blaze_arduino_uno_mg90s/blaze_arduino_uno_mg90s.ino
Python do kernel: .venv/bin/python
Formato serial enviado: X90.0,Y85.5,P22.0


## Instalações necessárias

No ambiente Python usado pelo VS Code:

```bash
pip install opencv-python numpy ipywidgets
```

Para comunicação com Arduino:

```bash
pip install pyserial
```

Para ativar a trava de segurança por YOLO humano:

```bash
pip install ultralytics
```

A trava YOLO fica desativada por padrão na célula 3. Ative apenas quando o pacote estiver instalado e quando a máquina suportar rodar o detector em paralelo à captura.
